# Notebook 10 - Simulator Kernel Integration

## Purpose

Notebook 10 converts HydroLab-3D from a collection of individually validated robotics components into the first reusable **simulator kernel**.

Notebooks 00 through 09 progressively established:

```text
3D kinematics
    ↓
orientation and reference frames
    ↓
force-based dynamics
    ↓
hydrodynamic drag
    ↓
gravity and buoyancy
    ↓
thruster models
    ↓
ocean currents
    ↓
complete lightweight 6-DOF UUV dynamics
    ↓
sensor models
    ↓
closed-loop control
```

Notebook 10 now introduces the orchestration layer that connects those components behind a small simulator-facing interface.

The central target is:

```python
state = simulator.step(action)
```

The objective is deliberately narrow.

Notebook 10 does **not** build another physics model.

It builds the mechanism that advances the already validated HydroLab-3D physics by one deterministic simulation timestep.

---

## Why This Matters

Until this point, individual notebooks have manually coordinated the simulation loop.

A typical experiment has had to perform operations such as:

```text
construct actuator wrench
        ↓
provide vehicle parameters
        ↓
provide ocean current
        ↓
call 6-DOF integration
        ↓
update pose
        ↓
update body velocity
        ↓
advance experiment time
        ↓
generate sensor measurements
        ↓
store the new state
```

That is appropriate while validating individual mathematical components.

It is not an appropriate permanent simulator interface.

An autonomy algorithm should not need to know how HydroLab-3D internally coordinates its mass matrix, hydrodynamic damping, hydrostatic restoring wrench, current-relative velocity, numerical integration, or state bookkeeping.

Instead, the intended abstraction is:

```text
external algorithm
        ↓
      action
        ↓
Simulator.step(...)
        ↓
physical simulation
        ↓
updated simulator state
        ↓
external algorithm
```

Notebook 10 therefore marks an architectural transition:

```text
validated component library
            ↓
      simulator kernel
            ↓
reusable HydroLab-3D simulation
```

This is the layer that later controllers, learning environments, interactive input systems, ROS 2 adapters, and multi-UUV logic can call without directly orchestrating vehicle physics themselves.

---

## Relationship to Notebooks 00-09

Notebook 10 inherits the validated technical lineage rather than reproducing it.

### Notebook 00 - Basic 3D Kinematics

Established deterministic timestep-based translational integration.

### Notebook 01 - Orientation and Reference Frames

Established the Z-Y-X roll-pitch-yaw convention and body-to-world transformations.

### Notebook 02 - Basic Dynamics

Introduced force-based translational dynamics and began promotion of validated notebook functionality into the production package.

### Notebook 03 - Hydrodynamic Drag

Established velocity-dependent hydrodynamic resistance.

### Notebook 04 - Gravity and Buoyancy

Established hydrostatic force behavior and the vertical world convention.

### Notebook 05 - Thruster Model

Established actuator-generated forces and their body/world-frame interpretation.

### Notebook 06 - Ocean Currents

Established water-relative velocity and current-aware vehicle behavior.

### Notebook 07 - Complete 6-DOF UUV

Combined the preceding physics into the main lightweight marine vehicle model:

$$
M\dot{\nu}
+
C(\nu)\nu
+
D(\nu_r)\nu_r
+
g(\eta)
=
\tau
+
\tau_{env}.
$$

The corresponding pose kinematics are:

$$
\dot{\eta}
=
J(\eta)\nu.
$$

The production vehicle state is represented by:

$$
\eta
=
[x,y,z,\phi,\theta,\psi]^T
$$

and:

$$
\nu
=
[u,v,w,p,q,r]^T.
$$

Notebook 07 promoted the main production integration interface:

```python
integrate_six_dof_euler(
    pose,
    body_velocity,
    dt,
    **dynamics_kwargs,
)
```

which returns:

```text
next_pose
next_body_velocity
diagnostics
```

Notebook 10 will reuse that implementation rather than reproduce its equations.

### Notebook 08 - Sensor Models

Separated physical truth from sensor observations.

The established architectural rule remains:

$$
\boxed{
\text{ground truth}
\neq
\text{measurement}
}
$$

The production sensor package currently provides lightweight models for:

```text
IMU
├── gyroscope
└── accelerometer

depth

DVL

heading
```

Sensor bias and noise affect observations only.

They must never overwrite simulator truth.

### Notebook 09 - Closed-Loop Control

Demonstrated the first autonomous feedback loop:

```text
reference / waypoint
        ↓
guidance
        ↓
controller
        ↓
actuator wrench
        ↓
6-DOF plant
        ↓
sensor feedback
        ↺
```

Notebook 09 also established an important architectural boundary:

```text
controller
    ↓
action
    ↓
vehicle dynamics
```

The simulator kernel should therefore remain below the controller.

It should not secretly contain waypoint guidance, PID logic, mission behavior, or localization.

This allows the same simulator later to accept actions from:

```text
classical controllers
learning policies
keyboard control
ROS 2 nodes
scripted experiments
future autonomy systems
```

without changing the underlying physical simulator.

---

## Existing Production Interfaces to Reuse

The most important inherited production interface is:

```python
six_dof_state_derivative(
    pose,
    body_velocity,
    *,
    mass,
    inertia,
    added_inertia,
    linear_damping,
    quadratic_damping,
    fluid_density,
    displaced_volume,
    center_of_gravity,
    center_of_buoyancy,
    current_velocity_world,
    actuator_wrench,
    environmental_wrench=None,
    gravity=9.81,
)
```

together with:

```python
integrate_six_dof_euler(
    pose,
    body_velocity,
    dt,
    **dynamics_kwargs,
)
```

The existing vehicle layer also exposes:

```python
current_generalized_acceleration(...)
simulate_six_dof(...)
```

while the sensor layer exposes production functions including:

```python
depth_measurement(...)
heading_measurement(...)
dvl_measurement(...)
gyroscope_measurement(...)
accelerometer_measurement(...)
wrap_angle_pi(...)
```

Notebook 10 will compose these existing production APIs.

It will not copy their internal mathematics into the simulator kernel.

---

## What `Simulator.step()` Means

A call to:

```python
simulator.step(action)
```

will conceptually mean:

> Advance one HydroLab-3D vehicle simulation by exactly one configured physics timestep using the supplied actuator action and the currently configured vehicle and environment parameters.

The intended execution chain is:

```text
action
  │
  ▼
validate action
  │
  ▼
current simulator truth
pose + body velocity + time
  │
  ▼
configured environmental state
current / disturbances
  │
  ▼
existing 6-DOF production integrator
  │
  ▼
next pose + next body velocity
  │
  ▼
advance deterministic simulation time
  │
  ▼
derive observations if enabled
  │
  ▼
return updated simulation output
```

One call to `step()` should correspond to exactly:

$$
\Delta t
$$

of simulated time.

Therefore:

$$
t_{k+1}
=
t_k+\Delta t.
$$

Simulation time will be model time.

It will not depend on:

```text
wall-clock time
CPU speed
rendering frame rate
sleep calls
real-time scheduling
```

This distinction is essential for reproducibility and future accelerated headless execution.

---

## Intended Action Boundary

The smallest action representation consistent with the validated six-DOF plant is a generalized body-frame actuator wrench:

$$
\tau
=
\begin{bmatrix}
F_x\\
F_y\\
F_z\\
M_x\\
M_y\\
M_z
\end{bmatrix}.
$$

Conceptually:

```text
action
=
[Fx, Fy, Fz, Mx, My, Mz]
```

with translational forces measured in newtons and moments measured in newton-metres.

This boundary is attractive because the existing six-DOF production model already accepts:

```python
actuator_wrench
```

directly.

It also preserves separation between:

```text
high-level control
```

and:

```text
vehicle physics
```

Notebook 10 therefore does not initially require a thruster-allocation layer.

The existing individual thruster functions remain available for experiments that need physical thruster forces, but the simulator kernel does not need to invent a multi-thruster geometry merely to create `step()`.

A future actuator-allocation layer may convert:

```text
individual thruster commands
        ↓
generalized wrench
        ↓
Simulator.step(...)
```

without changing the core physics interface.

The final action contract will still be verified against the current production implementation before promotion.

---

## Intended State Boundary

HydroLab-3D must clearly distinguish the state the simulator **knows internally** from the measurements an autonomous system may receive.

The minimum authoritative simulator truth is:

```text
simulation time
pose
body velocity
```

where:

$$
\eta
=
[x,y,z,\phi,\theta,\psi]^T
$$

and:

$$
\nu
=
[u,v,w,p,q,r]^T.
$$

Conceptually:

```text
ground truth
├── time
├── pose
└── body_velocity
```

The state returned by the kernel should be a snapshot rather than a direct mutable reference to internal simulator arrays.

Caller-owned arrays and simulator-owned arrays must not accidentally alias one another.

The kernel therefore needs explicit copy semantics at its public state boundary.

---

## Ground Truth and Observation Boundary

Sensor observations, when integrated, remain derived data.

Conceptually:

```text
Simulator
│
├── truth
│   ├── pose
│   ├── body_velocity
│   └── time
│
└── observations
    ├── depth
    ├── heading
    ├── DVL
    └── IMU
```

The relationship is:

$$
\text{observation}
=
h(\text{ground truth})
+
\text{sensor imperfections}.
$$

The reverse relationship must never occur.

In particular:

```text
noisy depth
```

must never replace:

```text
true z position
```

and:

```text
noisy heading
```

must never replace:

```text
true yaw
```

inside the physics state.

If Notebook 10 integrates noisy measurements into the kernel API, all stochastic behavior will continue to use explicit NumPy random generators so that seeded simulations remain reproducible.

Sensor integration will remain lightweight and configuration-driven.

It will not become an estimator or localization system.

---

## World and Coordinate Conventions

Notebook 10 inherits all previously validated coordinate conventions.

The world frame uses:

```text
+x_world
+y_world
+z_world upward
```

The default water surface is:

$$
z=0.
$$

Therefore underwater positions satisfy:

$$
z<0
$$

and depth is:

$$
\boxed{
depth=-z
}
$$

for the default surface reference.

Orientation remains:

```text
roll  = phi
pitch = theta
yaw   = psi
```

and heading remains equal to yaw.

The six-DOF kinematics use Z-Y-X Euler angles.

Consequently the mathematical singularity near:

$$
\theta=\pm90^\circ
$$

remains an inherited limitation.

Notebook 10 will not conceal that limitation through arbitrary clipping.

---

## Ocean Current Boundary

Ocean current remains an environmental input to the existing six-DOF vehicle dynamics.

Conceptually:

```text
configured world current
        ↓
current_velocity_world
        ↓
water-relative vehicle velocity
        ↓
hydrodynamic forces
        ↓
vehicle acceleration
```

The simulator kernel should therefore store or obtain the current environmental velocity required for each step and pass it into the existing plant implementation.

Notebook 10 will not implement CFD, turbulence simulation, or a new current model.

It will reuse the current representation already validated in Notebooks 06 and 07.

---

## Reset Semantics

A reusable simulator requires a deterministic state lifecycle.

Notebook 10 will therefore investigate the minimum useful:

```python
simulator.reset(...)
```

contract alongside:

```python
simulator.step(...)
```

The intended reset behavior is:

```text
configured initial pose
+
configured initial body velocity
+
simulation time = 0
+
reproducible stochastic state
        ↓
reset()
        ↓
exact initial simulator condition
```

Repeated runs initialized with identical configuration and random seeds should reproduce the same simulation trajectory and sensor sequence.

Reset must restore simulator state through the kernel's state lifecycle.

It must not depend on manually editing internal arrays from outside the simulator.

---

## Minimal Kernel Architecture

HydroLab-3D already contains a tentative `core/` package scaffold.

Earlier repository audits showed that several tentative files existed before their corresponding behavior had actually been implemented.

Their presence therefore does not constitute architectural justification.

Notebook 10 will promote only abstractions that are demonstrated to be useful.

The smallest defensible production target is expected to resemble:

```text
src/hydrolab3d/core/
├── __init__.py
└── simulator.py
```

with the exact existing files inspected again before anything is modified.

A separate:

```text
clock.py
state.py
entity.py
```

will **not** be implemented merely because the tentative README tree contains those names.

If scalar simulation time, copied NumPy state arrays, and a compact `Simulator` class are sufficient, then that is preferable.

The guiding rule is:

$$
\boxed{
\text{introduce an abstraction only when validated behavior requires it}
}
$$

This keeps the simulator understandable and prevents architecture from growing faster than capability.

---

## Intended Simulator Responsibility

The simulator kernel should own only the state and orchestration necessary to advance the physical model.

Conceptually:

```text
Simulator
│
├── configuration
│   ├── dt
│   ├── vehicle parameters
│   ├── environment/current parameters
│   └── optional sensor configuration
│
├── mutable simulator truth
│   ├── pose
│   ├── body velocity
│   └── simulation time
│
├── reset(...)
│
└── step(action)
        │
        └── inherited 6-DOF dynamics
```

The kernel should remain:

```text
NumPy-first
CPU-friendly
headless-first
deterministic where configured
explicit
small
modular
reproducible
```

The renderer remains completely outside this loop.

---

## What We Will Implement

Notebook 10 will develop and validate the minimum reusable simulator kernel required to support:

```text
initialization
state ownership
deterministic time
reset
one-step propagation
environment/current input
6-DOF plant integration
state snapshots
optional observation generation
input validation
reproducibility
```

The development will follow the existing HydroLab-3D notebook workflow:

```text
inspect existing production implementation
        ↓
prototype kernel locally in notebook
        ↓
validate isolated behavior
        ↓
validate direct plant parity
        ↓
validate reset and reproducibility
        ↓
validate longer rollouts
        ↓
promote only justified code
        ↓
fresh production re-import
        ↓
fresh-process validation
        ↓
persistent artifacts
        ↓
dedicated simulator tests
        ↓
complete repository regression suite
```

Production promotion will occur only after notebook-local behavior has been demonstrated.

---

## What We Will Not Implement

Notebook 10 will not become:

```text
a renderer
a game engine
a keyboard interface
a Gymnasium environment
an RL framework
a PPO implementation
a reward system
a mission planner
a localization system
a SLAM system
a sonar simulator
a communication simulator
a multi-UUV framework
a ROS 2 adapter
a CFD solver
a new controller architecture
a new hydrodynamic model
```

Notebook 10 will also not reimplement:

```text
rotation mathematics
6-DOF kinematics
mass matrices
Coriolis terms
hydrodynamic damping
gravity
buoyancy
thruster physics
ocean-current physics
sensor equations
PID control
waypoint geometry
```

Those capabilities already belong to validated lower layers.

The kernel coordinates them.

It does not compete with them.

---

## Direct Physics-Parity Requirement

The most important integration validation will compare:

```python
Simulator.step(action)
```

against the existing production call:

```python
integrate_six_dof_euler(...)
```

using identical:

```text
initial pose
initial body velocity
timestep
vehicle parameters
ocean current
actuator wrench
environmental wrench
```

For a correctly implemented orchestration layer:

$$
\eta_{kernel}
\approx
\eta_{direct}
$$

and:

$$
\nu_{kernel}
\approx
\nu_{direct}.
$$

Ideally, because both paths call the same production integrator with identical data, the discrepancy should approach floating-point identity.

This is critical.

If `Simulator.step()` changes the dynamics merely by wrapping them, then the wrapper is wrong.

---

## Validation Criteria

Notebook 10 will be considered successful only if the validated implementation demonstrates the following behavior.

### Single-Step Determinism

One call to:

```python
step(action)
```

must advance exactly one configured timestep.

If:

$$
t_k=t,
$$

then:

$$
t_{k+1}=t+\Delta t.
$$

### Direct Plant Parity

A simulator step must reproduce direct production 6-DOF integration for the same initial condition and inputs.

### Neutral-State Behavior

A physically balanced configuration under zero action should remain consistent with the inherited plant behavior.

The kernel must not introduce artificial drift independently of the physics model.

### Current Propagation

Changing the configured current must influence the vehicle only through the already validated current-aware dynamics.

### Reset Exactness

Calling:

```python
reset()
```

must recover the defined initial state and initial simulation time exactly.

### Replay Reproducibility

Two runs using identical:

```text
initial state
configuration
actions
environment
sensor seed
```

must reproduce identical results within the deterministic or seeded stochastic contract.

### State Ownership

Mutating an array supplied by the caller after initialization must not silently mutate the internal simulator state.

Likewise, mutating a returned state snapshot must not corrupt the simulator's internal truth.

### Input Validation

The kernel should reject malformed or physically invalid public inputs such as:

```text
incorrect pose shape
incorrect body-velocity shape
incorrect action shape
NaN action
infinite action
non-positive timestep
non-finite initial state
```

rather than allowing invalid state to propagate silently.

### Numerical Stability

Reasonable multi-step validation rollouts must remain finite:

$$
\eta_k\in\mathbb{R}^6
$$

and:

$$
\nu_k\in\mathbb{R}^6
$$

without unexpected:

```text
NaN
Inf
shape corruption
time corruption
```

within the validated operating regime.

### Ground-Truth Preservation

Sensor noise must never modify the underlying physical state.

### Headless Operation

All kernel validation must execute without requiring:

```text
graphics
display server
renderer
keyboard input
```

The simulator must remain fully usable in a headless process.

---

## Expected Result

At the end of Notebook 10, a user should be able to conceptually perform:

```python
simulator.reset()

state = simulator.step(action)

state = simulator.step(next_action)
```

without manually coordinating the internal six-degree-of-freedom model at every timestep.

The resulting architecture should resemble:

```text
Controller / Policy / User
          │
          ▼
        Action
          │
          ▼
   ┌───────────────┐
   │   Simulator   │
   │               │
   │ reset()       │
   │ step(action)  │
   └───────┬───────┘
           │
           ▼
 existing HydroLab-3D
    6-DOF physics
           │
           ▼
       next truth
           │
           ├──────────────► optional sensors
           │
           ▼
     state snapshot
```

The kernel should remain thin enough that the physical path is still obvious when reading the source.

No important dynamics should disappear behind opaque framework machinery.

---

## Production-Code Strategy

Reusable code will remain notebook-local until validated.

Only after the notebook-local simulator passes its integration tests will the justified kernel implementation be promoted into the production package.

The likely production location is:

```text
src/hydrolab3d/core/simulator.py
```

with:

```text
src/hydrolab3d/core/__init__.py
```

updated only as required to expose the validated public API.

Existing scaffold files will be inspected before modification.

No existing source file will be blindly replaced merely because it appears empty or tentative in earlier repository history.

After promotion, Notebook 10 will:

```text
discard notebook-local assumptions
        ↓
freshly import production code
        ↓
repeat parity validation
        ↓
launch a separate Python interpreter
        ↓
verify clean-package operation
```

This prevents notebook-local definitions or stale imports from masking production packaging errors.

---

## Automated-Test Strategy

The inherited pre-Notebook-10 repository baseline is:

```text
152 passed
0 failed
```

Notebook 10 will not predict the final test count in advance.

After production promotion:

```text
dedicated simulator-kernel tests
        ↓
pytest only those tests
        ↓
resolve failures if any
        ↓
complete repository pytest
```

The final notebook will report the actual pytest result discovered at runtime.

Existing production behavior from Notebooks 00 through 09 must remain regression-safe.

---

## Persistent Validation Artifacts

All Notebook 10 artifacts will remain isolated under:

```text
results/notebooks/10_simulator_kernel/
```

The exact artifact set will be determined by the validations that prove useful rather than by a requirement to manufacture plots.

Likely compact outputs include data describing:

```text
direct step parity
deterministic multi-step trajectory
reset / replay reproducibility
time progression
sensor-integrated rollout
summary validation metrics
```

Possible subdirectories such as:

```text
data/
plots/
```

will be created only if the final artifact set actually requires them.

No artifact from Notebooks 00 through 09 will be overwritten.

---

## Known Limitations That Remain

Notebook 10 does not remove limitations inherited from the current HydroLab-3D model.

These include:

### Euler-Angle Singularity

The Z-Y-X Euler representation remains singular near:

$$
\theta=\pm90^\circ.
$$

The simulator kernel will propagate the existing mathematical model rather than pretend this singularity does not exist.

### Lightweight Numerical Integration

The main vehicle propagation currently uses the validated explicit integration strategy from Notebook 07.

Notebook 10 is not an integrator-comparison study.

### Simplified Actuation Boundary

A generalized actuator wrench is not equivalent to a complete hardware-specific thruster allocator.

That distinction remains explicit.

### Simplified Environment

The current environment does not represent high-fidelity ocean turbulence or CFD.

### No Localization System

HydroLab-3D still distinguishes between simulator truth and what a real robot could estimate.

The perfect world position used in earlier waypoint validation was an explicit validation baseline, not a solved underwater localization system.

Notebook 10 will not disguise truth as localization.

### Lightweight Closed-Loop Controller

Notebook 09's waypoint controller remains a decoupled baseline.

Its coupled waypoint experiment exhibited a transient pitch excursion of approximately:

```text
42.68 deg
```

before recovering.

Notebook 10 does not solve that controller limitation because control design remains above the simulator kernel.

### No Sonar Yet

The existing sonar placeholder is not a validated sensor implementation.

Notebook 10 will not silently turn it into one.

---

## Relationship to the Overall HydroLab-3D Architecture

Notebook 10 establishes the first concrete version of the central simulation loop:

```text
                   external logic
                        │
                        ▼
                     action
                        │
                        ▼
                ┌───────────────┐
                │   Simulator   │
                └───────┬───────┘
                        │
        ┌───────────────┼───────────────┐
        ▼               ▼               ▼
    dynamics          world           sensors
        │               │               │
        └───────────────┼───────────────┘
                        ▼
                 updated state
```

Later systems should build around this loop rather than replacing it.

This preserves HydroLab-3D's central design philosophy:

$$
\boxed{
\text{maximum useful autonomy capability}
\over
\text{computational complexity}
}
$$

or, more practically:

```text
transparent physics
+
small interfaces
+
reproducible experiments
+
minimal overhead
```

---

## Handoff to Notebook 11

Notebook 11 will test whether this simulator kernel can be consumed by an external learning interface.

Its intended interaction becomes:

```text
temporary learning environment
          │
          ├── reset()
          │
          ▼
      HydroLab Simulator
          │
          ├── step(action)
          │
          ▼
        state
          │
          ▼
 temporary observation / reward logic
```

Notebook 11 may add notebook-local concepts such as:

```text
observations
rewards
episode termination
random actions
temporary Gymnasium wrapper
small PPO smoke test
```

but those concepts will remain **outside** the permanent HydroLab-3D core.

Notebook 10 therefore has one job before handing off:

$$
\boxed{
\text{Make HydroLab-3D behave like an actual simulator.}
}
$$

If Notebook 10 succeeds, Notebook 11 should be able to wrap the simulator rather than modify its physics.

That separation is the central architectural milestone of this notebook.

---

## Step 1 - Audit the Existing Simulator Integration Boundary

Before defining a `Simulator` class, Notebook 10 must inspect the **actual production boundary** that the simulator kernel will orchestrate.

The purpose of this step is not to design the kernel yet.

It is to establish exactly what HydroLab-3D currently contains after Notebooks 00 through 09 and distinguish:

```text
validated production implementation
```

from:

```text
tentative repository scaffold
```

This distinction is especially important for:

```text
src/hydrolab3d/core/
```

because files such as:

```text
simulator.py
clock.py
state.py
entity.py
```

may already exist as placeholders without representing validated architecture.

The simulator kernel must be designed around the production interfaces that actually exist, particularly:

```text
6-DOF integration
ocean-current handling
sensor generation
vehicle state conventions
control/action boundary
```

rather than around tentative filenames in the README.

This audit will therefore inspect:

1. the current repository root;
2. the complete `src/hydrolab3d/` production tree;
3. the existing `core/` package and its current contents;
4. the actual 6-DOF integration API;
5. the current sensor public API;
6. the current control public API;
7. the world/current public API;
8. the automated-test inventory;
9. the existing Notebook 10 result directory, if any;
10. and the inherited repository test baseline without modifying source code.

The most important production dependency is expected to be:

```python
integrate_six_dof_euler(...)
```

because the simulator kernel should orchestrate the validated vehicle model rather than implement a second dynamics engine.

Conceptually, Notebook 10 is trying to construct:

```text
action
  ↓
Simulator.step(...)
  ↓
existing production integration
  ↓
next pose + next body velocity
  ↓
time advancement
  ↓
state / observations
```

but no final action or state contract will be chosen until this inspection is complete.

### Validation Criteria

This step passes if:

1. the HydroLab-3D repository root is resolved successfully;
2. the current source tree is displayed;
3. every existing `core/` source file is inspected without modification;
4. the production 6-DOF integrator imports successfully and its real signature is reported;
5. the sensor, world, and control package APIs are reported;
6. the existing automated-test inventory is discovered;
7. the Notebook 10 result-directory state is reported;
8. no source, test, or result file is created or modified.

Only after this audit passes will we decide the smallest defensible Notebook 10 simulator architecture.

In [1]:
from pathlib import Path
import importlib
import inspect
import sys


# ---------------------------------------------------------------------
# Resolve HydroLab-3D repository root
# ---------------------------------------------------------------------

start = Path.cwd().resolve()
repo_root = None

for candidate in [start, *start.parents]:
    if (
        (candidate / "README.md").exists()
        and (candidate / "src").exists()
        and (candidate / "notebooks").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the HydroLab-3D repository root from "
        f"{start}"
    )


src_path = repo_root / "src"
package_root = src_path / "hydrolab3d"
tests_dir = repo_root / "tests"

if not package_root.exists():
    raise FileNotFoundError(
        f"HydroLab-3D package directory does not exist: {package_root}"
    )

if not tests_dir.exists():
    raise FileNotFoundError(
        f"HydroLab-3D tests directory does not exist: {tests_dir}"
    )


src_path_str = str(src_path)

if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)

importlib.invalidate_caches()


# ---------------------------------------------------------------------
# Utility helpers
# ---------------------------------------------------------------------

def relative(path):
    return path.relative_to(repo_root)


def list_python_tree(directory):
    """
    Return all Python files below a directory in deterministic order.
    """
    if not directory.exists():
        return []

    return sorted(
        path
        for path in directory.rglob("*.py")
        if path.is_file()
    )


def public_names(module):
    """
    Report the module's explicit public API when __all__ exists.
    Otherwise report public names defined/imported on the module.
    """
    if hasattr(module, "__all__"):
        return list(module.__all__)

    return sorted(
        name
        for name in vars(module)
        if not name.startswith("_")
    )


def display_source_file(path):
    """
    Display a complete source file without modifying it.
    """
    print("\n" + "=" * 80)
    print(relative(path))
    print("=" * 80)

    contents = path.read_text(encoding="utf-8")

    if contents.strip():
        print(contents)
    else:
        print("<EMPTY FILE>")


# ---------------------------------------------------------------------
# Discover actual production source tree
# ---------------------------------------------------------------------

production_python_files = list_python_tree(package_root)

if not production_python_files:
    raise RuntimeError(
        "No production Python files were discovered under "
        f"{package_root}"
    )


# ---------------------------------------------------------------------
# Inspect core package scaffold
# ---------------------------------------------------------------------

core_dir = package_root / "core"
core_python_files = list_python_tree(core_dir)


# ---------------------------------------------------------------------
# Import inherited production packages
# ---------------------------------------------------------------------

import hydrolab3d

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
    six_dof_state_derivative,
)

import hydrolab3d.sensors as sensors_module
import hydrolab3d.world as world_module
import hydrolab3d.control as control_module


# ---------------------------------------------------------------------
# Validate inherited central integration functions
# ---------------------------------------------------------------------

for function_name, function in [
    (
        "six_dof_state_derivative",
        six_dof_state_derivative,
    ),
    (
        "integrate_six_dof_euler",
        integrate_six_dof_euler,
    ),
]:
    if not callable(function):
        raise TypeError(
            f"{function_name} imported successfully but is not callable."
        )


# ---------------------------------------------------------------------
# Discover automated tests
# ---------------------------------------------------------------------

test_files = sorted(
    path
    for path in tests_dir.glob("test_*.py")
    if path.is_file()
)

if not test_files:
    raise RuntimeError(
        f"No test_*.py files were discovered in {tests_dir}"
    )


# ---------------------------------------------------------------------
# Inspect Notebook 10 output location
# ---------------------------------------------------------------------

results_root = (
    repo_root
    / "results"
    / "notebooks"
)

notebook10_results = (
    results_root
    / "10_simulator_kernel"
)

existing_notebook10_artifacts = []

if notebook10_results.exists():
    existing_notebook10_artifacts = sorted(
        path
        for path in notebook10_results.rglob("*")
        if path.is_file()
    )


# ---------------------------------------------------------------------
# Print audit report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 1 INTEGRATION AUDIT")
print("=" * 80)

print(f"\nRepository root:\n  {repo_root}")

print(f"\nProduction package:\n  {relative(package_root)}")


# ---------------------------------------------------------------------
# Actual production tree
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("CURRENT PRODUCTION PYTHON TREE")
print("-" * 80)

for path in production_python_files:
    print(relative(path))


# ---------------------------------------------------------------------
# Core package inspection
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("CURRENT CORE PACKAGE")
print("-" * 80)

if not core_dir.exists():
    print(
        "No src/hydrolab3d/core/ directory currently exists."
    )

elif not core_python_files:
    print(
        "src/hydrolab3d/core/ exists but contains no Python files."
    )

else:
    for path in core_python_files:
        print(relative(path))

    print(
        "\nComplete current core source follows."
    )

    for path in core_python_files:
        display_source_file(path)


# ---------------------------------------------------------------------
# Central 6-DOF integration boundary
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("6-DOF PRODUCTION INTEGRATION API")
print("-" * 80)

for name, function in [
    (
        "six_dof_state_derivative",
        six_dof_state_derivative,
    ),
    (
        "integrate_six_dof_euler",
        integrate_six_dof_euler,
    ),
]:
    print(f"\n{name}")
    print(f"  module    : {function.__module__}")
    print(f"  source    : {inspect.getsourcefile(function)}")
    print(f"  signature : {inspect.signature(function)}")


# ---------------------------------------------------------------------
# Public package APIs
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("SENSOR PUBLIC API")
print("-" * 80)

for name in public_names(sensors_module):
    print(name)


print("\n" + "-" * 80)
print("WORLD PUBLIC API")
print("-" * 80)

for name in public_names(world_module):
    print(name)


print("\n" + "-" * 80)
print("CONTROL PUBLIC API")
print("-" * 80)

for name in public_names(control_module):
    print(name)


# ---------------------------------------------------------------------
# Test inventory
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("AUTOMATED TEST INVENTORY")
print("-" * 80)

for path in test_files:
    print(relative(path))

print(f"\nDiscovered test files: {len(test_files)}")


# ---------------------------------------------------------------------
# Notebook 10 result-directory state
# ---------------------------------------------------------------------

print("\n" + "-" * 80)
print("NOTEBOOK 10 RESULT DIRECTORY")
print("-" * 80)

print(
    f"Target: {relative(notebook10_results)}"
)

if not notebook10_results.exists():
    print(
        "Status: directory does not currently exist."
    )

elif not existing_notebook10_artifacts:
    print(
        "Status: directory exists and contains no files."
    )

else:
    print(
        "Status: directory exists with the following files:"
    )

    for path in existing_notebook10_artifacts:
        print(relative(path))


# ---------------------------------------------------------------------
# Read-only audit confirmation
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 1 AUDIT COMPLETE")
print("=" * 80)

print(
    "Repository structure, existing core scaffold, inherited production "
    "APIs, tests, and Notebook 10 output state were inspected."
)

print(
    "This step did not intentionally create or modify any project file."
)

print(
    "\nNext decision after reviewing this output:"
    "\n  determine the smallest defensible Simulator state/action/reset "
    "contract."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 1 INTEGRATION AUDIT

Repository root:
  /home/agniacolyte/HydroLab-3D

Production package:
  src/hydrolab3d

--------------------------------------------------------------------------------
CURRENT PRODUCTION PYTHON TREE
--------------------------------------------------------------------------------
src/hydrolab3d/__init__.py
src/hydrolab3d/communication/__init__.py
src/hydrolab3d/communication/acoustic_channel.py
src/hydrolab3d/communication/network.py
src/hydrolab3d/communication/packet.py
src/hydrolab3d/control/__init__.py
src/hydrolab3d/control/controller.py
src/hydrolab3d/control/pid.py
src/hydrolab3d/control/waypoint.py
src/hydrolab3d/core/__init__.py
src/hydrolab3d/core/clock.py
src/hydrolab3d/core/entity.py
src/hydrolab3d/core/simulator.py
src/hydrolab3d/core/state.py
src/hydrolab3d/dynamics/__init__.py
src/hydrolab3d/dynamics/buoyancy.py
src/hydrolab3d/dynamics/drag.py
src/hydrolab3d/dynamics/hydrodynamics.py
src/hydrolab3d/dynamics/kinematics.p

## Step 2 - Inspect and Probe the Exact Production Contracts

Step 1 established that the HydroLab-3D `core/` package is currently only an empty scaffold and that the simulator kernel should therefore be designed around the production interfaces already validated by earlier notebooks.

The first Step 2 probe exposed an important production contract:

```text
rigid-body inertia
```

is represented by the principal-axis vector:

```text
[Ixx, Iyy, Izz]
```

with shape:

```text
(3,)
```

rather than by a $3\times3$ inertia matrix.

This is useful engineering evidence.

The failed probe did not indicate a defect in the six-degree-of-freedom model. It demonstrated that Notebook 10 must respect the exact parameter representations already established by the production hydrodynamics layer.

This replacement step therefore uses the validated parameter shapes and inspects the contracts that the future simulator kernel will compose.

The central propagation boundary remains:

```python
integrate_six_dof_euler(
    pose,
    body_velocity,
    dt,
    **dynamics_kwargs,
)
```

The goal is to determine:

```text
what it returns
what diagnostics it exposes
whether caller-owned inputs are mutated
which diagnostic quantities can support sensors
what arguments the production sensor functions require
```

No `Simulator` class is implemented yet.

Instead, this step performs one controlled direct-production integration using:

```text
pose                  shape (6,)
body velocity         shape (6,)
rigid-body inertia    shape (3,)
added inertia         shape (6,)
linear damping        shape (6,)
quadratic damping     shape (6,)
current velocity      shape (3,)
actuator wrench       shape (6,)
```

The probe deliberately uses:

```text
neutral buoyancy
small ocean current
small forward actuator wrench
```

so that the returned state should remain finite and physically interpretable.

### Sensor Boundary

The production sensors require different pieces of simulator truth:

```text
depth
    ← pose

heading
    ← pose

DVL
    ← pose
    + body velocity
    + world-frame current

gyroscope
    ← body velocity

accelerometer
    ← pose
    + body velocity
    + body acceleration
```

The accelerometer is particularly important for Notebook 10 because the six-DOF integrator diagnostics include:

```text
nu_dot
```

which may provide the body-acceleration quantity needed to generate the IMU observation without recomputing the vehicle dynamics.

### Validation Criteria

This step passes if:

1. the inherited 6-DOF and sensor signatures are reported;
2. a valid call to `integrate_six_dof_euler()` succeeds;
3. the call returns exactly:

```text
next_pose
next_body_velocity
diagnostics
```

4. both returned state arrays have shape `(6,)`;
5. all returned state values are finite;
6. the diagnostics expose the quantities generated by the production integrator;
7. the caller-owned input pose is unchanged;
8. the caller-owned input body velocity is unchanged;
9. representative ideal sensor measurements can be generated from the returned truth and diagnostics;
10. no source, test, or result file is created or modified.

The resulting contract will determine the minimum notebook-local `Simulator` architecture in the next step.

In [3]:
import inspect
import numpy as np

from hydrolab3d.vehicles.uuv import (
    six_dof_state_derivative,
    integrate_six_dof_euler,
)

from hydrolab3d.sensors import (
    depth_measurement,
    heading_measurement,
    dvl_measurement,
    gyroscope_measurement,
    accelerometer_measurement,
)


# ---------------------------------------------------------------------
# Utility helpers
# ---------------------------------------------------------------------

def report_signature(function):
    print(
        f"{function.__name__}{inspect.signature(function)}"
    )


def report_array(name, value):
    value = np.asarray(value)

    print(
        f"{name}: "
        f"shape={value.shape}, "
        f"finite={np.all(np.isfinite(value))}"
    )
    print(value)


# ---------------------------------------------------------------------
# Report inherited contracts
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 2 PRODUCTION CONTRACT PROBE")
print("=" * 80)

print("\n6-DOF production API")
print("-" * 80)

report_signature(
    six_dof_state_derivative
)

report_signature(
    integrate_six_dof_euler
)


print("\nSensor production API")
print("-" * 80)

for function in [
    depth_measurement,
    heading_measurement,
    dvl_measurement,
    gyroscope_measurement,
    accelerometer_measurement,
]:
    report_signature(function)


# ---------------------------------------------------------------------
# Define one valid production-model probe
# ---------------------------------------------------------------------

probe_pose = np.array(
    [
        0.0,   # x [m]
        0.0,   # y [m]
        -5.0,  # z [m]
        0.0,   # roll [rad]
        0.0,   # pitch [rad]
        0.0,   # yaw [rad]
    ],
    dtype=float,
)

probe_body_velocity = np.zeros(
    6,
    dtype=float,
)

probe_dt = 0.02

probe_mass = 50.0

# Production rigid-body inertia contract:
# [Ixx, Iyy, Izz]
probe_inertia = np.array(
    [
        2.0,
        3.0,
        4.0,
    ],
    dtype=float,
)

# Production added-inertia contract:
# [Xu_dot, Yv_dot, Zw_dot, Kp_dot, Mq_dot, Nr_dot]
probe_added_inertia = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.5,
        0.7,
        0.9,
    ],
    dtype=float,
)

probe_linear_damping = np.array(
    [
        8.0,
        12.0,
        15.0,
        1.5,
        2.0,
        2.5,
    ],
    dtype=float,
)

probe_quadratic_damping = np.array(
    [
        12.0,
        18.0,
        22.0,
        2.0,
        2.5,
        3.0,
    ],
    dtype=float,
)

probe_fluid_density = 1025.0

# Neutral buoyancy:
# rho * V = mass
probe_displaced_volume = (
    probe_mass
    / probe_fluid_density
)

probe_center_of_gravity = np.array(
    [0.0, 0.0, 0.0],
    dtype=float,
)

probe_center_of_buoyancy = np.array(
    [0.0, 0.0, 0.08],
    dtype=float,
)

probe_current_velocity_world = np.array(
    [
        0.15,
        -0.05,
        0.0,
    ],
    dtype=float,
)

probe_actuator_wrench = np.array(
    [
        20.0,  # Fx [N]
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)


# ---------------------------------------------------------------------
# Preserve caller-owned inputs for mutation checks
# ---------------------------------------------------------------------

probe_pose_before = probe_pose.copy()

probe_body_velocity_before = (
    probe_body_velocity.copy()
)


# ---------------------------------------------------------------------
# Execute exactly one direct production timestep
# ---------------------------------------------------------------------

(
    probe_next_pose,
    probe_next_body_velocity,
    probe_diagnostics,
) = integrate_six_dof_euler(
    probe_pose,
    probe_body_velocity,
    probe_dt,
    mass=probe_mass,
    inertia=probe_inertia,
    added_inertia=probe_added_inertia,
    linear_damping=probe_linear_damping,
    quadratic_damping=probe_quadratic_damping,
    fluid_density=probe_fluid_density,
    displaced_volume=probe_displaced_volume,
    center_of_gravity=probe_center_of_gravity,
    center_of_buoyancy=probe_center_of_buoyancy,
    current_velocity_world=(
        probe_current_velocity_world
    ),
    actuator_wrench=probe_actuator_wrench,
)


# ---------------------------------------------------------------------
# Validate returned state structure
# ---------------------------------------------------------------------

assert isinstance(
    probe_next_pose,
    np.ndarray,
)

assert isinstance(
    probe_next_body_velocity,
    np.ndarray,
)

assert isinstance(
    probe_diagnostics,
    dict,
)

assert probe_next_pose.shape == (6,)

assert probe_next_body_velocity.shape == (6,)

assert np.all(
    np.isfinite(probe_next_pose)
)

assert np.all(
    np.isfinite(probe_next_body_velocity)
)


# ---------------------------------------------------------------------
# Validate caller-owned arrays were not mutated
# ---------------------------------------------------------------------

probe_pose_unchanged = np.array_equal(
    probe_pose,
    probe_pose_before,
)

probe_velocity_unchanged = np.array_equal(
    probe_body_velocity,
    probe_body_velocity_before,
)

assert probe_pose_unchanged

assert probe_velocity_unchanged


# ---------------------------------------------------------------------
# Inspect returned diagnostics
# ---------------------------------------------------------------------

print("\nDirect integration result")
print("-" * 80)

report_array(
    "next_pose",
    probe_next_pose,
)

report_array(
    "next_body_velocity",
    probe_next_body_velocity,
)


print("\nDiagnostics")
print("-" * 80)

print(
    f"diagnostic keys ({len(probe_diagnostics)}):"
)

for key in sorted(
    probe_diagnostics.keys()
):
    value = probe_diagnostics[key]

    if isinstance(value, np.ndarray):
        print(
            f"  {key:<32} "
            f"shape={value.shape} "
            f"finite={np.all(np.isfinite(value))}"
        )
    else:
        print(
            f"  {key:<32} "
            f"type={type(value).__name__} "
            f"value={value}"
        )


# ---------------------------------------------------------------------
# Confirm the derivative quantities needed by the kernel
# ---------------------------------------------------------------------

required_diagnostic_keys = {
    "eta_dot",
    "nu_dot",
    "dt",
    "nu_r",
    "actuator_wrench",
}

missing_diagnostic_keys = (
    required_diagnostic_keys
    - set(probe_diagnostics)
)

assert not missing_diagnostic_keys, (
    "Missing expected integration diagnostics: "
    f"{sorted(missing_diagnostic_keys)}"
)


probe_body_acceleration = np.asarray(
    probe_diagnostics["nu_dot"],
    dtype=float,
)

assert probe_body_acceleration.shape == (6,)

assert np.all(
    np.isfinite(probe_body_acceleration)
)


# ---------------------------------------------------------------------
# Generate representative ideal sensor observations
#
# noise_std = 0 keeps this probe deterministic and avoids requiring RNG.
# ---------------------------------------------------------------------

probe_depth = depth_measurement(
    probe_next_pose,
    noise_std=0.0,
)

probe_heading = heading_measurement(
    probe_next_pose,
    noise_std=0.0,
)

probe_dvl = dvl_measurement(
    probe_next_pose,
    probe_next_body_velocity,
    probe_current_velocity_world,
    noise_std=0.0,
)

probe_gyro = gyroscope_measurement(
    probe_next_body_velocity,
    noise_std=0.0,
)

probe_accelerometer = accelerometer_measurement(
    probe_next_pose,
    probe_next_body_velocity,
    probe_body_acceleration,
    noise_std=0.0,
)


# ---------------------------------------------------------------------
# Validate sensor outputs
# ---------------------------------------------------------------------

assert np.isfinite(probe_depth)

assert np.isfinite(probe_heading)

probe_dvl = np.asarray(
    probe_dvl,
    dtype=float,
)

probe_gyro = np.asarray(
    probe_gyro,
    dtype=float,
)

probe_accelerometer = np.asarray(
    probe_accelerometer,
    dtype=float,
)

assert probe_dvl.shape == (3,)

assert probe_gyro.shape == (3,)

assert probe_accelerometer.shape == (3,)

assert np.all(
    np.isfinite(probe_dvl)
)

assert np.all(
    np.isfinite(probe_gyro)
)

assert np.all(
    np.isfinite(probe_accelerometer)
)


# ---------------------------------------------------------------------
# Report sensor boundary
# ---------------------------------------------------------------------

print("\nRepresentative ideal post-step observations")
print("-" * 80)

print(
    f"depth         : {probe_depth:.12f} m"
)

print(
    f"heading       : {probe_heading:.12f} rad"
)

report_array(
    "DVL",
    probe_dvl,
)

report_array(
    "gyroscope",
    probe_gyro,
)

report_array(
    "accelerometer",
    probe_accelerometer,
)


# ---------------------------------------------------------------------
# Final Step 2 report
# ---------------------------------------------------------------------

print("\nCaller-owned input protection")
print("-" * 80)

print(
    f"pose unchanged          : "
    f"{probe_pose_unchanged}"
)

print(
    f"body velocity unchanged : "
    f"{probe_velocity_unchanged}"
)


print("\n" + "=" * 80)
print("STEP 2 PRODUCTION CONTRACT PROBE PASSED")
print("=" * 80)

print(
    "One direct production timestep completed successfully."
)

print(
    "The production integrator returned finite pose, velocity, "
    "and diagnostics without mutating caller-owned state arrays."
)

print(
    "Post-step depth, heading, DVL, gyroscope, and accelerometer "
    "measurements were generated successfully from production truth."
)

print(
    "The discovered contract is sufficient to design the minimal "
    "notebook-local Simulator prototype in the next step."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 2 PRODUCTION CONTRACT PROBE

6-DOF production API
--------------------------------------------------------------------------------
six_dof_state_derivative(pose, body_velocity, *, mass, inertia, added_inertia, linear_damping, quadratic_damping, fluid_density, displaced_volume, center_of_gravity, center_of_buoyancy, current_velocity_world, actuator_wrench, environmental_wrench=None, gravity=9.81)
integrate_six_dof_euler(pose, body_velocity, dt, **dynamics_kwargs)

Sensor production API
--------------------------------------------------------------------------------
depth_measurement(pose, *, surface_z=0.0, bias=0.0, noise_std=0.0, rng=None)
heading_measurement(pose, *, bias=0.0, noise_std=0.0, rng=None)
dvl_measurement(pose, body_velocity, current_velocity_world, *, bias=0.0, noise_std=0.0, rng=None)
gyroscope_measurement(body_velocity, *, bias=0.0, noise_std=0.0, rng=None)
accelerometer_measurement(pose, body_velocity, body_acceleration, *, gravity=9.81, 

## Step 3 - Build the Minimal Notebook-Local Simulator Kernel

Steps 1 and 2 established the production boundary that Notebook 10 should orchestrate.

The central validated plant interface is:

```python
integrate_six_dof_euler(
    pose,
    body_velocity,
    dt,
    **dynamics_kwargs,
)
```

and the production probe confirmed that it returns:

```text
next_pose
next_body_velocity
diagnostics
```

without mutating caller-owned pose or velocity arrays.

Notebook 10 can therefore begin with a deliberately small simulator kernel.

The first prototype will remain notebook-local.

Nothing is promoted into:

```text
src/hydrolab3d/core/
```

yet.

The purpose of this step is to validate the smallest useful simulator lifecycle before any production architecture is committed.

---

### Minimal Simulator Responsibility

The notebook-local simulator will own:

```text
initial pose
initial body velocity
current pose
current body velocity
simulation time
fixed timestep
vehicle / environment parameters
latest integration diagnostics
```

It will expose:

```python
reset()
```

and:

```python
step(action)
```

plus a read-only-style copied state snapshot through:

```python
state()
```

Conceptually:

```text
            Simulator
                │
       ┌────────┴────────┐
       │                 │
     reset()          step(action)
       │                 │
       ▼                 ▼
initial truth      production 6-DOF
       │             integration
       │                 │
       └────────┬────────┘
                ▼
          simulator truth
                │
                ▼
             state()
```

---

### Action Contract

For this first kernel prototype, the action is the generalized six-axis actuator wrench already accepted by the production plant:

$$
\tau
=
\begin{bmatrix}
F_x &
F_y &
F_z &
M_x &
M_y &
M_z
\end{bmatrix}^T.
$$

Therefore:

```python
action.shape == (6,)
```

The simulator will validate that every action is finite.

This keeps the control boundary independent of any particular future actuator allocator.

---

### Truth-State Contract

The authoritative internal simulator state is:

```text
time
pose
body_velocity
```

where:

$$
\eta
=
[x,y,z,\phi,\theta,\psi]^T
$$

and:

$$
\nu
=
[u,v,w,p,q,r]^T.
$$

The prototype will return copies of its arrays rather than references to mutable internal storage.

This establishes the ownership rule:

$$
\boxed{
\text{external mutation must not modify simulator truth}
}
$$

---

### Time Semantics

Simulation time is deterministic model time.

One successful call to:

```python
step(action)
```

advances:

$$
t_{k+1}
=
t_k+\Delta t.
$$

The kernel does not use wall-clock time.

The configured timestep remains constant during this prototype.

---

### Reset Semantics

Calling:

```python
reset()
```

will restore:

```text
initial pose
initial body velocity
time = 0
latest diagnostics = None
```

The reset state must exactly reproduce the initial configured state.

The initial arrays will be copied when the simulator is created so that later mutation of the caller's original arrays cannot alter the simulator's reset reference.

---

### Scope of This Step

This prototype intentionally does **not** yet include:

```text
sensor observations
sensor noise
random generators
controllers
waypoint guidance
rewards
episode termination
rendering
logging
production promotion
```

Those layers will only be added after the basic kernel lifecycle is validated.

The immediate question is simpler:

> Can a small object reliably own HydroLab-3D state and advance the existing six-degree-of-freedom plant exactly one timestep per action?

### Validation Criteria

This step passes if:

1. the simulator can be constructed from valid initial conditions;
2. construction copies caller-owned initial-state arrays;
3. `state()` returns:

```text
time
pose
body_velocity
```

4. returned state arrays are copies rather than mutable internal references;
5. `step(action)` accepts one finite six-axis wrench;
6. one call advances simulation time by exactly one `dt`;
7. one call produces finite six-degree-of-freedom state;
8. `reset()` exactly restores the initial state and time;
9. malformed actions are rejected;
10. non-finite actions are rejected;
11. no production source or persistent result file is modified.

Direct numerical parity with the production integrator will be tested separately after this lifecycle itself passes.

In [4]:
import numpy as np

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)


# ---------------------------------------------------------------------
# Notebook-local Simulator prototype
# ---------------------------------------------------------------------

class NotebookSimulator:
    """
    Minimal notebook-local HydroLab-3D simulator kernel prototype.

    This class intentionally contains orchestration only.
    All physical propagation is delegated to the validated production
    integrate_six_dof_euler() implementation.
    """

    def __init__(
        self,
        *,
        initial_pose,
        initial_body_velocity,
        dt,
        dynamics_kwargs,
    ):
        initial_pose = np.asarray(
            initial_pose,
            dtype=float,
        )

        initial_body_velocity = np.asarray(
            initial_body_velocity,
            dtype=float,
        )

        dt = float(dt)

        if initial_pose.shape != (6,):
            raise ValueError(
                "initial_pose must have shape (6,)"
            )

        if initial_body_velocity.shape != (6,):
            raise ValueError(
                "initial_body_velocity must have shape (6,)"
            )

        if not np.all(
            np.isfinite(initial_pose)
        ):
            raise ValueError(
                "initial_pose must contain only finite values"
            )

        if not np.all(
            np.isfinite(initial_body_velocity)
        ):
            raise ValueError(
                "initial_body_velocity must contain only finite values"
            )

        if not np.isfinite(dt) or dt <= 0.0:
            raise ValueError(
                "dt must be finite and strictly positive"
            )

        if not isinstance(dynamics_kwargs, dict):
            raise TypeError(
                "dynamics_kwargs must be a dictionary"
            )

        # Preserve independent reset references.
        self._initial_pose = initial_pose.copy()

        self._initial_body_velocity = (
            initial_body_velocity.copy()
        )

        self._dt = dt

        # A shallow dictionary copy is sufficient for this prototype.
        # Individual parameter-array ownership will be examined in a
        # dedicated state/configuration robustness step.
        self._dynamics_kwargs = dict(
            dynamics_kwargs
        )

        self._pose = None
        self._body_velocity = None
        self._time = None
        self._diagnostics = None

        self.reset()


    @property
    def dt(self):
        return self._dt


    @property
    def time(self):
        return self._time


    def state(self):
        """
        Return an independent snapshot of simulator ground truth.
        """
        return {
            "time": float(self._time),
            "pose": self._pose.copy(),
            "body_velocity": (
                self._body_velocity.copy()
            ),
        }


    def reset(self):
        """
        Restore the exact configured initial simulator state.
        """
        self._pose = (
            self._initial_pose.copy()
        )

        self._body_velocity = (
            self._initial_body_velocity.copy()
        )

        self._time = 0.0

        self._diagnostics = None

        return self.state()


    def step(self, action):
        """
        Advance exactly one deterministic physics timestep.
        """
        action = np.asarray(
            action,
            dtype=float,
        )

        if action.shape != (6,):
            raise ValueError(
                "action must have shape (6,)"
            )

        if not np.all(
            np.isfinite(action)
        ):
            raise ValueError(
                "action must contain only finite values"
            )

        step_kwargs = dict(
            self._dynamics_kwargs
        )

        step_kwargs["actuator_wrench"] = (
            action.copy()
        )

        (
            next_pose,
            next_body_velocity,
            diagnostics,
        ) = integrate_six_dof_euler(
            self._pose,
            self._body_velocity,
            self._dt,
            **step_kwargs,
        )

        self._pose = (
            np.asarray(
                next_pose,
                dtype=float,
            ).copy()
        )

        self._body_velocity = (
            np.asarray(
                next_body_velocity,
                dtype=float,
            ).copy()
        )

        self._time += self._dt

        self._diagnostics = dict(
            diagnostics
        )

        return self.state()


# ---------------------------------------------------------------------
# Reuse the valid physical configuration established in Step 2
# ---------------------------------------------------------------------

kernel_initial_pose = np.array(
    [
        0.0,
        0.0,
        -5.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

kernel_initial_body_velocity = np.zeros(
    6,
    dtype=float,
)

kernel_dt = 0.02

kernel_dynamics_kwargs = {
    "mass": probe_mass,
    "inertia": probe_inertia.copy(),
    "added_inertia": (
        probe_added_inertia.copy()
    ),
    "linear_damping": (
        probe_linear_damping.copy()
    ),
    "quadratic_damping": (
        probe_quadratic_damping.copy()
    ),
    "fluid_density": (
        probe_fluid_density
    ),
    "displaced_volume": (
        probe_displaced_volume
    ),
    "center_of_gravity": (
        probe_center_of_gravity.copy()
    ),
    "center_of_buoyancy": (
        probe_center_of_buoyancy.copy()
    ),
    "current_velocity_world": (
        probe_current_velocity_world.copy()
    ),
}


# ---------------------------------------------------------------------
# Preserve caller-owned initial arrays
# ---------------------------------------------------------------------

kernel_pose_external = (
    kernel_initial_pose.copy()
)

kernel_velocity_external = (
    kernel_initial_body_velocity.copy()
)


# ---------------------------------------------------------------------
# Construct simulator
# ---------------------------------------------------------------------

kernel_simulator = NotebookSimulator(
    initial_pose=kernel_pose_external,
    initial_body_velocity=(
        kernel_velocity_external
    ),
    dt=kernel_dt,
    dynamics_kwargs=kernel_dynamics_kwargs,
)


# ---------------------------------------------------------------------
# Validate initial state
# ---------------------------------------------------------------------

kernel_state_initial = (
    kernel_simulator.state()
)

assert kernel_state_initial["time"] == 0.0

assert np.array_equal(
    kernel_state_initial["pose"],
    kernel_initial_pose,
)

assert np.array_equal(
    kernel_state_initial["body_velocity"],
    kernel_initial_body_velocity,
)


# ---------------------------------------------------------------------
# Confirm constructor copied caller-owned initial state
# ---------------------------------------------------------------------

kernel_pose_external[0] = 999.0

kernel_velocity_external[0] = 999.0

kernel_state_after_external_mutation = (
    kernel_simulator.state()
)

assert (
    kernel_state_after_external_mutation[
        "pose"
    ][0]
    != 999.0
)

assert (
    kernel_state_after_external_mutation[
        "body_velocity"
    ][0]
    != 999.0
)


# ---------------------------------------------------------------------
# Confirm returned state is also a copy
# ---------------------------------------------------------------------

kernel_snapshot = (
    kernel_simulator.state()
)

kernel_snapshot["pose"][1] = 777.0

kernel_snapshot["body_velocity"][1] = (
    777.0
)

kernel_internal_check = (
    kernel_simulator.state()
)

assert (
    kernel_internal_check["pose"][1]
    != 777.0
)

assert (
    kernel_internal_check[
        "body_velocity"
    ][1]
    != 777.0
)


# ---------------------------------------------------------------------
# Advance one simulator timestep
# ---------------------------------------------------------------------

kernel_action = np.array(
    [
        20.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

kernel_state_after_step = (
    kernel_simulator.step(
        kernel_action
    )
)


# ---------------------------------------------------------------------
# Validate one-step lifecycle
# ---------------------------------------------------------------------

assert np.isclose(
    kernel_state_after_step["time"],
    kernel_dt,
    rtol=0.0,
    atol=1e-15,
)

assert (
    kernel_state_after_step["pose"].shape
    == (6,)
)

assert (
    kernel_state_after_step[
        "body_velocity"
    ].shape
    == (6,)
)

assert np.all(
    np.isfinite(
        kernel_state_after_step["pose"]
    )
)

assert np.all(
    np.isfinite(
        kernel_state_after_step[
            "body_velocity"
        ]
    )
)


# ---------------------------------------------------------------------
# Validate action input protection
# ---------------------------------------------------------------------

kernel_action_before = (
    kernel_action.copy()
)

_ = kernel_simulator.reset()

_ = kernel_simulator.step(
    kernel_action
)

assert np.array_equal(
    kernel_action,
    kernel_action_before,
)


# ---------------------------------------------------------------------
# Validate reset semantics
# ---------------------------------------------------------------------

kernel_reset_state = (
    kernel_simulator.reset()
)

assert kernel_reset_state["time"] == 0.0

assert np.array_equal(
    kernel_reset_state["pose"],
    kernel_initial_pose,
)

assert np.array_equal(
    kernel_reset_state[
        "body_velocity"
    ],
    kernel_initial_body_velocity,
)

assert kernel_simulator._diagnostics is None


# ---------------------------------------------------------------------
# Validate malformed action rejection
# ---------------------------------------------------------------------

try:
    kernel_simulator.step(
        np.zeros(3)
    )
except ValueError as exc:
    wrong_shape_rejected = True
    wrong_shape_message = str(exc)
else:
    wrong_shape_rejected = False
    wrong_shape_message = "<not rejected>"


try:
    invalid_action = np.zeros(
        6,
        dtype=float,
    )

    invalid_action[2] = np.nan

    kernel_simulator.step(
        invalid_action
    )

except ValueError as exc:
    nonfinite_action_rejected = True
    nonfinite_action_message = str(exc)

else:
    nonfinite_action_rejected = False
    nonfinite_action_message = (
        "<not rejected>"
    )


assert wrong_shape_rejected

assert nonfinite_action_rejected


# ---------------------------------------------------------------------
# Concise validation report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 3 MINIMAL KERNEL PROTOTYPE")
print("=" * 80)

print("\nInitial state")
print("-" * 80)

print(
    f"time          : "
    f"{kernel_state_initial['time']:.12f} s"
)

print(
    "pose          :",
    kernel_state_initial["pose"],
)

print(
    "body velocity :",
    kernel_state_initial[
        "body_velocity"
    ],
)


print("\nAfter one step")
print("-" * 80)

print(
    f"time          : "
    f"{kernel_state_after_step['time']:.12f} s"
)

print(
    "pose          :",
    kernel_state_after_step["pose"],
)

print(
    "body velocity :",
    kernel_state_after_step[
        "body_velocity"
    ],
)


print("\nOwnership checks")
print("-" * 80)

print(
    "constructor copied initial pose        : True"
)

print(
    "constructor copied initial velocity    : True"
)

print(
    "state() returned independent arrays    : True"
)

print(
    "step() preserved caller-owned action   : True"
)


print("\nReset checks")
print("-" * 80)

print(
    f"reset time          : "
    f"{kernel_reset_state['time']:.12f} s"
)

print(
    "reset pose exact    :",
    np.array_equal(
        kernel_reset_state["pose"],
        kernel_initial_pose,
    ),
)

print(
    "reset velocity exact:",
    np.array_equal(
        kernel_reset_state[
            "body_velocity"
        ],
        kernel_initial_body_velocity,
    ),
)


print("\nInput-validation checks")
print("-" * 80)

print(
    "wrong-shape action rejected :",
    wrong_shape_rejected,
)

print(
    "  message:",
    wrong_shape_message,
)

print(
    "non-finite action rejected  :",
    nonfinite_action_rejected,
)

print(
    "  message:",
    nonfinite_action_message,
)


print("\n" + "=" * 80)
print("STEP 3 MINIMAL KERNEL PROTOTYPE PASSED")
print("=" * 80)

print(
    "The notebook-local simulator now owns ground-truth state, "
    "advances exactly one production physics timestep per action, "
    "maintains deterministic simulation time, protects state ownership, "
    "and resets exactly to its configured initial condition."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 3 MINIMAL KERNEL PROTOTYPE

Initial state
--------------------------------------------------------------------------------
time          : 0.000000000000 s
pose          : [ 0.  0. -5.  0.  0.  0.]
body velocity : [0. 0. 0. 0. 0. 0.]

After one step
--------------------------------------------------------------------------------
time          : 0.020000000000 s
pose          : [ 0.  0. -5.  0.  0.  0.]
body velocity : [ 7.80727273e-03 -2.22413793e-04  0.00000000e+00  0.00000000e+00
  0.00000000e+00  9.18367347e-05]

Ownership checks
--------------------------------------------------------------------------------
constructor copied initial pose        : True
constructor copied initial velocity    : True
state() returned independent arrays    : True
step() preserved caller-owned action   : True

Reset checks
--------------------------------------------------------------------------------
reset time          : 0.000000000000 s
reset pose exact    : True
rese

## Step 4 - Validate Exact Numerical Parity With the Production 6-DOF Integrator

Step 3 demonstrated that the notebook-local simulator can own state, advance time, validate actions, and reset deterministically.

Those lifecycle properties are necessary, but they are not sufficient.

The simulator kernel must also demonstrate that wrapping the existing HydroLab-3D vehicle model has **not changed the underlying physics**.

The strongest validation is therefore a direct comparison between:

```python
NotebookSimulator.step(action)
```

and:

```python
integrate_six_dof_euler(...)
```

using exactly the same:

```text
initial pose
initial body velocity
timestep
vehicle parameters
ocean current
actuator wrench
```

Conceptually:

```text
                 identical initial condition
                           │
              ┌────────────┴────────────┐
              │                         │
              ▼                         ▼
     NotebookSimulator.step()   integrate_six_dof_euler()
              │                         │
              ▼                         ▼
       simulator result           direct result
              │                         │
              └────────────┬────────────┘
                           ▼
                     parity check
```

The notebook-local kernel contains no alternative numerical integrator.

Its `step()` method delegates directly to the existing production function.

Therefore, for one identical timestep, we expect:

$$
\eta_{kernel}
=
\eta_{direct}
$$

and:

$$
\nu_{kernel}
=
\nu_{direct}
$$

to floating-point identity.

The integration diagnostics should also agree because both paths execute the same production dynamics.

This validation is stronger than merely checking that the two trajectories look similar.

It tests whether the wrapper is truly an orchestration layer rather than an accidental second dynamics implementation.

---

### What Will Be Compared

The validation will compare:

```text
next pose
next body velocity
simulation timestep
integration diagnostic keys
diagnostic scalar values
diagnostic arrays
```

Important diagnostic quantities include:

```text
eta_dot
nu_dot
nu_r
net_wrench
damping_wrench
hydrostatic_wrench
actuator_wrench
mass matrices
Coriolis matrices
```

The simulator's internal diagnostics are inspected here only because this is a notebook-local architecture validation.

They are not yet being declared part of the final public simulator API.

---

### Validation Criteria

This step passes if:

1. both paths begin from exactly the same initial state;
2. both paths receive exactly the same action and physical parameters;
3. the simulator pose equals the direct-integrator pose;
4. the simulator body velocity equals the direct-integrator body velocity;
5. their maximum absolute state differences are zero or at floating-point roundoff;
6. both diagnostic dictionaries contain the same keys;
7. all comparable diagnostic values agree numerically;
8. simulator time advances to exactly one configured timestep;
9. neither comparison path requires modification of production source.

Passing this step establishes the central Notebook 10 architectural invariant:

$$
\boxed{
\text{Simulator.step()} =
\text{orchestration of existing HydroLab-3D physics}
}
$$

In [5]:
import numpy as np

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)


# ---------------------------------------------------------------------
# Establish one identical initial condition for both propagation paths
# ---------------------------------------------------------------------

parity_initial_pose = np.array(
    [
        1.25,
        -0.75,
        -6.0,
        np.deg2rad(4.0),
        np.deg2rad(-3.0),
        np.deg2rad(20.0),
    ],
    dtype=float,
)

parity_initial_body_velocity = np.array(
    [
        0.35,
        -0.08,
        0.04,
        0.015,
        -0.010,
        0.025,
    ],
    dtype=float,
)

parity_action = np.array(
    [
        17.0,
        -3.0,
        4.5,
        0.20,
        -0.15,
        0.35,
    ],
    dtype=float,
)

parity_dt = kernel_dt


# ---------------------------------------------------------------------
# Use the same validated physical configuration from Steps 2 and 3
# ---------------------------------------------------------------------

parity_dynamics_kwargs = {
    "mass": probe_mass,
    "inertia": probe_inertia.copy(),
    "added_inertia": (
        probe_added_inertia.copy()
    ),
    "linear_damping": (
        probe_linear_damping.copy()
    ),
    "quadratic_damping": (
        probe_quadratic_damping.copy()
    ),
    "fluid_density": (
        probe_fluid_density
    ),
    "displaced_volume": (
        probe_displaced_volume
    ),
    "center_of_gravity": (
        probe_center_of_gravity.copy()
    ),
    "center_of_buoyancy": (
        probe_center_of_buoyancy.copy()
    ),
    "current_velocity_world": (
        probe_current_velocity_world.copy()
    ),
}


# ---------------------------------------------------------------------
# Path A:
# propagate through the notebook-local Simulator wrapper
# ---------------------------------------------------------------------

parity_simulator = NotebookSimulator(
    initial_pose=parity_initial_pose,
    initial_body_velocity=(
        parity_initial_body_velocity
    ),
    dt=parity_dt,
    dynamics_kwargs=(
        parity_dynamics_kwargs
    ),
)

parity_kernel_state = (
    parity_simulator.step(
        parity_action
    )
)

parity_kernel_diagnostics = dict(
    parity_simulator._diagnostics
)


# ---------------------------------------------------------------------
# Path B:
# call the production integrator directly
# ---------------------------------------------------------------------

(
    parity_direct_pose,
    parity_direct_body_velocity,
    parity_direct_diagnostics,
) = integrate_six_dof_euler(
    parity_initial_pose,
    parity_initial_body_velocity,
    parity_dt,
    **parity_dynamics_kwargs,
    actuator_wrench=parity_action,
)


# ---------------------------------------------------------------------
# State parity
# ---------------------------------------------------------------------

parity_pose_difference = (
    parity_kernel_state["pose"]
    - parity_direct_pose
)

parity_velocity_difference = (
    parity_kernel_state["body_velocity"]
    - parity_direct_body_velocity
)

parity_pose_max_abs_error = float(
    np.max(
        np.abs(
            parity_pose_difference
        )
    )
)

parity_velocity_max_abs_error = float(
    np.max(
        np.abs(
            parity_velocity_difference
        )
    )
)


# Because both paths call the same deterministic production function
# with identical inputs, exact floating-point equality is expected.
assert np.array_equal(
    parity_kernel_state["pose"],
    parity_direct_pose,
)

assert np.array_equal(
    parity_kernel_state[
        "body_velocity"
    ],
    parity_direct_body_velocity,
)

assert np.isclose(
    parity_kernel_state["time"],
    parity_dt,
    rtol=0.0,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Diagnostic-key parity
# ---------------------------------------------------------------------

kernel_diagnostic_keys = set(
    parity_kernel_diagnostics
)

direct_diagnostic_keys = set(
    parity_direct_diagnostics
)

assert (
    kernel_diagnostic_keys
    == direct_diagnostic_keys
)


# ---------------------------------------------------------------------
# Diagnostic-value parity
# ---------------------------------------------------------------------

diagnostic_parity_results = {}

for key in sorted(
    kernel_diagnostic_keys
):
    kernel_value = (
        parity_kernel_diagnostics[key]
    )

    direct_value = (
        parity_direct_diagnostics[key]
    )

    if isinstance(
        kernel_value,
        np.ndarray,
    ):
        direct_value = np.asarray(
            direct_value
        )

        exact_match = np.array_equal(
            kernel_value,
            direct_value,
        )

        max_abs_error = float(
            np.max(
                np.abs(
                    kernel_value
                    - direct_value
                )
            )
        )

    else:
        exact_match = (
            kernel_value
            == direct_value
        )

        if (
            np.isscalar(kernel_value)
            and np.isscalar(direct_value)
        ):
            max_abs_error = float(
                abs(
                    kernel_value
                    - direct_value
                )
            )
        else:
            max_abs_error = (
                0.0
                if exact_match
                else np.nan
            )

    diagnostic_parity_results[key] = {
        "exact_match": exact_match,
        "max_abs_error": (
            max_abs_error
        ),
    }

    assert exact_match, (
        f"Diagnostic parity failed "
        f"for key: {key}"
    )


# ---------------------------------------------------------------------
# Ensure the wrapper did not mutate the action
# ---------------------------------------------------------------------

parity_action_reference = np.array(
    [
        17.0,
        -3.0,
        4.5,
        0.20,
        -0.15,
        0.35,
    ],
    dtype=float,
)

assert np.array_equal(
    parity_action,
    parity_action_reference,
)


# ---------------------------------------------------------------------
# Report results
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 4 DIRECT PHYSICS PARITY")
print("=" * 80)


print("\nInitial condition")
print("-" * 80)

print(
    "pose          :",
    parity_initial_pose,
)

print(
    "body velocity :",
    parity_initial_body_velocity,
)

print(
    "action        :",
    parity_action,
)

print(
    f"dt            : "
    f"{parity_dt:.12f} s"
)


print("\nKernel result")
print("-" * 80)

print(
    "pose          :",
    parity_kernel_state["pose"],
)

print(
    "body velocity :",
    parity_kernel_state[
        "body_velocity"
    ],
)

print(
    f"time          : "
    f"{parity_kernel_state['time']:.12f} s"
)


print("\nDirect production result")
print("-" * 80)

print(
    "pose          :",
    parity_direct_pose,
)

print(
    "body velocity :",
    parity_direct_body_velocity,
)


print("\nState parity")
print("-" * 80)

print(
    f"maximum |pose difference|     : "
    f"{parity_pose_max_abs_error:.18e}"
)

print(
    f"maximum |velocity difference| : "
    f"{parity_velocity_max_abs_error:.18e}"
)

print(
    "pose exact match              :",
    np.array_equal(
        parity_kernel_state["pose"],
        parity_direct_pose,
    ),
)

print(
    "velocity exact match          :",
    np.array_equal(
        parity_kernel_state[
            "body_velocity"
        ],
        parity_direct_body_velocity,
    ),
)


print("\nDiagnostic parity")
print("-" * 80)

for key in sorted(
    diagnostic_parity_results
):
    result = (
        diagnostic_parity_results[key]
    )

    print(
        f"{key:<32} "
        f"exact={str(result['exact_match']):<5} "
        f"max_abs_error="
        f"{result['max_abs_error']:.3e}"
    )


print("\n" + "=" * 80)
print("STEP 4 DIRECT PHYSICS PARITY PASSED")
print("=" * 80)

print(
    "NotebookSimulator.step() reproduced the direct production "
    "6-DOF integration result exactly for pose, body velocity, "
    "and all returned integration diagnostics."
)

print(
    "The simulator wrapper therefore introduced no additional "
    "physics or numerical behavior in this validated timestep."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 4 DIRECT PHYSICS PARITY

Initial condition
--------------------------------------------------------------------------------
pose          : [ 1.25       -0.75       -6.          0.06981317 -0.05235988  0.34906585]
body velocity : [ 0.35  -0.08   0.04   0.015 -0.01   0.025]
action        : [17.   -3.    4.5   0.2  -0.15  0.35]
dt            : 0.020000000000 s

Kernel result
--------------------------------------------------------------------------------
pose          : [ 1.25710006 -0.74917371 -5.99894815  0.07008776 -0.05259427  0.34955135]
body velocity : [ 0.35526916 -0.0812699   0.0412484  -0.00546087  0.00064668  0.02611528]
time          : 0.020000000000 s

Direct production result
--------------------------------------------------------------------------------
pose          : [ 1.25710006 -0.74917371 -5.99894815  0.07008776 -0.05259427  0.34955135]
body velocity : [ 0.35526916 -0.0812699   0.0412484  -0.00546087  0.00064668  0.02611528]

State parit

## Step 5 - Validate Deterministic Reset and Replay

Step 4 established exact one-step numerical parity between:

```python
NotebookSimulator.step(action)
```

and:

```python
integrate_six_dof_euler(...)
```

for identical initial conditions and inputs.

The next requirement is stronger.

A reusable simulator must support deterministic experiment replay.

Conceptually:

```text
initial state
    ↓
action sequence
    ↓
trajectory A
    ↓
reset()
    ↓
same initial state
    ↓
same action sequence
    ↓
trajectory B
```

For a deterministic configuration:

$$
\boxed{
\text{trajectory A}
=
\text{trajectory B}
}
$$

This property matters for:

```text
controller benchmarking
algorithm comparisons
regression testing
Monte Carlo experiments
learning-environment resets
scientific reproducibility
```

At this stage there is no sensor noise and no stochastic environment component, so exact floating-point replay is expected.

---

### Multi-Step Action Sequence

Instead of applying only one constant force, this validation uses a deterministic sequence containing:

```text
forward thrust
lateral force
vertical force
roll moment
pitch moment
yaw moment
mixed six-axis actuation
zero-action coasting
```

This makes the replay test exercise more of the coupled six-degree-of-freedom model.

At every step we record:

```text
time
pose
body velocity
```

The simulator is then reset and the exact same sequence is replayed.

---

### Time Validation

For step index $k$:

$$
t_k = k\Delta t.
$$

After $N$ actions:

$$
t_N=N\Delta t.
$$

The simulation clock must therefore remain deterministic and independent of wall-clock execution time.

---

### Reset Validation

After the first rollout, calling:

```python
reset()
```

must restore exactly:

```text
time = 0
initial pose
initial body velocity
```

No state from the previous rollout may leak into the replay.

This includes:

```text
position
orientation
linear velocity
angular velocity
simulation time
latest integration diagnostics
```

---

### Replay Criterion

For every timestep:

$$
\eta_k^{(A)}
=
\eta_k^{(B)}
$$

and:

$$
\nu_k^{(A)}
=
\nu_k^{(B)}.
$$

The full stored trajectory arrays should therefore match exactly.

---

### Scope of This Step

This remains a deterministic ground-truth validation.

We are not yet introducing:

```text
sensor noise
random seeds
observation dictionaries
stochastic disturbances
persistent output files
production promotion
```

Those should only be added after the deterministic kernel lifecycle itself is proven reliable.

### Validation Criteria

This step passes if:

1. a nontrivial multi-axis action sequence executes successfully;
2. every state remains finite;
3. time advances exactly by one configured timestep per action;
4. the first rollout finishes with the expected total simulation time;
5. `reset()` restores the exact initial state;
6. replaying the same action sequence produces exactly the same time history;
7. replaying produces exactly the same pose history;
8. replaying produces exactly the same body-velocity history;
9. no state from the first rollout leaks into the second;
10. no production or result file is modified.

Passing this step establishes deterministic simulation replay as a core HydroLab-3D kernel property.

In [6]:
import numpy as np


# ---------------------------------------------------------------------
# Build a deterministic multi-axis action sequence
# ---------------------------------------------------------------------

replay_actions = np.array(
    [
        [12.0,  0.0,  0.0,  0.00,  0.00,  0.00],
        [18.0,  0.0,  0.0,  0.00,  0.00,  0.00],
        [15.0, -3.0,  0.0,  0.00,  0.00,  0.15],
        [10.0, -5.0, -2.0,  0.10,  0.00,  0.20],
        [ 8.0,  0.0, -4.0,  0.15, -0.10,  0.00],
        [ 5.0,  2.0,  3.0, -0.10,  0.12, -0.15],
        [ 0.0,  0.0,  0.0,  0.00,  0.00,  0.00],
        [-4.0,  1.0,  0.0,  0.00,  0.00, -0.10],
        [ 6.0, -2.0,  1.0,  0.05, -0.05,  0.08],
        [ 0.0,  0.0,  0.0,  0.00,  0.00,  0.00],
    ],
    dtype=float,
)

assert replay_actions.ndim == 2
assert replay_actions.shape[1] == 6
assert np.all(np.isfinite(replay_actions))


# ---------------------------------------------------------------------
# Initial condition for replay validation
# ---------------------------------------------------------------------

replay_initial_pose = np.array(
    [
        2.0,
        -1.0,
        -7.0,
        np.deg2rad(6.0),
        np.deg2rad(-4.0),
        np.deg2rad(25.0),
    ],
    dtype=float,
)

replay_initial_body_velocity = np.array(
    [
        0.20,
        -0.03,
        0.02,
        0.010,
        -0.008,
        0.015,
    ],
    dtype=float,
)


# ---------------------------------------------------------------------
# Create simulator
# ---------------------------------------------------------------------

replay_simulator = NotebookSimulator(
    initial_pose=replay_initial_pose,
    initial_body_velocity=(
        replay_initial_body_velocity
    ),
    dt=kernel_dt,
    dynamics_kwargs=(
        parity_dynamics_kwargs
    ),
)


# ---------------------------------------------------------------------
# Rollout helper
# ---------------------------------------------------------------------

def run_replay_rollout(
    simulator,
    actions,
):
    times = [
        simulator.state()["time"]
    ]

    poses = [
        simulator.state()["pose"]
    ]

    velocities = [
        simulator.state()[
            "body_velocity"
        ]
    ]

    for action in actions:
        state = simulator.step(
            action
        )

        assert np.all(
            np.isfinite(
                state["pose"]
            )
        )

        assert np.all(
            np.isfinite(
                state["body_velocity"]
            )
        )

        times.append(
            state["time"]
        )

        poses.append(
            state["pose"]
        )

        velocities.append(
            state["body_velocity"]
        )

    return (
        np.asarray(
            times,
            dtype=float,
        ),
        np.asarray(
            poses,
            dtype=float,
        ),
        np.asarray(
            velocities,
            dtype=float,
        ),
    )


# ---------------------------------------------------------------------
# Rollout A
# ---------------------------------------------------------------------

(
    replay_times_a,
    replay_poses_a,
    replay_velocities_a,
) = run_replay_rollout(
    replay_simulator,
    replay_actions,
)


# ---------------------------------------------------------------------
# Validate time progression for rollout A
# ---------------------------------------------------------------------

expected_replay_times = (
    np.arange(
        len(replay_actions) + 1,
        dtype=float,
    )
    * kernel_dt
)

assert np.allclose(
    replay_times_a,
    expected_replay_times,
    rtol=0.0,
    atol=1e-15,
)

expected_final_time = (
    len(replay_actions)
    * kernel_dt
)

assert np.isclose(
    replay_times_a[-1],
    expected_final_time,
    rtol=0.0,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Reset between rollouts
# ---------------------------------------------------------------------

replay_reset_state = (
    replay_simulator.reset()
)

assert (
    replay_reset_state["time"]
    == 0.0
)

assert np.array_equal(
    replay_reset_state["pose"],
    replay_initial_pose,
)

assert np.array_equal(
    replay_reset_state[
        "body_velocity"
    ],
    replay_initial_body_velocity,
)

assert (
    replay_simulator._diagnostics
    is None
)


# ---------------------------------------------------------------------
# Rollout B
# ---------------------------------------------------------------------

(
    replay_times_b,
    replay_poses_b,
    replay_velocities_b,
) = run_replay_rollout(
    replay_simulator,
    replay_actions,
)


# ---------------------------------------------------------------------
# Exact deterministic replay checks
# ---------------------------------------------------------------------

time_exact_match = np.array_equal(
    replay_times_a,
    replay_times_b,
)

pose_exact_match = np.array_equal(
    replay_poses_a,
    replay_poses_b,
)

velocity_exact_match = (
    np.array_equal(
        replay_velocities_a,
        replay_velocities_b,
    )
)


assert time_exact_match
assert pose_exact_match
assert velocity_exact_match


# ---------------------------------------------------------------------
# Numerical difference diagnostics
# ---------------------------------------------------------------------

maximum_time_difference = float(
    np.max(
        np.abs(
            replay_times_a
            - replay_times_b
        )
    )
)

maximum_pose_difference = float(
    np.max(
        np.abs(
            replay_poses_a
            - replay_poses_b
        )
    )
)

maximum_velocity_difference = float(
    np.max(
        np.abs(
            replay_velocities_a
            - replay_velocities_b
        )
    )
)


# ---------------------------------------------------------------------
# Check final state remains finite
# ---------------------------------------------------------------------

assert np.all(
    np.isfinite(
        replay_poses_a
    )
)

assert np.all(
    np.isfinite(
        replay_velocities_a
    )
)


# ---------------------------------------------------------------------
# Compact validation report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 5 RESET / REPLAY VALIDATION")
print("=" * 80)

print("\nRollout configuration")
print("-" * 80)

print(
    f"number of actions : "
    f"{len(replay_actions)}"
)

print(
    f"dt                : "
    f"{kernel_dt:.12f} s"
)

print(
    f"expected duration : "
    f"{expected_final_time:.12f} s"
)


print("\nTime validation")
print("-" * 80)

print(
    f"rollout A final time : "
    f"{replay_times_a[-1]:.12f} s"
)

print(
    f"rollout B final time : "
    f"{replay_times_b[-1]:.12f} s"
)

print(
    "time sequence exact match :",
    time_exact_match,
)


print("\nReset validation")
print("-" * 80)

print(
    "reset time exact     :",
    replay_reset_state["time"]
    == 0.0,
)

print(
    "reset pose exact     :",
    np.array_equal(
        replay_reset_state["pose"],
        replay_initial_pose,
    ),
)

print(
    "reset velocity exact :",
    np.array_equal(
        replay_reset_state[
            "body_velocity"
        ],
        replay_initial_body_velocity,
    ),
)


print("\nReplay parity")
print("-" * 80)

print(
    "pose trajectory exact     :",
    pose_exact_match,
)

print(
    "velocity trajectory exact :",
    velocity_exact_match,
)

print(
    f"maximum |time difference|     : "
    f"{maximum_time_difference:.18e}"
)

print(
    f"maximum |pose difference|     : "
    f"{maximum_pose_difference:.18e}"
)

print(
    f"maximum |velocity difference| : "
    f"{maximum_velocity_difference:.18e}"
)


print("\nFinal state")
print("-" * 80)

print(
    "final pose          :",
    replay_poses_a[-1],
)

print(
    "final body velocity :",
    replay_velocities_a[-1],
)


print("\n" + "=" * 80)
print("STEP 5 RESET / REPLAY VALIDATION PASSED")
print("=" * 80)

print(
    "Reset restored the exact configured initial condition."
)

print(
    "Replaying the identical action sequence produced an exactly "
    "identical time, pose, and body-velocity history."
)

print(
    "The notebook-local kernel is therefore deterministic across "
    "multi-step reset/replay cycles for this non-stochastic configuration."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 5 RESET / REPLAY VALIDATION

Rollout configuration
--------------------------------------------------------------------------------
number of actions : 10
dt                : 0.020000000000 s
expected duration : 0.200000000000 s

Time validation
--------------------------------------------------------------------------------
rollout A final time : 0.200000000000 s
rollout B final time : 0.200000000000 s
time sequence exact match : True

Reset validation
--------------------------------------------------------------------------------
reset time exact     : True
reset pose exact     : True
reset velocity exact : True

Replay parity
--------------------------------------------------------------------------------
pose trajectory exact     : True
velocity trajectory exact : True
maximum |time difference|     : 0.000000000000000000e+00
maximum |pose difference|     : 0.000000000000000000e+00
maximum |velocity difference| : 0.000000000000000000e+00

Final state


## Step 6 - Validate Ocean-Current Influence and Longer-Rollout Stability

Steps 3 through 5 established the core simulator lifecycle:

```text
state ownership
deterministic time
one-step propagation
direct production parity
exact reset
exact replay
```

The next requirement is to confirm that the simulator kernel correctly propagates environmental configuration into the existing HydroLab-3D dynamics.

The most important current environmental input is:

```python
current_velocity_world
```

which was validated in earlier notebooks and is already consumed by the six-degree-of-freedom production model.

Notebook 10 must not implement a new current model.

It only needs to verify that different configured currents produce different physical evolution through the inherited production dynamics.

---

### Experimental Comparison

Two otherwise identical simulator instances will be created.

#### Case A - Still Water

```text
current_velocity_world = [0, 0, 0]
```

#### Case B - Uniform Current

```text
current_velocity_world = [0.4, -0.15, 0]
```

Both simulations will use:

```text
identical initial pose
identical initial body velocity
identical vehicle parameters
identical timestep
zero actuator wrench
identical rollout duration
```

The only changed quantity is the ocean current.

Conceptually:

```text
                  identical UUV
                       │
             ┌─────────┴─────────┐
             │                   │
             ▼                   ▼
        still water          ocean current
             │                   │
             ▼                   ▼
       rollout A             rollout B
             │                   │
             └─────────┬─────────┘
                       ▼
                 compare states
```

If the simulator is wiring environmental inputs correctly, the two trajectories should diverge.

---

### Why Zero Action Is Useful

The actuator wrench will be:

$$
\tau=0.
$$

This prevents commanded thrust from hiding the effect of the current.

The vehicle may still accelerate because hydrodynamic forces depend on velocity relative to the surrounding water.

The production model computes water-relative motion through quantities such as:

```text
nu_r
```

and uses that relative state in the inherited hydrodynamic terms.

Therefore the environmental current affects dynamics without the simulator kernel needing to modify the equations.

---

### Longer-Rollout Stability

Previous validation used only short sequences.

This step will run substantially more integration steps to test that the wrapper remains numerically well behaved over a longer horizon.

For every step we require:

$$
\eta_k\in\mathbb{R}^6
$$

and:

$$
\nu_k\in\mathbb{R}^6
$$

with every component finite.

The simulation time must also satisfy:

$$
t_N=N\Delta t.
$$

No:

```text
NaN
Inf
shape corruption
time corruption
unexpected exception
```

should occur in the validated operating regime.

---

### Expected Physical Result

The two cases are not expected to produce identical trajectories.

We expect:

$$
\eta_{current}(t)
\neq
\eta_{still}(t)
$$

and generally:

$$
\nu_{current}(t)
\neq
\nu_{still}(t).
$$

The exact magnitude is not prescribed in advance because it emerges from the inherited six-degree-of-freedom model.

The purpose is not to impose a desired answer on the physics.

The purpose is to confirm that:

```text
changing only current
        ↓
changes the resulting physical evolution
```

while both simulations remain finite and deterministic.

---

### Validation Criteria

This step passes if:

1. two identical simulator configurations can be run with different current vectors;
2. both complete the full rollout without exceptions;
3. all poses remain finite;
4. all body velocities remain finite;
5. both time histories advance correctly;
6. the still-water and current trajectories differ;
7. the still-water and current body-velocity histories differ;
8. each rollout remains exactly reproducible after reset;
9. the difference arises only from the configured current;
10. no production or result file is modified.

Passing this step demonstrates that the simulator kernel is not merely advancing vehicle state but is correctly preserving the inherited world-to-dynamics interaction.

In [7]:
import numpy as np


# ---------------------------------------------------------------------
# Longer-rollout configuration
# ---------------------------------------------------------------------

current_validation_steps = 500

current_validation_dt = kernel_dt

current_validation_duration = (
    current_validation_steps
    * current_validation_dt
)

zero_action = np.zeros(
    6,
    dtype=float,
)


# ---------------------------------------------------------------------
# Identical initial vehicle state
# ---------------------------------------------------------------------

current_validation_initial_pose = np.array(
    [
        0.0,
        0.0,
        -8.0,
        0.0,
        0.0,
        np.deg2rad(15.0),
    ],
    dtype=float,
)

current_validation_initial_velocity = np.zeros(
    6,
    dtype=float,
)


# ---------------------------------------------------------------------
# Two world-current configurations
# ---------------------------------------------------------------------

still_water_velocity = np.array(
    [
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

uniform_current_velocity = np.array(
    [
        0.40,
        -0.15,
        0.0,
    ],
    dtype=float,
)


# ---------------------------------------------------------------------
# Build shared physical configuration
# ---------------------------------------------------------------------

def make_current_validation_kwargs(
    current_velocity_world,
):
    return {
        "mass": probe_mass,
        "inertia": (
            probe_inertia.copy()
        ),
        "added_inertia": (
            probe_added_inertia.copy()
        ),
        "linear_damping": (
            probe_linear_damping.copy()
        ),
        "quadratic_damping": (
            probe_quadratic_damping.copy()
        ),
        "fluid_density": (
            probe_fluid_density
        ),
        "displaced_volume": (
            probe_displaced_volume
        ),
        "center_of_gravity": (
            probe_center_of_gravity.copy()
        ),
        "center_of_buoyancy": (
            probe_center_of_buoyancy.copy()
        ),
        "current_velocity_world": (
            np.asarray(
                current_velocity_world,
                dtype=float,
            ).copy()
        ),
    }


still_water_kwargs = (
    make_current_validation_kwargs(
        still_water_velocity
    )
)

uniform_current_kwargs = (
    make_current_validation_kwargs(
        uniform_current_velocity
    )
)


# ---------------------------------------------------------------------
# Create both simulator instances
# ---------------------------------------------------------------------

still_water_simulator = NotebookSimulator(
    initial_pose=(
        current_validation_initial_pose
    ),
    initial_body_velocity=(
        current_validation_initial_velocity
    ),
    dt=current_validation_dt,
    dynamics_kwargs=still_water_kwargs,
)

uniform_current_simulator = NotebookSimulator(
    initial_pose=(
        current_validation_initial_pose
    ),
    initial_body_velocity=(
        current_validation_initial_velocity
    ),
    dt=current_validation_dt,
    dynamics_kwargs=uniform_current_kwargs,
)


# ---------------------------------------------------------------------
# Rollout helper
# ---------------------------------------------------------------------

def run_zero_action_rollout(
    simulator,
    number_of_steps,
):
    times = np.empty(
        number_of_steps + 1,
        dtype=float,
    )

    poses = np.empty(
        (
            number_of_steps + 1,
            6,
        ),
        dtype=float,
    )

    velocities = np.empty(
        (
            number_of_steps + 1,
            6,
        ),
        dtype=float,
    )

    initial_state = simulator.state()

    times[0] = initial_state["time"]

    poses[0] = initial_state["pose"]

    velocities[0] = (
        initial_state[
            "body_velocity"
        ]
    )

    for step_index in range(
        number_of_steps
    ):
        state = simulator.step(
            zero_action
        )

        assert state["pose"].shape == (6,)

        assert (
            state["body_velocity"].shape
            == (6,)
        )

        assert np.all(
            np.isfinite(
                state["pose"]
            )
        )

        assert np.all(
            np.isfinite(
                state[
                    "body_velocity"
                ]
            )
        )

        assert np.isfinite(
            state["time"]
        )

        times[
            step_index + 1
        ] = state["time"]

        poses[
            step_index + 1
        ] = state["pose"]

        velocities[
            step_index + 1
        ] = state[
            "body_velocity"
        ]

    return (
        times,
        poses,
        velocities,
    )


# ---------------------------------------------------------------------
# Run still-water case
# ---------------------------------------------------------------------

(
    still_times_a,
    still_poses_a,
    still_velocities_a,
) = run_zero_action_rollout(
    still_water_simulator,
    current_validation_steps,
)


# ---------------------------------------------------------------------
# Run uniform-current case
# ---------------------------------------------------------------------

(
    current_times_a,
    current_poses_a,
    current_velocities_a,
) = run_zero_action_rollout(
    uniform_current_simulator,
    current_validation_steps,
)


# ---------------------------------------------------------------------
# Validate time progression
# ---------------------------------------------------------------------

expected_current_validation_times = (
    np.arange(
        current_validation_steps + 1,
        dtype=float,
    )
    * current_validation_dt
)

assert np.allclose(
    still_times_a,
    expected_current_validation_times,
    rtol=0.0,
    atol=1e-12,
)

assert np.allclose(
    current_times_a,
    expected_current_validation_times,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    still_times_a[-1],
    current_validation_duration,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    current_times_a[-1],
    current_validation_duration,
    rtol=0.0,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# Confirm current changes the physical trajectory
# ---------------------------------------------------------------------

final_pose_difference = (
    current_poses_a[-1]
    - still_poses_a[-1]
)

final_velocity_difference = (
    current_velocities_a[-1]
    - still_velocities_a[-1]
)

position_difference_norm = float(
    np.linalg.norm(
        final_pose_difference[:3]
    )
)

velocity_difference_norm = float(
    np.linalg.norm(
        final_velocity_difference[:3]
    )
)

trajectory_max_abs_difference = float(
    np.max(
        np.abs(
            current_poses_a
            - still_poses_a
        )
    )
)

velocity_history_max_abs_difference = float(
    np.max(
        np.abs(
            current_velocities_a
            - still_velocities_a
        )
    )
)


assert position_difference_norm > 0.0

assert velocity_difference_norm > 0.0

assert trajectory_max_abs_difference > 0.0

assert (
    velocity_history_max_abs_difference
    > 0.0
)


# ---------------------------------------------------------------------
# Replay both cases after reset
# ---------------------------------------------------------------------

still_water_simulator.reset()

uniform_current_simulator.reset()


(
    still_times_b,
    still_poses_b,
    still_velocities_b,
) = run_zero_action_rollout(
    still_water_simulator,
    current_validation_steps,
)

(
    current_times_b,
    current_poses_b,
    current_velocities_b,
) = run_zero_action_rollout(
    uniform_current_simulator,
    current_validation_steps,
)


# ---------------------------------------------------------------------
# Exact deterministic replay checks
# ---------------------------------------------------------------------

still_replay_exact = (
    np.array_equal(
        still_times_a,
        still_times_b,
    )
    and np.array_equal(
        still_poses_a,
        still_poses_b,
    )
    and np.array_equal(
        still_velocities_a,
        still_velocities_b,
    )
)

current_replay_exact = (
    np.array_equal(
        current_times_a,
        current_times_b,
    )
    and np.array_equal(
        current_poses_a,
        current_poses_b,
    )
    and np.array_equal(
        current_velocities_a,
        current_velocities_b,
    )
)


assert still_replay_exact

assert current_replay_exact


# ---------------------------------------------------------------------
# Finite-state checks over complete histories
# ---------------------------------------------------------------------

assert np.all(
    np.isfinite(
        still_poses_a
    )
)

assert np.all(
    np.isfinite(
        still_velocities_a
    )
)

assert np.all(
    np.isfinite(
        current_poses_a
    )
)

assert np.all(
    np.isfinite(
        current_velocities_a
    )
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 6 CURRENT / STABILITY VALIDATION")
print("=" * 80)


print("\nRollout configuration")
print("-" * 80)

print(
    f"steps             : "
    f"{current_validation_steps}"
)

print(
    f"dt                : "
    f"{current_validation_dt:.12f} s"
)

print(
    f"duration          : "
    f"{current_validation_duration:.12f} s"
)

print(
    "still-water current :",
    still_water_velocity,
)

print(
    "uniform current     :",
    uniform_current_velocity,
)


print("\nStill-water final state")
print("-" * 80)

print(
    "pose          :",
    still_poses_a[-1],
)

print(
    "body velocity :",
    still_velocities_a[-1],
)


print("\nUniform-current final state")
print("-" * 80)

print(
    "pose          :",
    current_poses_a[-1],
)

print(
    "body velocity :",
    current_velocities_a[-1],
)


print("\nEnvironmental influence")
print("-" * 80)

print(
    f"final XYZ separation            : "
    f"{position_difference_norm:.12f} m"
)

print(
    f"final linear-velocity difference: "
    f"{velocity_difference_norm:.12f} m/s"
)

print(
    f"max |pose-history difference|   : "
    f"{trajectory_max_abs_difference:.12e}"
)

print(
    f"max |velocity-history difference|: "
    f"{velocity_history_max_abs_difference:.12e}"
)


print("\nTime validation")
print("-" * 80)

print(
    f"still-water final time : "
    f"{still_times_a[-1]:.12f} s"
)

print(
    f"current final time     : "
    f"{current_times_a[-1]:.12f} s"
)


print("\nReplay validation")
print("-" * 80)

print(
    "still-water replay exact :",
    still_replay_exact,
)

print(
    "current replay exact     :",
    current_replay_exact,
)


print("\nFinite-state validation")
print("-" * 80)

print(
    "still-water pose history finite     :",
    np.all(
        np.isfinite(
            still_poses_a
        )
    ),
)

print(
    "still-water velocity history finite :",
    np.all(
        np.isfinite(
            still_velocities_a
        )
    ),
)

print(
    "current pose history finite         :",
    np.all(
        np.isfinite(
            current_poses_a
        )
    ),
)

print(
    "current velocity history finite     :",
    np.all(
        np.isfinite(
            current_velocities_a
        )
    ),
)


print("\n" + "=" * 80)
print("STEP 6 CURRENT / STABILITY VALIDATION PASSED")
print("=" * 80)

print(
    "The simulator completed the longer zero-action rollouts "
    "without non-finite states or time corruption."
)

print(
    "Changing only current_velocity_world produced a different "
    "physical trajectory and body-velocity history."
)

print(
    "Both environmental configurations remained exactly reproducible "
    "after reset."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 6 CURRENT / STABILITY VALIDATION

Rollout configuration
--------------------------------------------------------------------------------
steps             : 500
dt                : 0.020000000000 s
duration          : 10.000000000000 s
still-water current : [0. 0. 0.]
uniform current     : [ 0.4  -0.15  0.  ]

Still-water final state
--------------------------------------------------------------------------------
pose          : [ 0.          0.         -8.          0.          0.          0.26179939]
body velocity : [0. 0. 0. 0. 0. 0.]

Uniform-current final state
--------------------------------------------------------------------------------
pose          : [ 2.29868266 -1.05162815 -8.          0.          0.          0.4776761 ]
body velocity : [ 0.2360373  -0.28663428  0.          0.          0.          0.00511041]

Environmental influence
--------------------------------------------------------------------------------
final XYZ separation          

## Step 7 - Integrate a Reproducible Sensor-Observation Layer

Steps 3 through 6 established a deterministic simulator kernel with:

```text
state ownership
reset
step
deterministic simulation time
exact production-model parity
multi-step replay
ocean-current propagation
longer-rollout finite-state stability
```

Notebook 08 established a second architectural boundary that Notebook 10 must preserve:

$$
\boxed{
\text{simulator truth}
\neq
\text{sensor observation}
}
$$

The simulator internally knows the physical state required to propagate the model.

An autonomy algorithm may instead receive imperfect measurements derived from that truth.

Notebook 10 must therefore verify that sensor generation can be attached to the kernel without allowing noisy measurements to overwrite physical state.

---

### Sensor Inputs Already Available

Step 2 confirmed the production sensor contracts.

Depth requires:

```text
pose
```

Heading requires:

```text
pose
```

DVL requires:

```text
pose
body velocity
world-frame current
```

Gyroscope requires:

```text
body velocity
```

Accelerometer requires:

```text
pose
body velocity
body acceleration
```

The production integrator already returns:

```text
diagnostics["nu_dot"]
```

after every physical step.

Therefore the post-step sensor path can be:

```text
Simulator.step(action)
        ↓
production 6-DOF integration
        ↓
post-step truth
+
nu_dot diagnostics
        ↓
production sensor functions
        ↓
observation
```

No second dynamics calculation is required.

---

### Observation Contract

For this notebook-local prototype, a post-step observation will contain:

```text
depth
heading
dvl
gyroscope
accelerometer
```

The simulator truth remains separately available as:

```text
time
pose
body_velocity
```

Conceptually:

```text
                 Simulator
                     │
          ┌──────────┴──────────┐
          │                     │
          ▼                     ▼
     ground truth          observations
          │                     │
      pose, nu             depth
        time               heading
                           DVL
                           gyro
                           accelerometer
```

No observation value may be fed back into the internal truth state.

---

### Explicit Randomness

Notebook 08 established that sensor randomness must use an explicit:

```python
numpy.random.Generator
```

rather than global NumPy random state.

The sensor-enabled prototype will therefore own an RNG created from a configured seed:

```python
np.random.default_rng(seed)
```

This gives the simulator a reproducible stochastic lifecycle.

If:

```text
same initial state
same action sequence
same sensor configuration
same sensor seed
```

are used, then the noisy observation sequence should reproduce exactly after reset.

---

### RNG Reset Semantics

A simulator reset should restore not only physical truth but also the configured stochastic starting condition.

Therefore:

```python
reset()
```

will reconstruct the sensor RNG from the original seed.

This means:

```text
run A
    ↓
reset()
    ↓
run B
```

with identical actions should reproduce:

```text
identical truth trajectory
identical noisy observation trajectory
```

when the same configured seed is retained.

A different seed should preserve identical physical truth while producing a different noisy measurement sequence.

This distinction is essential:

$$
\boxed{
\text{sensor seed changes observations, not physics}
}
$$

---

### Sensor Configuration

This validation will use small, explicit imperfections for each sensor.

Representative settings will include:

```text
depth bias and Gaussian noise
heading bias and Gaussian noise
DVL bias and Gaussian noise
gyroscope bias and Gaussian noise
accelerometer bias and Gaussian noise
```

The values are not intended to model one specific real UUV sensor suite.

They exist to validate:

```text
observation plumbing
truth separation
explicit RNG ownership
seeded reproducibility
```

---

### Reset Observation Boundary

The accelerometer requires the instantaneous body acceleration:

```text
nu_dot
```

which becomes naturally available after a physics step.

Notebook 10 will therefore keep this first sensor contract deliberately conservative:

```text
reset()
    → returns ground truth

step(action)
    → advances physics
    → generates post-step observation
```

This avoids inventing an arbitrary accelerometer acceleration at reset time.

A future higher-level environment may define its own initial observation policy if required.

Notebook 11 does not require the simulator core to fabricate physics merely to satisfy a particular ML API.

---

### Validation Criteria

This step passes if:

1. post-step observations can be generated using only production sensor functions;
2. observations contain depth, heading, DVL, gyroscope, and accelerometer measurements;
3. observation values remain finite;
4. noisy measurements do not modify physical pose;
5. noisy measurements do not modify physical body velocity;
6. two identical seeded runs produce exactly identical truth trajectories;
7. two identical seeded runs produce exactly identical observation trajectories;
8. changing only the sensor seed leaves physical truth unchanged;
9. changing only the sensor seed changes at least one noisy observation;
10. reset restores the original physical state and stochastic sequence;
11. no production or persistent result file is modified.

Passing this step establishes a clean simulator boundary between authoritative physical state and reproducible imperfect sensing.

In [8]:
import numpy as np

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)

from hydrolab3d.sensors import (
    depth_measurement,
    heading_measurement,
    dvl_measurement,
    gyroscope_measurement,
    accelerometer_measurement,
)


# ---------------------------------------------------------------------
# Sensor-enabled notebook-local Simulator prototype
# ---------------------------------------------------------------------

class NotebookSensorSimulator:
    """
    Notebook-local HydroLab-3D simulator with optional production sensors.

    Physics remains delegated entirely to integrate_six_dof_euler().
    Sensor measurements remain separate from simulator ground truth.
    """

    def __init__(
        self,
        *,
        initial_pose,
        initial_body_velocity,
        dt,
        dynamics_kwargs,
        sensor_config=None,
        sensor_seed=None,
    ):
        initial_pose = np.asarray(
            initial_pose,
            dtype=float,
        )

        initial_body_velocity = np.asarray(
            initial_body_velocity,
            dtype=float,
        )

        dt = float(dt)

        if initial_pose.shape != (6,):
            raise ValueError(
                "initial_pose must have shape (6,)"
            )

        if initial_body_velocity.shape != (6,):
            raise ValueError(
                "initial_body_velocity must have shape (6,)"
            )

        if not np.all(np.isfinite(initial_pose)):
            raise ValueError(
                "initial_pose must contain only finite values"
            )

        if not np.all(
            np.isfinite(initial_body_velocity)
        ):
            raise ValueError(
                "initial_body_velocity must contain only finite values"
            )

        if not np.isfinite(dt) or dt <= 0.0:
            raise ValueError(
                "dt must be finite and strictly positive"
            )

        if not isinstance(dynamics_kwargs, dict):
            raise TypeError(
                "dynamics_kwargs must be a dictionary"
            )

        if sensor_config is None:
            sensor_config = {}

        if not isinstance(sensor_config, dict):
            raise TypeError(
                "sensor_config must be a dictionary"
            )

        self._initial_pose = (
            initial_pose.copy()
        )

        self._initial_body_velocity = (
            initial_body_velocity.copy()
        )

        self._dt = dt

        self._dynamics_kwargs = dict(
            dynamics_kwargs
        )

        self._sensor_config = dict(
            sensor_config
        )

        self._sensor_seed = sensor_seed

        self._pose = None
        self._body_velocity = None
        self._time = None
        self._diagnostics = None
        self._rng = None

        self.reset()


    @property
    def dt(self):
        return self._dt


    @property
    def time(self):
        return self._time


    def state(self):
        """
        Return an independent ground-truth snapshot.
        """
        return {
            "time": float(self._time),
            "pose": self._pose.copy(),
            "body_velocity": (
                self._body_velocity.copy()
            ),
        }


    def reset(self):
        """
        Restore physical truth and the configured sensor RNG sequence.
        """
        self._pose = (
            self._initial_pose.copy()
        )

        self._body_velocity = (
            self._initial_body_velocity.copy()
        )

        self._time = 0.0

        self._diagnostics = None

        self._rng = np.random.default_rng(
            self._sensor_seed
        )

        return self.state()


    def _sensor_settings(
        self,
        sensor_name,
    ):
        settings = self._sensor_config.get(
            sensor_name,
            {},
        )

        if not isinstance(settings, dict):
            raise TypeError(
                f"sensor_config['{sensor_name}'] "
                "must be a dictionary"
            )

        return dict(settings)


    def _generate_observation(
        self,
        body_acceleration,
    ):
        """
        Generate post-step measurements from physical truth.

        This method never modifies pose or body velocity.
        """
        body_acceleration = np.asarray(
            body_acceleration,
            dtype=float,
        )

        if body_acceleration.shape != (6,):
            raise ValueError(
                "body_acceleration must have shape (6,)"
            )

        if not np.all(
            np.isfinite(body_acceleration)
        ):
            raise ValueError(
                "body_acceleration must contain only finite values"
            )

        current_velocity_world = np.asarray(
            self._dynamics_kwargs[
                "current_velocity_world"
            ],
            dtype=float,
        )

        depth_settings = (
            self._sensor_settings(
                "depth"
            )
        )

        heading_settings = (
            self._sensor_settings(
                "heading"
            )
        )

        dvl_settings = (
            self._sensor_settings(
                "dvl"
            )
        )

        gyro_settings = (
            self._sensor_settings(
                "gyroscope"
            )
        )

        accel_settings = (
            self._sensor_settings(
                "accelerometer"
            )
        )

        depth = depth_measurement(
            self._pose,
            rng=self._rng,
            **depth_settings,
        )

        heading = heading_measurement(
            self._pose,
            rng=self._rng,
            **heading_settings,
        )

        dvl = dvl_measurement(
            self._pose,
            self._body_velocity,
            current_velocity_world,
            rng=self._rng,
            **dvl_settings,
        )

        gyroscope = gyroscope_measurement(
            self._body_velocity,
            rng=self._rng,
            **gyro_settings,
        )

        accelerometer = (
            accelerometer_measurement(
                self._pose,
                self._body_velocity,
                body_acceleration,
                rng=self._rng,
                **accel_settings,
            )
        )

        return {
            "depth": float(depth),
            "heading": float(heading),
            "dvl": np.asarray(
                dvl,
                dtype=float,
            ).copy(),
            "gyroscope": np.asarray(
                gyroscope,
                dtype=float,
            ).copy(),
            "accelerometer": np.asarray(
                accelerometer,
                dtype=float,
            ).copy(),
        }


    def step(self, action):
        """
        Advance one timestep and generate a post-step observation.
        """
        action = np.asarray(
            action,
            dtype=float,
        )

        if action.shape != (6,):
            raise ValueError(
                "action must have shape (6,)"
            )

        if not np.all(np.isfinite(action)):
            raise ValueError(
                "action must contain only finite values"
            )

        step_kwargs = dict(
            self._dynamics_kwargs
        )

        step_kwargs[
            "actuator_wrench"
        ] = action.copy()

        (
            next_pose,
            next_body_velocity,
            diagnostics,
        ) = integrate_six_dof_euler(
            self._pose,
            self._body_velocity,
            self._dt,
            **step_kwargs,
        )

        self._pose = np.asarray(
            next_pose,
            dtype=float,
        ).copy()

        self._body_velocity = np.asarray(
            next_body_velocity,
            dtype=float,
        ).copy()

        self._time += self._dt

        self._diagnostics = dict(
            diagnostics
        )

        body_acceleration = np.asarray(
            self._diagnostics["nu_dot"],
            dtype=float,
        )

        observation = (
            self._generate_observation(
                body_acceleration
            )
        )

        return {
            "truth": self.state(),
            "observation": observation,
        }


# ---------------------------------------------------------------------
# Explicit sensor configuration
# ---------------------------------------------------------------------

sensor_validation_config = {
    "depth": {
        "bias": 0.03,
        "noise_std": 0.04,
    },
    "heading": {
        "bias": np.deg2rad(0.75),
        "noise_std": np.deg2rad(1.00),
    },
    "dvl": {
        "bias": np.array(
            [0.01, -0.01, 0.005],
            dtype=float,
        ),
        "noise_std": 0.015,
    },
    "gyroscope": {
        "bias": np.array(
            [0.001, -0.001, 0.0005],
            dtype=float,
        ),
        "noise_std": 0.002,
    },
    "accelerometer": {
        "bias": np.array(
            [0.02, -0.01, 0.03],
            dtype=float,
        ),
        "noise_std": 0.025,
    },
}


# ---------------------------------------------------------------------
# Validation initial condition and action sequence
# ---------------------------------------------------------------------

sensor_validation_initial_pose = np.array(
    [
        0.5,
        -0.25,
        -6.0,
        np.deg2rad(3.0),
        np.deg2rad(-2.0),
        np.deg2rad(12.0),
    ],
    dtype=float,
)

sensor_validation_initial_velocity = np.array(
    [
        0.10,
        -0.02,
        0.01,
        0.005,
        -0.004,
        0.008,
    ],
    dtype=float,
)

sensor_validation_actions = np.array(
    [
        [10.0,  0.0,  0.0,  0.00,  0.00,  0.00],
        [12.0, -1.5,  0.0,  0.02,  0.00,  0.08],
        [ 8.0,  0.0, -2.0,  0.00, -0.04,  0.00],
        [ 6.0,  2.0,  1.0, -0.03,  0.02, -0.06],
        [ 0.0,  0.0,  0.0,  0.00,  0.00,  0.00],
        [ 5.0, -1.0,  0.5,  0.01, -0.01,  0.04],
    ],
    dtype=float,
)


# ---------------------------------------------------------------------
# Dynamics configuration
# ---------------------------------------------------------------------

sensor_validation_dynamics = {
    "mass": probe_mass,
    "inertia": probe_inertia.copy(),
    "added_inertia": (
        probe_added_inertia.copy()
    ),
    "linear_damping": (
        probe_linear_damping.copy()
    ),
    "quadratic_damping": (
        probe_quadratic_damping.copy()
    ),
    "fluid_density": (
        probe_fluid_density
    ),
    "displaced_volume": (
        probe_displaced_volume
    ),
    "center_of_gravity": (
        probe_center_of_gravity.copy()
    ),
    "center_of_buoyancy": (
        probe_center_of_buoyancy.copy()
    ),
    "current_velocity_world": np.array(
        [0.20, -0.05, 0.0],
        dtype=float,
    ),
}


# ---------------------------------------------------------------------
# Rollout helper
# ---------------------------------------------------------------------

def run_sensor_validation_rollout(
    simulator,
    actions,
):
    truth_times = []
    truth_poses = []
    truth_velocities = []

    observation_depth = []
    observation_heading = []
    observation_dvl = []
    observation_gyro = []
    observation_accel = []

    for action in actions:
        result = simulator.step(
            action
        )

        truth = result["truth"]
        observation = (
            result["observation"]
        )

        assert np.isfinite(
            observation["depth"]
        )

        assert np.isfinite(
            observation["heading"]
        )

        assert observation["dvl"].shape == (3,)
        assert observation["gyroscope"].shape == (3,)
        assert observation["accelerometer"].shape == (3,)

        assert np.all(
            np.isfinite(
                observation["dvl"]
            )
        )

        assert np.all(
            np.isfinite(
                observation["gyroscope"]
            )
        )

        assert np.all(
            np.isfinite(
                observation["accelerometer"]
            )
        )

        truth_times.append(
            truth["time"]
        )

        truth_poses.append(
            truth["pose"]
        )

        truth_velocities.append(
            truth["body_velocity"]
        )

        observation_depth.append(
            observation["depth"]
        )

        observation_heading.append(
            observation["heading"]
        )

        observation_dvl.append(
            observation["dvl"]
        )

        observation_gyro.append(
            observation["gyroscope"]
        )

        observation_accel.append(
            observation["accelerometer"]
        )

    return {
        "times": np.asarray(
            truth_times,
            dtype=float,
        ),
        "poses": np.asarray(
            truth_poses,
            dtype=float,
        ),
        "velocities": np.asarray(
            truth_velocities,
            dtype=float,
        ),
        "depth": np.asarray(
            observation_depth,
            dtype=float,
        ),
        "heading": np.asarray(
            observation_heading,
            dtype=float,
        ),
        "dvl": np.asarray(
            observation_dvl,
            dtype=float,
        ),
        "gyroscope": np.asarray(
            observation_gyro,
            dtype=float,
        ),
        "accelerometer": np.asarray(
            observation_accel,
            dtype=float,
        ),
    }


# ---------------------------------------------------------------------
# Seeded run A
# ---------------------------------------------------------------------

sensor_seed_a = 202610

sensor_simulator_a = (
    NotebookSensorSimulator(
        initial_pose=(
            sensor_validation_initial_pose
        ),
        initial_body_velocity=(
            sensor_validation_initial_velocity
        ),
        dt=kernel_dt,
        dynamics_kwargs=(
            sensor_validation_dynamics
        ),
        sensor_config=(
            sensor_validation_config
        ),
        sensor_seed=sensor_seed_a,
    )
)

sensor_run_a = (
    run_sensor_validation_rollout(
        sensor_simulator_a,
        sensor_validation_actions,
    )
)


# ---------------------------------------------------------------------
# Reset and replay with the same simulator / same seed
# ---------------------------------------------------------------------

sensor_simulator_a.reset()

sensor_run_a_replay = (
    run_sensor_validation_rollout(
        sensor_simulator_a,
        sensor_validation_actions,
    )
)


# ---------------------------------------------------------------------
# Exact same-seed replay checks
# ---------------------------------------------------------------------

same_seed_truth_exact = (
    np.array_equal(
        sensor_run_a["times"],
        sensor_run_a_replay["times"],
    )
    and np.array_equal(
        sensor_run_a["poses"],
        sensor_run_a_replay["poses"],
    )
    and np.array_equal(
        sensor_run_a["velocities"],
        sensor_run_a_replay["velocities"],
    )
)

same_seed_observations_exact = (
    np.array_equal(
        sensor_run_a["depth"],
        sensor_run_a_replay["depth"],
    )
    and np.array_equal(
        sensor_run_a["heading"],
        sensor_run_a_replay["heading"],
    )
    and np.array_equal(
        sensor_run_a["dvl"],
        sensor_run_a_replay["dvl"],
    )
    and np.array_equal(
        sensor_run_a["gyroscope"],
        sensor_run_a_replay["gyroscope"],
    )
    and np.array_equal(
        sensor_run_a["accelerometer"],
        sensor_run_a_replay["accelerometer"],
    )
)

assert same_seed_truth_exact
assert same_seed_observations_exact


# ---------------------------------------------------------------------
# Run B with a different sensor seed
# ---------------------------------------------------------------------

sensor_seed_b = 202611

sensor_simulator_b = (
    NotebookSensorSimulator(
        initial_pose=(
            sensor_validation_initial_pose
        ),
        initial_body_velocity=(
            sensor_validation_initial_velocity
        ),
        dt=kernel_dt,
        dynamics_kwargs=(
            sensor_validation_dynamics
        ),
        sensor_config=(
            sensor_validation_config
        ),
        sensor_seed=sensor_seed_b,
    )
)

sensor_run_b = (
    run_sensor_validation_rollout(
        sensor_simulator_b,
        sensor_validation_actions,
    )
)


# ---------------------------------------------------------------------
# Different seed must not change physical truth
# ---------------------------------------------------------------------

different_seed_truth_exact = (
    np.array_equal(
        sensor_run_a["times"],
        sensor_run_b["times"],
    )
    and np.array_equal(
        sensor_run_a["poses"],
        sensor_run_b["poses"],
    )
    and np.array_equal(
        sensor_run_a["velocities"],
        sensor_run_b["velocities"],
    )
)

assert different_seed_truth_exact


# ---------------------------------------------------------------------
# Different seed should change noisy measurements
# ---------------------------------------------------------------------

depth_seed_difference = float(
    np.max(
        np.abs(
            sensor_run_a["depth"]
            - sensor_run_b["depth"]
        )
    )
)

heading_seed_difference = float(
    np.max(
        np.abs(
            sensor_run_a["heading"]
            - sensor_run_b["heading"]
        )
    )
)

dvl_seed_difference = float(
    np.max(
        np.abs(
            sensor_run_a["dvl"]
            - sensor_run_b["dvl"]
        )
    )
)

gyro_seed_difference = float(
    np.max(
        np.abs(
            sensor_run_a["gyroscope"]
            - sensor_run_b["gyroscope"]
        )
    )
)

accel_seed_difference = float(
    np.max(
        np.abs(
            sensor_run_a["accelerometer"]
            - sensor_run_b["accelerometer"]
        )
    )
)

different_seed_observation_changed = any(
    difference > 0.0
    for difference in [
        depth_seed_difference,
        heading_seed_difference,
        dvl_seed_difference,
        gyro_seed_difference,
        accel_seed_difference,
    ]
)

assert (
    different_seed_observation_changed
)


# ---------------------------------------------------------------------
# Verify observation generation did not corrupt physical truth
#
# Compare the noisy-sensor run against the deterministic ground-truth
# NotebookSimulator using identical physics and actions.
# ---------------------------------------------------------------------

truth_reference_simulator = (
    NotebookSimulator(
        initial_pose=(
            sensor_validation_initial_pose
        ),
        initial_body_velocity=(
            sensor_validation_initial_velocity
        ),
        dt=kernel_dt,
        dynamics_kwargs=(
            sensor_validation_dynamics
        ),
    )
)

truth_reference_times = []
truth_reference_poses = []
truth_reference_velocities = []

for action in sensor_validation_actions:
    state = truth_reference_simulator.step(
        action
    )

    truth_reference_times.append(
        state["time"]
    )

    truth_reference_poses.append(
        state["pose"]
    )

    truth_reference_velocities.append(
        state["body_velocity"]
    )

truth_reference_times = np.asarray(
    truth_reference_times,
    dtype=float,
)

truth_reference_poses = np.asarray(
    truth_reference_poses,
    dtype=float,
)

truth_reference_velocities = np.asarray(
    truth_reference_velocities,
    dtype=float,
)


truth_preserved_with_sensors = (
    np.array_equal(
        sensor_run_a["times"],
        truth_reference_times,
    )
    and np.array_equal(
        sensor_run_a["poses"],
        truth_reference_poses,
    )
    and np.array_equal(
        sensor_run_a["velocities"],
        truth_reference_velocities,
    )
)

assert truth_preserved_with_sensors


# ---------------------------------------------------------------------
# Compact validation report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 7 SENSOR INTEGRATION VALIDATION")
print("=" * 80)


print("\nSensor configuration")
print("-" * 80)

print(
    f"seed A : {sensor_seed_a}"
)

print(
    f"seed B : {sensor_seed_b}"
)

print(
    f"number of actions : "
    f"{len(sensor_validation_actions)}"
)


print("\nTruth / observation separation")
print("-" * 80)

print(
    "sensor-enabled truth matches "
    "sensor-free physics exactly :",
    truth_preserved_with_sensors,
)

print(
    "different sensor seeds preserve "
    "identical physical truth       :",
    different_seed_truth_exact,
)


print("\nSame-seed reset / replay")
print("-" * 80)

print(
    "truth trajectory exact       :",
    same_seed_truth_exact,
)

print(
    "observation sequence exact   :",
    same_seed_observations_exact,
)


print("\nDifferent-seed observation differences")
print("-" * 80)

print(
    f"max depth difference         : "
    f"{depth_seed_difference:.12e}"
)

print(
    f"max heading difference       : "
    f"{heading_seed_difference:.12e}"
)

print(
    f"max DVL difference           : "
    f"{dvl_seed_difference:.12e}"
)

print(
    f"max gyroscope difference     : "
    f"{gyro_seed_difference:.12e}"
)

print(
    f"max accelerometer difference : "
    f"{accel_seed_difference:.12e}"
)

print(
    "at least one noisy observation "
    "changed with seed           :",
    different_seed_observation_changed,
)


print("\nExample final truth")
print("-" * 80)

print(
    "pose          :",
    sensor_run_a["poses"][-1],
)

print(
    "body velocity :",
    sensor_run_a[
        "velocities"
    ][-1],
)


print("\nExample final noisy observation")
print("-" * 80)

print(
    f"depth         : "
    f"{sensor_run_a['depth'][-1]:.12f} m"
)

print(
    f"heading       : "
    f"{sensor_run_a['heading'][-1]:.12f} rad"
)

print(
    "DVL           :",
    sensor_run_a["dvl"][-1],
)

print(
    "gyroscope     :",
    sensor_run_a[
        "gyroscope"
    ][-1],
)

print(
    "accelerometer :",
    sensor_run_a[
        "accelerometer"
    ][-1],
)


print("\n" + "=" * 80)
print("STEP 7 SENSOR INTEGRATION VALIDATION PASSED")
print("=" * 80)

print(
    "Production sensor models were integrated as derived post-step "
    "observations without modifying simulator ground truth."
)

print(
    "Reset restored both the exact physical initial condition and the "
    "configured sensor random sequence."
)

print(
    "Changing only the sensor seed changed noisy observations while "
    "leaving the physical trajectory exactly unchanged."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 7 SENSOR INTEGRATION VALIDATION

Sensor configuration
--------------------------------------------------------------------------------
seed A : 202610
seed B : 202611
number of actions : 6

Truth / observation separation
--------------------------------------------------------------------------------
sensor-enabled truth matches sensor-free physics exactly : True
different sensor seeds preserve identical physical truth       : True

Same-seed reset / replay
--------------------------------------------------------------------------------
truth trajectory exact       : True
observation sequence exact   : True

Different-seed observation differences
--------------------------------------------------------------------------------
max depth difference         : 8.114349013883e-02
max heading difference       : 6.221347384324e-02
max DVL difference           : 4.079808158342e-02
max gyroscope difference     : 6.110311108955e-03
max accelerometer difference : 5.

## Step 8 - Validate Configuration Ownership and Eliminate Hidden Aliasing

The simulator kernel now correctly owns its physical state and sensor random sequence.

However, a second ownership boundary remains:

```text
configuration
```

The notebook-local prototypes currently receive dictionaries containing arrays such as:

```text
inertia
added inertia
linear damping
quadratic damping
center of gravity
center of buoyancy
current velocity
sensor biases
```

A shallow dictionary copy such as:

```python
dict(configuration)
```

copies only the dictionary container.

It does **not** necessarily copy mutable NumPy arrays stored inside that dictionary.

This creates a potential failure mode:

```text
caller creates simulator
        ↓
simulator stores references to configuration arrays
        ↓
caller later modifies original array
        ↓
simulator physics silently changes
```

That behavior would make experiments difficult to reproduce and would violate the simulator's state-ownership philosophy.

The production kernel should therefore own an independent copy of configuration values that materially affect simulation behavior.

---

### Desired Ownership Rule

After simulator construction:

$$
\boxed{
\text{external mutation of configuration}
\not\Rightarrow
\text{internal simulator mutation}
}
$$

This applies to:

```text
dynamics parameter arrays
current vectors
sensor bias arrays
sensor-noise configuration
```

Likewise, configuration exposed externally in the future should not provide writable references to simulator-owned arrays.

---

### Why This Matters

Consider:

```python
current = np.array([0.2, 0.0, 0.0])

simulator = Simulator(
    dynamics_kwargs={
        "current_velocity_world": current,
        ...
    }
)

current[:] = 1000.0
```

The simulator should not suddenly experience a thousand-meter-per-second ocean current.

The configuration supplied during construction should define the simulator unless an explicit future configuration-update API is intentionally introduced.

Silent mutation through shared NumPy memory is therefore unacceptable.

---

### Strategy

This step will first demonstrate whether the existing notebook-local implementation is vulnerable to nested mutable configuration aliasing.

Then a corrected notebook-local prototype will introduce explicit configuration copying.

The copied values will preserve normal immutable scalars while independently copying mutable arrays.

The corrected simulator will retain all previously validated behavior:

```text
reset
step
deterministic time
physics parity
sensor truth separation
seeded replay
```

while strengthening configuration ownership.

---

### Validation Criteria

This step passes if:

1. dynamics configuration arrays are independently owned by the simulator;
2. mutating caller-owned current velocity after construction does not alter simulator behavior;
3. mutating caller-owned damping or inertia arrays does not alter simulator behavior;
4. sensor bias arrays are independently owned;
5. mutating caller-owned sensor configuration after construction does not change the configured observation sequence;
6. two simulators created from equivalent configurations still produce identical results;
7. state snapshots remain independent copies;
8. reset and seeded replay remain deterministic;
9. no production source or result file is modified.

Passing this step means the notebook-local simulator architecture is sufficiently hardened for production promotion.

In [9]:
import copy
import numpy as np

from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)

from hydrolab3d.sensors import (
    depth_measurement,
    heading_measurement,
    dvl_measurement,
    gyroscope_measurement,
    accelerometer_measurement,
)


# ---------------------------------------------------------------------
# Configuration-copy helper
# ---------------------------------------------------------------------

def copy_configuration(value):
    """
    Recursively copy simulator configuration.

    NumPy arrays receive independent storage.
    Dictionaries are recursively copied.
    Lists and tuples are recursively copied.
    Immutable scalar-like values are deep-copied safely.
    """

    if isinstance(value, np.ndarray):
        return value.copy()

    if isinstance(value, dict):
        return {
            key: copy_configuration(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            copy_configuration(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            copy_configuration(item)
            for item in value
        )

    return copy.deepcopy(value)


# ---------------------------------------------------------------------
# Hardened notebook-local simulator
# ---------------------------------------------------------------------

class NotebookHardenedSimulator:
    """
    Notebook-local HydroLab-3D simulator prototype with:

    - deterministic 6-DOF stepping
    - reset
    - copied truth-state snapshots
    - independently owned dynamics configuration
    - optional reproducible production sensors
    """

    def __init__(
        self,
        *,
        initial_pose,
        initial_body_velocity,
        dt,
        dynamics_kwargs,
        sensor_config=None,
        sensor_seed=None,
    ):
        initial_pose = np.asarray(
            initial_pose,
            dtype=float,
        )

        initial_body_velocity = np.asarray(
            initial_body_velocity,
            dtype=float,
        )

        dt = float(dt)

        if initial_pose.shape != (6,):
            raise ValueError(
                "initial_pose must have shape (6,)"
            )

        if initial_body_velocity.shape != (6,):
            raise ValueError(
                "initial_body_velocity must have shape (6,)"
            )

        if not np.all(
            np.isfinite(initial_pose)
        ):
            raise ValueError(
                "initial_pose must contain only finite values"
            )

        if not np.all(
            np.isfinite(initial_body_velocity)
        ):
            raise ValueError(
                "initial_body_velocity must contain only finite values"
            )

        if not np.isfinite(dt) or dt <= 0.0:
            raise ValueError(
                "dt must be finite and strictly positive"
            )

        if not isinstance(dynamics_kwargs, dict):
            raise TypeError(
                "dynamics_kwargs must be a dictionary"
            )

        if sensor_config is None:
            sensor_config = {}

        if not isinstance(sensor_config, dict):
            raise TypeError(
                "sensor_config must be a dictionary"
            )

        self._initial_pose = (
            initial_pose.copy()
        )

        self._initial_body_velocity = (
            initial_body_velocity.copy()
        )

        self._dt = dt

        self._dynamics_kwargs = (
            copy_configuration(
                dynamics_kwargs
            )
        )

        self._sensor_config = (
            copy_configuration(
                sensor_config
            )
        )

        self._sensor_seed = (
            copy.deepcopy(
                sensor_seed
            )
        )

        self._pose = None
        self._body_velocity = None
        self._time = None
        self._diagnostics = None
        self._rng = None

        self.reset()


    @property
    def dt(self):
        return self._dt


    @property
    def time(self):
        return self._time


    def state(self):
        return {
            "time": float(self._time),
            "pose": self._pose.copy(),
            "body_velocity": (
                self._body_velocity.copy()
            ),
        }


    def reset(self):
        self._pose = (
            self._initial_pose.copy()
        )

        self._body_velocity = (
            self._initial_body_velocity.copy()
        )

        self._time = 0.0

        self._diagnostics = None

        self._rng = np.random.default_rng(
            self._sensor_seed
        )

        return self.state()


    def _sensor_settings(
        self,
        sensor_name,
    ):
        settings = self._sensor_config.get(
            sensor_name,
            {},
        )

        if not isinstance(settings, dict):
            raise TypeError(
                f"sensor_config['{sensor_name}'] "
                "must be a dictionary"
            )

        return copy_configuration(
            settings
        )


    def _generate_observation(
        self,
        body_acceleration,
    ):
        body_acceleration = np.asarray(
            body_acceleration,
            dtype=float,
        )

        if body_acceleration.shape != (6,):
            raise ValueError(
                "body_acceleration must have shape (6,)"
            )

        if not np.all(
            np.isfinite(
                body_acceleration
            )
        ):
            raise ValueError(
                "body_acceleration must contain only finite values"
            )

        current_velocity_world = np.asarray(
            self._dynamics_kwargs[
                "current_velocity_world"
            ],
            dtype=float,
        )

        depth_settings = (
            self._sensor_settings(
                "depth"
            )
        )

        heading_settings = (
            self._sensor_settings(
                "heading"
            )
        )

        dvl_settings = (
            self._sensor_settings(
                "dvl"
            )
        )

        gyro_settings = (
            self._sensor_settings(
                "gyroscope"
            )
        )

        accel_settings = (
            self._sensor_settings(
                "accelerometer"
            )
        )

        return {
            "depth": float(
                depth_measurement(
                    self._pose,
                    rng=self._rng,
                    **depth_settings,
                )
            ),
            "heading": float(
                heading_measurement(
                    self._pose,
                    rng=self._rng,
                    **heading_settings,
                )
            ),
            "dvl": np.asarray(
                dvl_measurement(
                    self._pose,
                    self._body_velocity,
                    current_velocity_world,
                    rng=self._rng,
                    **dvl_settings,
                ),
                dtype=float,
            ).copy(),
            "gyroscope": np.asarray(
                gyroscope_measurement(
                    self._body_velocity,
                    rng=self._rng,
                    **gyro_settings,
                ),
                dtype=float,
            ).copy(),
            "accelerometer": np.asarray(
                accelerometer_measurement(
                    self._pose,
                    self._body_velocity,
                    body_acceleration,
                    rng=self._rng,
                    **accel_settings,
                ),
                dtype=float,
            ).copy(),
        }


    def step(self, action):
        action = np.asarray(
            action,
            dtype=float,
        )

        if action.shape != (6,):
            raise ValueError(
                "action must have shape (6,)"
            )

        if not np.all(
            np.isfinite(action)
        ):
            raise ValueError(
                "action must contain only finite values"
            )

        step_kwargs = (
            copy_configuration(
                self._dynamics_kwargs
            )
        )

        step_kwargs[
            "actuator_wrench"
        ] = action.copy()

        (
            next_pose,
            next_body_velocity,
            diagnostics,
        ) = integrate_six_dof_euler(
            self._pose,
            self._body_velocity,
            self._dt,
            **step_kwargs,
        )

        self._pose = np.asarray(
            next_pose,
            dtype=float,
        ).copy()

        self._body_velocity = np.asarray(
            next_body_velocity,
            dtype=float,
        ).copy()

        self._time += self._dt

        self._diagnostics = (
            copy_configuration(
                diagnostics
            )
        )

        observation = (
            self._generate_observation(
                self._diagnostics[
                    "nu_dot"
                ]
            )
        )

        return {
            "truth": self.state(),
            "observation": (
                copy_configuration(
                    observation
                )
            ),
        }


# ---------------------------------------------------------------------
# Caller-owned configuration objects
# ---------------------------------------------------------------------

ownership_current = np.array(
    [0.20, -0.05, 0.0],
    dtype=float,
)

ownership_inertia = (
    probe_inertia.copy()
)

ownership_linear_damping = (
    probe_linear_damping.copy()
)

ownership_depth_bias = 0.03

ownership_dvl_bias = np.array(
    [0.01, -0.01, 0.005],
    dtype=float,
)


ownership_dynamics = {
    "mass": probe_mass,
    "inertia": ownership_inertia,
    "added_inertia": (
        probe_added_inertia.copy()
    ),
    "linear_damping": (
        ownership_linear_damping
    ),
    "quadratic_damping": (
        probe_quadratic_damping.copy()
    ),
    "fluid_density": (
        probe_fluid_density
    ),
    "displaced_volume": (
        probe_displaced_volume
    ),
    "center_of_gravity": (
        probe_center_of_gravity.copy()
    ),
    "center_of_buoyancy": (
        probe_center_of_buoyancy.copy()
    ),
    "current_velocity_world": (
        ownership_current
    ),
}


ownership_sensor_config = {
    "depth": {
        "bias": ownership_depth_bias,
        "noise_std": 0.04,
    },
    "heading": {
        "bias": np.deg2rad(0.75),
        "noise_std": np.deg2rad(1.0),
    },
    "dvl": {
        "bias": ownership_dvl_bias,
        "noise_std": 0.015,
    },
    "gyroscope": {
        "bias": np.array(
            [0.001, -0.001, 0.0005],
            dtype=float,
        ),
        "noise_std": 0.002,
    },
    "accelerometer": {
        "bias": np.array(
            [0.02, -0.01, 0.03],
            dtype=float,
        ),
        "noise_std": 0.025,
    },
}


# ---------------------------------------------------------------------
# Independent untouched reference configuration
# ---------------------------------------------------------------------

ownership_reference_dynamics = (
    copy_configuration(
        ownership_dynamics
    )
)

ownership_reference_sensors = (
    copy_configuration(
        ownership_sensor_config
    )
)


# ---------------------------------------------------------------------
# Construct simulator BEFORE mutating caller-owned objects
# ---------------------------------------------------------------------

ownership_initial_pose = np.array(
    [
        0.0,
        0.0,
        -6.0,
        0.02,
        -0.01,
        0.15,
    ],
    dtype=float,
)

ownership_initial_velocity = np.array(
    [
        0.10,
        0.0,
        0.0,
        0.0,
        0.0,
        0.01,
    ],
    dtype=float,
)

ownership_seed = 8801

ownership_simulator = (
    NotebookHardenedSimulator(
        initial_pose=(
            ownership_initial_pose
        ),
        initial_body_velocity=(
            ownership_initial_velocity
        ),
        dt=kernel_dt,
        dynamics_kwargs=(
            ownership_dynamics
        ),
        sensor_config=(
            ownership_sensor_config
        ),
        sensor_seed=ownership_seed,
    )
)


# ---------------------------------------------------------------------
# Construct independent reference simulator
# ---------------------------------------------------------------------

ownership_reference_simulator = (
    NotebookHardenedSimulator(
        initial_pose=(
            ownership_initial_pose
        ),
        initial_body_velocity=(
            ownership_initial_velocity
        ),
        dt=kernel_dt,
        dynamics_kwargs=(
            ownership_reference_dynamics
        ),
        sensor_config=(
            ownership_reference_sensors
        ),
        sensor_seed=ownership_seed,
    )
)


# ---------------------------------------------------------------------
# Mutate the original caller-owned configuration aggressively
# ---------------------------------------------------------------------

ownership_current[:] = [
    999.0,
    999.0,
    999.0,
]

ownership_inertia[:] = [
    999.0,
    999.0,
    999.0,
]

ownership_linear_damping[:] = (
    999.0
)

ownership_dvl_bias[:] = [
    999.0,
    999.0,
    999.0,
]

ownership_sensor_config[
    "depth"
]["bias"] = 999.0

ownership_sensor_config[
    "depth"
]["noise_std"] = 999.0


# ---------------------------------------------------------------------
# Run both simulators using identical actions
# ---------------------------------------------------------------------

ownership_actions = np.array(
    [
        [8.0, 0.0, 0.0, 0.00, 0.00, 0.00],
        [6.0, 1.0, 0.0, 0.02, 0.00, 0.05],
        [4.0, 0.0, -1.0, 0.00, -0.03, 0.00],
        [0.0, 0.0, 0.0, 0.00, 0.00, 0.00],
    ],
    dtype=float,
)


def run_ownership_rollout(
    simulator,
    actions,
):
    truth_times = []
    truth_poses = []
    truth_velocities = []

    depth = []
    heading = []
    dvl = []
    gyro = []
    accel = []

    for action in actions:
        result = simulator.step(
            action
        )

        truth = result["truth"]
        observation = (
            result["observation"]
        )

        truth_times.append(
            truth["time"]
        )

        truth_poses.append(
            truth["pose"]
        )

        truth_velocities.append(
            truth["body_velocity"]
        )

        depth.append(
            observation["depth"]
        )

        heading.append(
            observation["heading"]
        )

        dvl.append(
            observation["dvl"]
        )

        gyro.append(
            observation["gyroscope"]
        )

        accel.append(
            observation["accelerometer"]
        )

    return {
        "times": np.asarray(
            truth_times,
            dtype=float,
        ),
        "poses": np.asarray(
            truth_poses,
            dtype=float,
        ),
        "velocities": np.asarray(
            truth_velocities,
            dtype=float,
        ),
        "depth": np.asarray(
            depth,
            dtype=float,
        ),
        "heading": np.asarray(
            heading,
            dtype=float,
        ),
        "dvl": np.asarray(
            dvl,
            dtype=float,
        ),
        "gyroscope": np.asarray(
            gyro,
            dtype=float,
        ),
        "accelerometer": np.asarray(
            accel,
            dtype=float,
        ),
    }


ownership_run = (
    run_ownership_rollout(
        ownership_simulator,
        ownership_actions,
    )
)

ownership_reference_run = (
    run_ownership_rollout(
        ownership_reference_simulator,
        ownership_actions,
    )
)


# ---------------------------------------------------------------------
# Exact parity proves external mutations did not leak inward
# ---------------------------------------------------------------------

ownership_truth_exact = (
    np.array_equal(
        ownership_run["times"],
        ownership_reference_run[
            "times"
        ],
    )
    and np.array_equal(
        ownership_run["poses"],
        ownership_reference_run[
            "poses"
        ],
    )
    and np.array_equal(
        ownership_run["velocities"],
        ownership_reference_run[
            "velocities"
        ],
    )
)

ownership_observations_exact = (
    np.array_equal(
        ownership_run["depth"],
        ownership_reference_run[
            "depth"
        ],
    )
    and np.array_equal(
        ownership_run["heading"],
        ownership_reference_run[
            "heading"
        ],
    )
    and np.array_equal(
        ownership_run["dvl"],
        ownership_reference_run[
            "dvl"
        ],
    )
    and np.array_equal(
        ownership_run["gyroscope"],
        ownership_reference_run[
            "gyroscope"
        ],
    )
    and np.array_equal(
        ownership_run[
            "accelerometer"
        ],
        ownership_reference_run[
            "accelerometer"
        ],
    )
)


assert ownership_truth_exact

assert ownership_observations_exact


# ---------------------------------------------------------------------
# Reset / replay after caller mutation
# ---------------------------------------------------------------------

ownership_simulator.reset()

ownership_replay = (
    run_ownership_rollout(
        ownership_simulator,
        ownership_actions,
    )
)

ownership_replay_exact = (
    np.array_equal(
        ownership_run["times"],
        ownership_replay["times"],
    )
    and np.array_equal(
        ownership_run["poses"],
        ownership_replay["poses"],
    )
    and np.array_equal(
        ownership_run["velocities"],
        ownership_replay["velocities"],
    )
    and np.array_equal(
        ownership_run["depth"],
        ownership_replay["depth"],
    )
    and np.array_equal(
        ownership_run["heading"],
        ownership_replay["heading"],
    )
    and np.array_equal(
        ownership_run["dvl"],
        ownership_replay["dvl"],
    )
    and np.array_equal(
        ownership_run["gyroscope"],
        ownership_replay[
            "gyroscope"
        ],
    )
    and np.array_equal(
        ownership_run[
            "accelerometer"
        ],
        ownership_replay[
            "accelerometer"
        ],
    )
)

assert ownership_replay_exact


# ---------------------------------------------------------------------
# Snapshot-copy check remains valid
# ---------------------------------------------------------------------

ownership_snapshot = (
    ownership_simulator.state()
)

ownership_snapshot["pose"][0] = (
    123456.0
)

ownership_snapshot[
    "body_velocity"
][0] = 123456.0

ownership_snapshot_check = (
    ownership_simulator.state()
)

snapshot_still_independent = (
    ownership_snapshot_check[
        "pose"
    ][0]
    != 123456.0
    and ownership_snapshot_check[
        "body_velocity"
    ][0]
    != 123456.0
)

assert snapshot_still_independent


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 8 CONFIGURATION OWNERSHIP")
print("=" * 80)


print("\nCaller mutations applied after simulator construction")
print("-" * 80)

print(
    "external current now        :",
    ownership_current,
)

print(
    "external inertia now        :",
    ownership_inertia,
)

print(
    "external linear damping now :",
    ownership_linear_damping,
)

print(
    "external DVL bias now       :",
    ownership_dvl_bias,
)

print(
    "external depth config now   :",
    ownership_sensor_config[
        "depth"
    ],
)


print("\nOwnership validation")
print("-" * 80)

print(
    "physical truth unaffected by external config mutation :",
    ownership_truth_exact,
)

print(
    "sensor sequence unaffected by external config mutation:",
    ownership_observations_exact,
)

print(
    "reset/replay still exact                              :",
    ownership_replay_exact,
)

print(
    "state snapshot still independent                     :",
    snapshot_still_independent,
)


print("\nInternal physical outcome")
print("-" * 80)

print(
    "final pose          :",
    ownership_run["poses"][-1],
)

print(
    "final body velocity :",
    ownership_run[
        "velocities"
    ][-1],
)

print(
    f"final depth observation : "
    f"{ownership_run['depth'][-1]:.12f}"
)


print("\n" + "=" * 80)
print("STEP 8 CONFIGURATION OWNERSHIP PASSED")
print("=" * 80)

print(
    "The hardened notebook-local simulator owns independent copies "
    "of mutable dynamics and sensor configuration."
)

print(
    "Aggressive mutation of caller-owned configuration after "
    "construction did not alter physics, observations, or replay."
)

print(
    "The notebook-local kernel is now sufficiently isolated for "
    "production promotion."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 8 CONFIGURATION OWNERSHIP

Caller mutations applied after simulator construction
--------------------------------------------------------------------------------
external current now        : [999. 999. 999.]
external inertia now        : [999. 999. 999.]
external linear damping now : [999. 999. 999. 999. 999. 999.]
external DVL bias now       : [999. 999. 999.]
external depth config now   : {'bias': 999.0, 'noise_std': 999.0}

Ownership validation
--------------------------------------------------------------------------------
physical truth unaffected by external config mutation : True
sensor sequence unaffected by external config mutation: True
reset/replay still exact                              : True
state snapshot still independent                     : True

Internal physical outcome
--------------------------------------------------------------------------------
final pose          : [ 8.23657109e-03  1.21443550e-03 -5.99992421e+00  1.92518195e-

## Step 9 - Promote the Validated Simulator Kernel Into Production Source

Steps 3 through 8 validated the simulator architecture notebook-locally.

The prototype demonstrated:

```text
state ownership
configuration ownership
deterministic reset
deterministic simulation time
one-step propagation
exact direct-physics parity
multi-step replay
ocean-current propagation
finite longer rollouts
sensor integration
truth / observation separation
seeded sensor reproducibility
```

The simulator kernel is therefore ready for production promotion.

The repository already contains:

```text
src/hydrolab3d/core/
├── __init__.py
├── clock.py
├── entity.py
├── simulator.py
└── state.py
```

Step 1 established that these files are empty scaffolds.

Notebook 10 will promote only the abstractions that have been validated.

This step will therefore modify only:

```text
src/hydrolab3d/core/simulator.py
src/hydrolab3d/core/__init__.py
```

while leaving:

```text
clock.py
state.py
entity.py
```

untouched.

---

### Production `Simulator`

The production class will own:

```text
initial pose
initial body velocity
current pose
current body velocity
simulation time
fixed timestep
dynamics configuration
sensor configuration
sensor RNG
latest integration diagnostics
```

Its public lifecycle is:

```python
simulator = Simulator(...)

state = simulator.reset()

result = simulator.step(action)
```

where:

```text
reset()
```

returns ground truth:

```text
time
pose
body_velocity
```

and:

```text
step(action)
```

returns:

```text
truth
observation
```

with:

```text
truth
├── time
├── pose
└── body_velocity

observation
├── depth
├── heading
├── dvl
├── gyroscope
└── accelerometer
```

---

### Physics Boundary

The production simulator will continue to delegate all physical propagation to:

```python
integrate_six_dof_euler(...)
```

The `Simulator` class will not implement its own:

```text
kinematics
mass matrices
Coriolis terms
hydrodynamic damping
hydrostatics
current-relative dynamics
integration equations
sensor equations
```

The kernel remains an orchestration layer.

---

### Ownership Boundary

The validated recursive configuration-copy strategy will also be promoted.

This preserves:

$$
\boxed{
\text{caller-owned mutation}
\not\Rightarrow
\text{simulator mutation}
}
$$

for both:

```text
state
configuration
```

---

### Randomness

Sensor randomness remains explicit through:

```python
np.random.default_rng(sensor_seed)
```

Calling:

```python
reset()
```

reconstructs the configured random sequence.

Thus identical:

```text
initial state
configuration
action sequence
sensor seed
```

remain reproducible.

---

### Scope Discipline

This step does **not** implement:

```text
clock.py
state.py
entity.py
controller ownership
reward logic
Gymnasium
RL
rendering
keyboard control
ROS 2
multi-UUV management
sonar
localization
mission planning
```

Those remain outside the validated Notebook 10 kernel.

---

### Validation Criteria

This step passes if:

1. `src/hydrolab3d/core/simulator.py` is populated with the validated simulator implementation;
2. `src/hydrolab3d/core/__init__.py` exposes `Simulator`;
3. `clock.py`, `state.py`, and `entity.py` remain unchanged;
4. the promoted files can be inspected immediately;
5. the production module imports successfully in the current Python process;
6. the imported class resolves to:

```text
hydrolab3d.core.simulator.Simulator
```

7. no tests or result artifacts are created yet.

Production parity and fresh-process validation occur only after this promotion step passes.

In [12]:
from pathlib import Path
import importlib
import textwrap


# ---------------------------------------------------------------------
# Resolve production paths
# ---------------------------------------------------------------------

core_dir = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "core"
)

core_init_path = (
    core_dir
    / "__init__.py"
)

simulator_path = (
    core_dir
    / "simulator.py"
)

clock_path = (
    core_dir
    / "clock.py"
)

state_path = (
    core_dir
    / "state.py"
)

entity_path = (
    core_dir
    / "entity.py"
)


# ---------------------------------------------------------------------
# Confirm expected scaffold files exist
# ---------------------------------------------------------------------

for path in [
    core_init_path,
    simulator_path,
    clock_path,
    state_path,
    entity_path,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Expected core scaffold file does not exist: {path}"
        )


# ---------------------------------------------------------------------
# Preserve files that must remain untouched
# ---------------------------------------------------------------------

clock_before = clock_path.read_text(
    encoding="utf-8"
)

state_before = state_path.read_text(
    encoding="utf-8"
)

entity_before = entity_path.read_text(
    encoding="utf-8"
)


# ---------------------------------------------------------------------
# Production simulator source
# ---------------------------------------------------------------------

simulator_source = textwrap.dedent(
    '''
    """Core HydroLab-3D simulator kernel."""

    from __future__ import annotations

    import copy

    import numpy as np

    from hydrolab3d.sensors import (
        accelerometer_measurement,
        depth_measurement,
        dvl_measurement,
        gyroscope_measurement,
        heading_measurement,
    )
    from hydrolab3d.vehicles.uuv import (
        integrate_six_dof_euler,
    )


    def _copy_configuration(value):
        """Recursively copy mutable simulator configuration."""
        if isinstance(value, np.ndarray):
            return value.copy()

        if isinstance(value, dict):
            return {
                key: _copy_configuration(item)
                for key, item in value.items()
            }

        if isinstance(value, list):
            return [
                _copy_configuration(item)
                for item in value
            ]

        if isinstance(value, tuple):
            return tuple(
                _copy_configuration(item)
                for item in value
            )

        return copy.deepcopy(value)


    class Simulator:
        """
        Lightweight HydroLab-3D simulator kernel.

        The kernel owns simulator truth, deterministic model time,
        vehicle/environment configuration, and a reproducible
        sensor-observation layer.

        Vehicle propagation is delegated entirely to the production
        six-degree-of-freedom integrator.
        """

        def __init__(
            self,
            *,
            initial_pose,
            initial_body_velocity,
            dt,
            dynamics_kwargs,
            sensor_config=None,
            sensor_seed=None,
        ):
            initial_pose = np.asarray(
                initial_pose,
                dtype=float,
            )

            initial_body_velocity = np.asarray(
                initial_body_velocity,
                dtype=float,
            )

            dt = float(dt)

            if initial_pose.shape != (6,):
                raise ValueError(
                    "initial_pose must have shape (6,)"
                )

            if initial_body_velocity.shape != (6,):
                raise ValueError(
                    "initial_body_velocity must have shape (6,)"
                )

            if not np.all(
                np.isfinite(initial_pose)
            ):
                raise ValueError(
                    "initial_pose must contain only finite values"
                )

            if not np.all(
                np.isfinite(initial_body_velocity)
            ):
                raise ValueError(
                    "initial_body_velocity must contain only finite values"
                )

            if not np.isfinite(dt) or dt <= 0.0:
                raise ValueError(
                    "dt must be finite and strictly positive"
                )

            if not isinstance(
                dynamics_kwargs,
                dict,
            ):
                raise TypeError(
                    "dynamics_kwargs must be a dictionary"
                )

            if sensor_config is None:
                sensor_config = {}

            if not isinstance(
                sensor_config,
                dict,
            ):
                raise TypeError(
                    "sensor_config must be a dictionary"
                )

            self._initial_pose = (
                initial_pose.copy()
            )

            self._initial_body_velocity = (
                initial_body_velocity.copy()
            )

            self._dt = dt

            self._dynamics_kwargs = (
                _copy_configuration(
                    dynamics_kwargs
                )
            )

            self._sensor_config = (
                _copy_configuration(
                    sensor_config
                )
            )

            self._sensor_seed = copy.deepcopy(
                sensor_seed
            )

            self._pose = None
            self._body_velocity = None
            self._time = None
            self._diagnostics = None
            self._rng = None

            self.reset()


        @property
        def dt(self):
            """Return the fixed simulator timestep."""
            return self._dt


        @property
        def time(self):
            """Return current deterministic simulation time."""
            return self._time


        def state(self):
            """Return an independent ground-truth state snapshot."""
            return {
                "time": float(self._time),
                "pose": self._pose.copy(),
                "body_velocity": (
                    self._body_velocity.copy()
                ),
            }


        def reset(self):
            """
            Restore initial physical truth and sensor RNG state.
            """
            self._pose = (
                self._initial_pose.copy()
            )

            self._body_velocity = (
                self._initial_body_velocity.copy()
            )

            self._time = 0.0
            self._diagnostics = None

            self._rng = np.random.default_rng(
                self._sensor_seed
            )

            return self.state()


        def _sensor_settings(
            self,
            sensor_name,
        ):
            settings = (
                self._sensor_config.get(
                    sensor_name,
                    {},
                )
            )

            if not isinstance(
                settings,
                dict,
            ):
                raise TypeError(
                    f"sensor_config['{sensor_name}'] "
                    "must be a dictionary"
                )

            return _copy_configuration(
                settings
            )


        def _generate_observation(
            self,
            body_acceleration,
        ):
            """
            Generate post-step sensor measurements from truth.
            """
            body_acceleration = np.asarray(
                body_acceleration,
                dtype=float,
            )

            if body_acceleration.shape != (6,):
                raise ValueError(
                    "body_acceleration must have shape (6,)"
                )

            if not np.all(
                np.isfinite(body_acceleration)
            ):
                raise ValueError(
                    "body_acceleration must contain only finite values"
                )

            current_velocity_world = np.asarray(
                self._dynamics_kwargs[
                    "current_velocity_world"
                ],
                dtype=float,
            )

            depth_settings = (
                self._sensor_settings(
                    "depth"
                )
            )

            heading_settings = (
                self._sensor_settings(
                    "heading"
                )
            )

            dvl_settings = (
                self._sensor_settings(
                    "dvl"
                )
            )

            gyro_settings = (
                self._sensor_settings(
                    "gyroscope"
                )
            )

            accel_settings = (
                self._sensor_settings(
                    "accelerometer"
                )
            )

            return {
                "depth": float(
                    depth_measurement(
                        self._pose,
                        rng=self._rng,
                        **depth_settings,
                    )
                ),
                "heading": float(
                    heading_measurement(
                        self._pose,
                        rng=self._rng,
                        **heading_settings,
                    )
                ),
                "dvl": np.asarray(
                    dvl_measurement(
                        self._pose,
                        self._body_velocity,
                        current_velocity_world,
                        rng=self._rng,
                        **dvl_settings,
                    ),
                    dtype=float,
                ).copy(),
                "gyroscope": np.asarray(
                    gyroscope_measurement(
                        self._body_velocity,
                        rng=self._rng,
                        **gyro_settings,
                    ),
                    dtype=float,
                ).copy(),
                "accelerometer": np.asarray(
                    accelerometer_measurement(
                        self._pose,
                        self._body_velocity,
                        body_acceleration,
                        rng=self._rng,
                        **accel_settings,
                    ),
                    dtype=float,
                ).copy(),
            }


        def step(self, action):
            """
            Advance exactly one physics timestep.

            Parameters
            ----------
            action : array-like, shape (6,)
                Generalized body-frame actuator wrench
                [Fx, Fy, Fz, Mx, My, Mz].

            Returns
            -------
            dict
                Dictionary containing copied ground truth and post-step
                sensor observations.
            """
            action = np.asarray(
                action,
                dtype=float,
            )

            if action.shape != (6,):
                raise ValueError(
                    "action must have shape (6,)"
                )

            if not np.all(
                np.isfinite(action)
            ):
                raise ValueError(
                    "action must contain only finite values"
                )

            step_kwargs = (
                _copy_configuration(
                    self._dynamics_kwargs
                )
            )

            step_kwargs[
                "actuator_wrench"
            ] = action.copy()

            (
                next_pose,
                next_body_velocity,
                diagnostics,
            ) = integrate_six_dof_euler(
                self._pose,
                self._body_velocity,
                self._dt,
                **step_kwargs,
            )

            self._pose = np.asarray(
                next_pose,
                dtype=float,
            ).copy()

            self._body_velocity = np.asarray(
                next_body_velocity,
                dtype=float,
            ).copy()

            self._time += self._dt

            self._diagnostics = (
                _copy_configuration(
                    diagnostics
                )
            )

            observation = (
                self._generate_observation(
                    self._diagnostics[
                        "nu_dot"
                    ]
                )
            )

            return {
                "truth": self.state(),
                "observation": (
                    _copy_configuration(
                        observation
                    )
                ),
            }
    '''
).lstrip()


# ---------------------------------------------------------------------
# Production core package export
# ---------------------------------------------------------------------

core_init_source = textwrap.dedent(
    '''
    """Core simulator interfaces."""

    from .simulator import Simulator

    __all__ = [
        "Simulator",
    ]
    '''
).lstrip()


# ---------------------------------------------------------------------
# Write validated production implementation
# ---------------------------------------------------------------------

simulator_path.write_text(
    simulator_source,
    encoding="utf-8",
)

core_init_path.write_text(
    core_init_source,
    encoding="utf-8",
)


# ---------------------------------------------------------------------
# Verify files that should remain untouched
# ---------------------------------------------------------------------

clock_after = clock_path.read_text(
    encoding="utf-8"
)

state_after = state_path.read_text(
    encoding="utf-8"
)

entity_after = entity_path.read_text(
    encoding="utf-8"
)

assert clock_after == clock_before
assert state_after == state_before
assert entity_after == entity_before


# ---------------------------------------------------------------------
# Inspect promoted production files
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 9 PRODUCTION PROMOTION")
print("=" * 80)


print("\nWritten production files")
print("-" * 80)

print(
    simulator_path.relative_to(
        repo_root
    )
)

print(
    core_init_path.relative_to(
        repo_root
    )
)


print("\nIntentionally untouched scaffold files")
print("-" * 80)

print(
    f"{clock_path.relative_to(repo_root)} "
    f"unchanged={clock_after == clock_before}"
)

print(
    f"{state_path.relative_to(repo_root)} "
    f"unchanged={state_after == state_before}"
)

print(
    f"{entity_path.relative_to(repo_root)} "
    f"unchanged={entity_after == entity_before}"
)


print("\n" + "-" * 80)
print("PROMOTED simulator.py")
print("-" * 80)

print(
    simulator_path.read_text(
        encoding="utf-8"
    )
)


print("\n" + "-" * 80)
print("PROMOTED core/__init__.py")
print("-" * 80)

print(
    core_init_path.read_text(
        encoding="utf-8"
    )
)


# ---------------------------------------------------------------------
# Fresh import inside current notebook process
# ---------------------------------------------------------------------

importlib.invalidate_caches()

production_simulator_module = importlib.import_module(
    "hydrolab3d.core.simulator"
)

production_simulator_module = importlib.reload(
    production_simulator_module
)

production_core_module = importlib.import_module(
    "hydrolab3d.core"
)

production_core_module = importlib.reload(
    production_core_module
)

ProductionSimulator = (
    production_core_module.Simulator
)


# ---------------------------------------------------------------------
# Import validation
# ---------------------------------------------------------------------

assert (
    ProductionSimulator.__name__
    == "Simulator"
)

assert (
    ProductionSimulator.__module__
    == "hydrolab3d.core.simulator"
)

assert (
    production_core_module.__all__
    == ["Simulator"]
)


print("\nImport validation")
print("-" * 80)

print(
    "class name   :",
    ProductionSimulator.__name__,
)

print(
    "class module :",
    ProductionSimulator.__module__,
)

print(
    "core __all__ :",
    production_core_module.__all__,
)


print("\n" + "=" * 80)
print("STEP 9 PRODUCTION PROMOTION PASSED")
print("=" * 80)

print(
    "The validated notebook-local simulator kernel was promoted into "
    "src/hydrolab3d/core/simulator.py."
)

print(
    "src/hydrolab3d/core/__init__.py now exposes Simulator as the "
    "validated public core API."
)

print(
    "clock.py, state.py, and entity.py remained untouched empty "
    "scaffolds because Notebook 10 has not justified those abstractions."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 9 PRODUCTION PROMOTION

Written production files
--------------------------------------------------------------------------------
src/hydrolab3d/core/simulator.py
src/hydrolab3d/core/__init__.py

Intentionally untouched scaffold files
--------------------------------------------------------------------------------
src/hydrolab3d/core/clock.py unchanged=True
src/hydrolab3d/core/state.py unchanged=True
src/hydrolab3d/core/entity.py unchanged=True

--------------------------------------------------------------------------------
PROMOTED simulator.py
--------------------------------------------------------------------------------
"""Core HydroLab-3D simulator kernel."""

from __future__ import annotations

import copy

import numpy as np

from hydrolab3d.sensors import (
    accelerometer_measurement,
    depth_measurement,
    dvl_measurement,
    gyroscope_measurement,
    heading_measurement,
)
from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_eu

## Step 10 - Validate the Promoted Production `Simulator`

Step 9 promoted the validated notebook-local architecture into:

```text
src/hydrolab3d/core/simulator.py
```

and exposed:

```python
from hydrolab3d.core import Simulator
```

The next step must therefore stop relying on notebook-local simulator classes.

From this point onward, validation should exercise the actual production implementation.

The purpose of this step is to verify that promotion introduced no difference between:

```text
validated notebook behavior
```

and:

```text
production package behavior
```

---

### Production Validation Targets

The promoted `Simulator` must preserve the contracts already established in Steps 3 through 8.

These include:

```text
exact one-step physics parity
deterministic simulation time
exact reset
exact multi-step replay
ground-truth / observation separation
seeded sensor reproducibility
configuration ownership
state-snapshot ownership
```

This step will exercise all of those properties directly through:

```python
from hydrolab3d.core import Simulator
```

---

### Direct Physics Parity

The strongest physics check remains:

```text
production Simulator.step(action)
```

versus:

```text
integrate_six_dof_euler(...)
```

under identical:

```text
initial pose
initial body velocity
dt
vehicle parameters
current
action
```

The returned production truth should exactly match the direct integrator.

---

### Reset and Replay

The production simulator will execute a multi-step action sequence, reset, and replay it.

For a fixed configuration and sensor seed:

$$
\eta_k^{(A)}
=
\eta_k^{(B)}
$$

and:

$$
\nu_k^{(A)}
=
\nu_k^{(B)}
$$

while noisy observations should also reproduce exactly.

---

### Truth / Observation Separation

A second production simulator will use a different sensor seed but identical physical configuration and actions.

The expected result is:

```text
physical truth identical
sensor observations different
```

Therefore:

$$
\boxed{
\text{sensor randomness does not enter physical propagation}
}
$$

---

### Configuration Ownership

The production constructor receives caller-owned NumPy arrays inside its configuration dictionaries.

After construction, those external arrays will be deliberately mutated.

The resulting simulator trajectory must remain identical to an independent reference simulator created from the original configuration.

This verifies that the ownership guarantees proven notebook-locally survived promotion.

---

### Validation Criteria

This step passes if:

1. `Simulator` imports from `hydrolab3d.core`;
2. one production `step()` exactly matches `integrate_six_dof_euler()`;
3. production simulation time advances correctly;
4. reset restores the exact initial state;
5. same-seed replay reproduces truth exactly;
6. same-seed replay reproduces noisy observations exactly;
7. a different sensor seed leaves physical truth exactly unchanged;
8. a different sensor seed changes noisy observations;
9. caller-owned configuration mutation does not alter simulator behavior;
10. returned state arrays remain independent copies;
11. all validated production states and observations remain finite;
12. no new production, test, or result file is written in this step.

Passing this step establishes production parity inside the current Python process.

A separate fresh-process validation will follow so that stale notebook imports cannot hide packaging problems.

In [13]:
import numpy as np

from hydrolab3d.core import Simulator
from hydrolab3d.vehicles.uuv import (
    integrate_six_dof_euler,
)


# ---------------------------------------------------------------------
# Shared production-validation configuration
# ---------------------------------------------------------------------

production_initial_pose = np.array(
    [
        1.0,
        -0.5,
        -6.5,
        np.deg2rad(4.0),
        np.deg2rad(-2.0),
        np.deg2rad(18.0),
    ],
    dtype=float,
)

production_initial_velocity = np.array(
    [
        0.18,
        -0.03,
        0.015,
        0.008,
        -0.006,
        0.012,
    ],
    dtype=float,
)

production_dt = kernel_dt

production_current = np.array(
    [
        0.22,
        -0.07,
        0.0,
    ],
    dtype=float,
)

production_inertia = (
    probe_inertia.copy()
)

production_linear_damping = (
    probe_linear_damping.copy()
)

production_dvl_bias = np.array(
    [
        0.01,
        -0.01,
        0.005,
    ],
    dtype=float,
)


production_dynamics = {
    "mass": probe_mass,
    "inertia": production_inertia,
    "added_inertia": (
        probe_added_inertia.copy()
    ),
    "linear_damping": (
        production_linear_damping
    ),
    "quadratic_damping": (
        probe_quadratic_damping.copy()
    ),
    "fluid_density": (
        probe_fluid_density
    ),
    "displaced_volume": (
        probe_displaced_volume
    ),
    "center_of_gravity": (
        probe_center_of_gravity.copy()
    ),
    "center_of_buoyancy": (
        probe_center_of_buoyancy.copy()
    ),
    "current_velocity_world": (
        production_current
    ),
}


production_sensor_config = {
    "depth": {
        "bias": 0.03,
        "noise_std": 0.04,
    },
    "heading": {
        "bias": np.deg2rad(0.75),
        "noise_std": np.deg2rad(1.0),
    },
    "dvl": {
        "bias": production_dvl_bias,
        "noise_std": 0.015,
    },
    "gyroscope": {
        "bias": np.array(
            [0.001, -0.001, 0.0005],
            dtype=float,
        ),
        "noise_std": 0.002,
    },
    "accelerometer": {
        "bias": np.array(
            [0.02, -0.01, 0.03],
            dtype=float,
        ),
        "noise_std": 0.025,
    },
}


# ---------------------------------------------------------------------
# 1. Direct one-step physics parity
# ---------------------------------------------------------------------

production_parity_action = np.array(
    [
        14.0,
        -2.5,
        3.0,
        0.15,
        -0.10,
        0.25,
    ],
    dtype=float,
)

production_parity_simulator = Simulator(
    initial_pose=production_initial_pose,
    initial_body_velocity=(
        production_initial_velocity
    ),
    dt=production_dt,
    dynamics_kwargs=production_dynamics,
    sensor_config={},
    sensor_seed=123,
)

production_parity_result = (
    production_parity_simulator.step(
        production_parity_action
    )
)

production_parity_truth = (
    production_parity_result["truth"]
)

(
    production_direct_pose,
    production_direct_velocity,
    production_direct_diagnostics,
) = integrate_six_dof_euler(
    production_initial_pose,
    production_initial_velocity,
    production_dt,
    **production_dynamics,
    actuator_wrench=(
        production_parity_action
    ),
)


production_pose_exact = (
    np.array_equal(
        production_parity_truth["pose"],
        production_direct_pose,
    )
)

production_velocity_exact = (
    np.array_equal(
        production_parity_truth[
            "body_velocity"
        ],
        production_direct_velocity,
    )
)

assert production_pose_exact
assert production_velocity_exact

assert np.isclose(
    production_parity_truth["time"],
    production_dt,
    rtol=0.0,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# Shared rollout sequence
# ---------------------------------------------------------------------

production_actions = np.array(
    [
        [10.0,  0.0,  0.0,  0.00,  0.00,  0.00],
        [12.0, -1.0,  0.0,  0.02,  0.00,  0.06],
        [ 9.0,  0.0, -1.5,  0.00, -0.03,  0.00],
        [ 6.0,  2.0,  0.5, -0.02,  0.02, -0.05],
        [ 0.0,  0.0,  0.0,  0.00,  0.00,  0.00],
        [ 4.0, -1.0,  1.0,  0.01, -0.01,  0.03],
        [ 3.0,  0.0,  0.0,  0.00,  0.00, -0.02],
        [ 0.0,  0.0,  0.0,  0.00,  0.00,  0.00],
    ],
    dtype=float,
)


# ---------------------------------------------------------------------
# Rollout helper for production Simulator
# ---------------------------------------------------------------------

def run_production_rollout(
    simulator,
    actions,
):
    times = []
    poses = []
    velocities = []

    depth = []
    heading = []
    dvl = []
    gyroscope = []
    accelerometer = []

    for action in actions:
        result = simulator.step(
            action
        )

        truth = result["truth"]
        observation = (
            result["observation"]
        )

        assert truth["pose"].shape == (6,)
        assert truth["body_velocity"].shape == (6,)

        assert np.all(
            np.isfinite(
                truth["pose"]
            )
        )

        assert np.all(
            np.isfinite(
                truth["body_velocity"]
            )
        )

        assert np.isfinite(
            observation["depth"]
        )

        assert np.isfinite(
            observation["heading"]
        )

        assert observation["dvl"].shape == (3,)
        assert observation["gyroscope"].shape == (3,)
        assert observation["accelerometer"].shape == (3,)

        assert np.all(
            np.isfinite(
                observation["dvl"]
            )
        )

        assert np.all(
            np.isfinite(
                observation["gyroscope"]
            )
        )

        assert np.all(
            np.isfinite(
                observation["accelerometer"]
            )
        )

        times.append(
            truth["time"]
        )

        poses.append(
            truth["pose"]
        )

        velocities.append(
            truth["body_velocity"]
        )

        depth.append(
            observation["depth"]
        )

        heading.append(
            observation["heading"]
        )

        dvl.append(
            observation["dvl"]
        )

        gyroscope.append(
            observation["gyroscope"]
        )

        accelerometer.append(
            observation[
                "accelerometer"
            ]
        )

    return {
        "times": np.asarray(
            times,
            dtype=float,
        ),
        "poses": np.asarray(
            poses,
            dtype=float,
        ),
        "velocities": np.asarray(
            velocities,
            dtype=float,
        ),
        "depth": np.asarray(
            depth,
            dtype=float,
        ),
        "heading": np.asarray(
            heading,
            dtype=float,
        ),
        "dvl": np.asarray(
            dvl,
            dtype=float,
        ),
        "gyroscope": np.asarray(
            gyroscope,
            dtype=float,
        ),
        "accelerometer": np.asarray(
            accelerometer,
            dtype=float,
        ),
    }


# ---------------------------------------------------------------------
# 2. Same-seed production reset / replay
# ---------------------------------------------------------------------

production_seed_a = 10101

production_simulator_a = Simulator(
    initial_pose=production_initial_pose,
    initial_body_velocity=(
        production_initial_velocity
    ),
    dt=production_dt,
    dynamics_kwargs=production_dynamics,
    sensor_config=production_sensor_config,
    sensor_seed=production_seed_a,
)

production_run_a = (
    run_production_rollout(
        production_simulator_a,
        production_actions,
    )
)

production_reset_state = (
    production_simulator_a.reset()
)

assert (
    production_reset_state["time"]
    == 0.0
)

assert np.array_equal(
    production_reset_state["pose"],
    production_initial_pose,
)

assert np.array_equal(
    production_reset_state[
        "body_velocity"
    ],
    production_initial_velocity,
)

production_run_a_replay = (
    run_production_rollout(
        production_simulator_a,
        production_actions,
    )
)


production_same_seed_truth_exact = (
    np.array_equal(
        production_run_a["times"],
        production_run_a_replay[
            "times"
        ],
    )
    and np.array_equal(
        production_run_a["poses"],
        production_run_a_replay[
            "poses"
        ],
    )
    and np.array_equal(
        production_run_a["velocities"],
        production_run_a_replay[
            "velocities"
        ],
    )
)

production_same_seed_obs_exact = (
    np.array_equal(
        production_run_a["depth"],
        production_run_a_replay[
            "depth"
        ],
    )
    and np.array_equal(
        production_run_a["heading"],
        production_run_a_replay[
            "heading"
        ],
    )
    and np.array_equal(
        production_run_a["dvl"],
        production_run_a_replay[
            "dvl"
        ],
    )
    and np.array_equal(
        production_run_a["gyroscope"],
        production_run_a_replay[
            "gyroscope"
        ],
    )
    and np.array_equal(
        production_run_a[
            "accelerometer"
        ],
        production_run_a_replay[
            "accelerometer"
        ],
    )
)

assert production_same_seed_truth_exact
assert production_same_seed_obs_exact


# ---------------------------------------------------------------------
# 3. Different seed preserves physics but changes observations
# ---------------------------------------------------------------------

production_seed_b = 10102

production_simulator_b = Simulator(
    initial_pose=production_initial_pose,
    initial_body_velocity=(
        production_initial_velocity
    ),
    dt=production_dt,
    dynamics_kwargs=production_dynamics,
    sensor_config=production_sensor_config,
    sensor_seed=production_seed_b,
)

production_run_b = (
    run_production_rollout(
        production_simulator_b,
        production_actions,
    )
)

production_different_seed_truth_exact = (
    np.array_equal(
        production_run_a["times"],
        production_run_b["times"],
    )
    and np.array_equal(
        production_run_a["poses"],
        production_run_b["poses"],
    )
    and np.array_equal(
        production_run_a["velocities"],
        production_run_b["velocities"],
    )
)

assert production_different_seed_truth_exact


production_seed_observation_difference = max(
    float(
        np.max(
            np.abs(
                production_run_a["depth"]
                - production_run_b["depth"]
            )
        )
    ),
    float(
        np.max(
            np.abs(
                production_run_a["heading"]
                - production_run_b["heading"]
            )
        )
    ),
    float(
        np.max(
            np.abs(
                production_run_a["dvl"]
                - production_run_b["dvl"]
            )
        )
    ),
    float(
        np.max(
            np.abs(
                production_run_a["gyroscope"]
                - production_run_b["gyroscope"]
            )
        )
    ),
    float(
        np.max(
            np.abs(
                production_run_a["accelerometer"]
                - production_run_b["accelerometer"]
            )
        )
    ),
)

assert (
    production_seed_observation_difference
    > 0.0
)


# ---------------------------------------------------------------------
# 4. Configuration ownership in production
# ---------------------------------------------------------------------

ownership_external_current = np.array(
    [0.22, -0.07, 0.0],
    dtype=float,
)

ownership_external_inertia = (
    probe_inertia.copy()
)

ownership_external_damping = (
    probe_linear_damping.copy()
)

ownership_external_dvl_bias = np.array(
    [0.01, -0.01, 0.005],
    dtype=float,
)


ownership_prod_dynamics = {
    "mass": probe_mass,
    "inertia": ownership_external_inertia,
    "added_inertia": (
        probe_added_inertia.copy()
    ),
    "linear_damping": (
        ownership_external_damping
    ),
    "quadratic_damping": (
        probe_quadratic_damping.copy()
    ),
    "fluid_density": (
        probe_fluid_density
    ),
    "displaced_volume": (
        probe_displaced_volume
    ),
    "center_of_gravity": (
        probe_center_of_gravity.copy()
    ),
    "center_of_buoyancy": (
        probe_center_of_buoyancy.copy()
    ),
    "current_velocity_world": (
        ownership_external_current
    ),
}


ownership_prod_sensors = {
    "depth": {
        "bias": 0.03,
        "noise_std": 0.04,
    },
    "heading": {
        "bias": np.deg2rad(0.75),
        "noise_std": np.deg2rad(1.0),
    },
    "dvl": {
        "bias": ownership_external_dvl_bias,
        "noise_std": 0.015,
    },
    "gyroscope": {
        "bias": np.array(
            [0.001, -0.001, 0.0005],
            dtype=float,
        ),
        "noise_std": 0.002,
    },
    "accelerometer": {
        "bias": np.array(
            [0.02, -0.01, 0.03],
            dtype=float,
        ),
        "noise_std": 0.025,
    },
}


ownership_reference_dynamics = (
    copy_configuration(
        ownership_prod_dynamics
    )
)

ownership_reference_sensors = (
    copy_configuration(
        ownership_prod_sensors
    )
)


ownership_prod_simulator = Simulator(
    initial_pose=production_initial_pose,
    initial_body_velocity=(
        production_initial_velocity
    ),
    dt=production_dt,
    dynamics_kwargs=ownership_prod_dynamics,
    sensor_config=ownership_prod_sensors,
    sensor_seed=555,
)

ownership_prod_reference = Simulator(
    initial_pose=production_initial_pose,
    initial_body_velocity=(
        production_initial_velocity
    ),
    dt=production_dt,
    dynamics_kwargs=(
        ownership_reference_dynamics
    ),
    sensor_config=(
        ownership_reference_sensors
    ),
    sensor_seed=555,
)


# Mutate all caller-owned objects after construction.
ownership_external_current[:] = 999.0
ownership_external_inertia[:] = 999.0
ownership_external_damping[:] = 999.0
ownership_external_dvl_bias[:] = 999.0

ownership_prod_sensors[
    "depth"
]["bias"] = 999.0

ownership_prod_sensors[
    "depth"
]["noise_std"] = 999.0


ownership_prod_run = (
    run_production_rollout(
        ownership_prod_simulator,
        production_actions,
    )
)

ownership_reference_run = (
    run_production_rollout(
        ownership_prod_reference,
        production_actions,
    )
)


production_configuration_owned = (
    np.array_equal(
        ownership_prod_run["times"],
        ownership_reference_run[
            "times"
        ],
    )
    and np.array_equal(
        ownership_prod_run["poses"],
        ownership_reference_run[
            "poses"
        ],
    )
    and np.array_equal(
        ownership_prod_run["velocities"],
        ownership_reference_run[
            "velocities"
        ],
    )
    and np.array_equal(
        ownership_prod_run["depth"],
        ownership_reference_run[
            "depth"
        ],
    )
    and np.array_equal(
        ownership_prod_run["heading"],
        ownership_reference_run[
            "heading"
        ],
    )
    and np.array_equal(
        ownership_prod_run["dvl"],
        ownership_reference_run[
            "dvl"
        ],
    )
    and np.array_equal(
        ownership_prod_run["gyroscope"],
        ownership_reference_run[
            "gyroscope"
        ],
    )
    and np.array_equal(
        ownership_prod_run[
            "accelerometer"
        ],
        ownership_reference_run[
            "accelerometer"
        ],
    )
)

assert production_configuration_owned


# ---------------------------------------------------------------------
# 5. Returned state snapshot ownership
# ---------------------------------------------------------------------

production_snapshot = (
    production_simulator_a.state()
)

production_snapshot["pose"][0] = 123456.0

production_snapshot[
    "body_velocity"
][0] = 123456.0

production_snapshot_check = (
    production_simulator_a.state()
)

production_snapshot_independent = (
    production_snapshot_check[
        "pose"
    ][0]
    != 123456.0
    and production_snapshot_check[
        "body_velocity"
    ][0]
    != 123456.0
)

assert production_snapshot_independent


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 10 PRODUCTION VALIDATION")
print("=" * 80)


print("\nProduction import")
print("-" * 80)

print(
    "class  :",
    Simulator,
)

print(
    "module :",
    Simulator.__module__,
)


print("\nDirect physics parity")
print("-" * 80)

print(
    "pose exact match     :",
    production_pose_exact,
)

print(
    "velocity exact match :",
    production_velocity_exact,
)

print(
    "maximum pose error   :",
    np.max(
        np.abs(
            production_parity_truth[
                "pose"
            ]
            - production_direct_pose
        )
    ),
)

print(
    "maximum velocity error:",
    np.max(
        np.abs(
            production_parity_truth[
                "body_velocity"
            ]
            - production_direct_velocity
        )
    ),
)


print("\nReset / replay")
print("-" * 80)

print(
    "same-seed truth replay exact       :",
    production_same_seed_truth_exact,
)

print(
    "same-seed observation replay exact :",
    production_same_seed_obs_exact,
)


print("\nTruth / observation separation")
print("-" * 80)

print(
    "different sensor seed preserves "
    "truth exactly :",
    production_different_seed_truth_exact,
)

print(
    f"largest noisy observation difference: "
    f"{production_seed_observation_difference:.12e}"
)


print("\nOwnership")
print("-" * 80)

print(
    "configuration independently owned :",
    production_configuration_owned,
)

print(
    "state snapshot independent        :",
    production_snapshot_independent,
)


print("\nFinal production state")
print("-" * 80)

print(
    "pose          :",
    production_run_a["poses"][-1],
)

print(
    "body velocity :",
    production_run_a[
        "velocities"
    ][-1],
)

print(
    f"time          : "
    f"{production_run_a['times'][-1]:.12f} s"
)


print("\n" + "=" * 80)
print("STEP 10 PRODUCTION VALIDATION PASSED")
print("=" * 80)

print(
    "The promoted hydrolab3d.core.Simulator reproduces direct "
    "production physics exactly."
)

print(
    "Reset, seeded sensing, truth separation, configuration ownership, "
    "and state-snapshot ownership all match the validated notebook-local "
    "architecture."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 10 PRODUCTION VALIDATION

Production import
--------------------------------------------------------------------------------
class  : <class 'hydrolab3d.core.simulator.Simulator'>
module : hydrolab3d.core.simulator

Direct physics parity
--------------------------------------------------------------------------------
pose exact match     : True
velocity exact match : True
maximum pose error   : 0.0
maximum velocity error: 0.0

Reset / replay
--------------------------------------------------------------------------------
same-seed truth replay exact       : True
same-seed observation replay exact : True

Truth / observation separation
--------------------------------------------------------------------------------
different sensor seed preserves truth exactly : True
largest noisy observation difference: 1.154431587159e-01

Ownership
--------------------------------------------------------------------------------
configuration independently owned : True
st

## Step 11 - Validate the Production Kernel in a Fresh Python Process

Step 10 validated the promoted production `Simulator` inside the current notebook process.

That is necessary, but a Jupyter kernel can hide packaging problems because it may retain:

```text
previous imports
notebook-local definitions
modified sys.path state
reloaded modules
temporary variables
```

A reusable simulator must also work from a completely fresh Python interpreter.

This step therefore launches a separate Python process and imports:

```python
from hydrolab3d.core import Simulator
```

directly from the repository's `src/` package.

No notebook-local simulator class will exist in that process.

---

### Fresh-Process Validation Goals

The external interpreter will verify:

```text
HydroLab-3D imports successfully
Simulator resolves from hydrolab3d.core
Simulator can be constructed
reset() returns valid ground truth
step(action) executes successfully
time advances exactly one timestep
pose and velocity remain finite
sensor observations are generated
observation shapes are valid
```

It will also report:

```text
Python executable
production source path
Simulator module
returned truth
returned observation
```

so that the notebook can confirm that the process actually used:

```text
src/hydrolab3d/core/simulator.py
```

rather than any notebook-local implementation.

---

### Why This Matters

The intended dependency chain is:

```text
fresh Python process
        ↓
repository src/
        ↓
hydrolab3d.core
        ↓
Simulator
        ↓
vehicles.uuv
        ↓
production dynamics
        ↓
production sensors
```

If this works from a clean interpreter, Notebook 10 has demonstrated that the simulator kernel is a real package capability rather than an artifact of notebook execution order.

---

### Validation Criteria

This step passes if:

1. a fresh Python process exits with return code `0`;
2. `hydrolab3d.core.Simulator` imports successfully;
3. the imported class resolves to:

```text
hydrolab3d.core.simulator.Simulator
```

4. the production source path points inside the current repository;
5. simulator construction succeeds;
6. `reset()` returns finite ground truth;
7. one `step()` succeeds;
8. time advances exactly by the configured timestep;
9. returned pose and body velocity have shape `(6,)`;
10. depth and heading are finite;
11. DVL, gyroscope, and accelerometer each have shape `(3,)`;
12. no project source, test, or result file is modified.

Passing this step establishes that the promoted simulator kernel works independently of notebook-local state.

In [14]:
import json
import os
import subprocess
import sys
import textwrap
from pathlib import Path


# ---------------------------------------------------------------------
# Build fresh-process validation script
# ---------------------------------------------------------------------

fresh_process_script = textwrap.dedent(
    f'''
    import json
    import sys
    from pathlib import Path

    import numpy as np


    repo_root = Path(
        {str(repo_root)!r}
    ).resolve()

    src_path = (
        repo_root
        / "src"
    )

    sys.path.insert(
        0,
        str(src_path),
    )


    from hydrolab3d.core import Simulator
    import hydrolab3d.core.simulator as simulator_module


    initial_pose = np.array(
        [
            0.0,
            0.0,
            -5.0,
            0.0,
            0.0,
            0.0,
        ],
        dtype=float,
    )

    initial_body_velocity = np.zeros(
        6,
        dtype=float,
    )

    dt = 0.02

    mass = 50.0

    inertia = np.array(
        [
            2.0,
            3.0,
            4.0,
        ],
        dtype=float,
    )

    added_inertia = np.array(
        [
            5.0,
            8.0,
            10.0,
            0.5,
            0.7,
            0.9,
        ],
        dtype=float,
    )

    linear_damping = np.array(
        [
            8.0,
            12.0,
            15.0,
            1.5,
            2.0,
            2.5,
        ],
        dtype=float,
    )

    quadratic_damping = np.array(
        [
            12.0,
            18.0,
            22.0,
            2.0,
            2.5,
            3.0,
        ],
        dtype=float,
    )

    fluid_density = 1025.0

    displaced_volume = (
        mass
        / fluid_density
    )

    dynamics_kwargs = {{
        "mass": mass,
        "inertia": inertia,
        "added_inertia": added_inertia,
        "linear_damping": linear_damping,
        "quadratic_damping": quadratic_damping,
        "fluid_density": fluid_density,
        "displaced_volume": displaced_volume,
        "center_of_gravity": np.array(
            [0.0, 0.0, 0.0],
            dtype=float,
        ),
        "center_of_buoyancy": np.array(
            [0.0, 0.0, 0.08],
            dtype=float,
        ),
        "current_velocity_world": np.array(
            [0.15, -0.05, 0.0],
            dtype=float,
        ),
    }}

    sensor_config = {{
        "depth": {{
            "bias": 0.03,
            "noise_std": 0.04,
        }},
        "heading": {{
            "bias": np.deg2rad(0.75),
            "noise_std": np.deg2rad(1.0),
        }},
        "dvl": {{
            "bias": np.array(
                [0.01, -0.01, 0.005],
                dtype=float,
            ),
            "noise_std": 0.015,
        }},
        "gyroscope": {{
            "bias": np.array(
                [0.001, -0.001, 0.0005],
                dtype=float,
            ),
            "noise_std": 0.002,
        }},
        "accelerometer": {{
            "bias": np.array(
                [0.02, -0.01, 0.03],
                dtype=float,
            ),
            "noise_std": 0.025,
        }},
    }}


    simulator = Simulator(
        initial_pose=initial_pose,
        initial_body_velocity=(
            initial_body_velocity
        ),
        dt=dt,
        dynamics_kwargs=(
            dynamics_kwargs
        ),
        sensor_config=(
            sensor_config
        ),
        sensor_seed=1111,
    )


    reset_state = simulator.reset()

    action = np.array(
        [
            20.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
        ],
        dtype=float,
    )

    result = simulator.step(
        action
    )

    truth = result["truth"]
    observation = (
        result["observation"]
    )


    assert reset_state["time"] == 0.0

    assert np.array_equal(
        reset_state["pose"],
        initial_pose,
    )

    assert np.array_equal(
        reset_state[
            "body_velocity"
        ],
        initial_body_velocity,
    )

    assert np.isclose(
        truth["time"],
        dt,
        rtol=0.0,
        atol=1e-15,
    )

    assert truth["pose"].shape == (6,)

    assert (
        truth[
            "body_velocity"
        ].shape
        == (6,)
    )

    assert np.all(
        np.isfinite(
            truth["pose"]
        )
    )

    assert np.all(
        np.isfinite(
            truth[
                "body_velocity"
            ]
        )
    )

    assert np.isfinite(
        observation["depth"]
    )

    assert np.isfinite(
        observation["heading"]
    )

    assert (
        observation["dvl"].shape
        == (3,)
    )

    assert (
        observation[
            "gyroscope"
        ].shape
        == (3,)
    )

    assert (
        observation[
            "accelerometer"
        ].shape
        == (3,)
    )

    assert np.all(
        np.isfinite(
            observation["dvl"]
        )
    )

    assert np.all(
        np.isfinite(
            observation[
                "gyroscope"
            ]
        )
    )

    assert np.all(
        np.isfinite(
            observation[
                "accelerometer"
            ]
        )
    )


    report = {{
        "python_executable": (
            sys.executable
        ),
        "simulator_class": (
            f"{{Simulator.__module__}}."
            f"{{Simulator.__name__}}"
        ),
        "simulator_source": str(
            Path(
                simulator_module.__file__
            ).resolve()
        ),
        "reset_time": (
            reset_state["time"]
        ),
        "step_time": (
            truth["time"]
        ),
        "pose": (
            truth["pose"].tolist()
        ),
        "body_velocity": (
            truth[
                "body_velocity"
            ].tolist()
        ),
        "depth": float(
            observation["depth"]
        ),
        "heading": float(
            observation["heading"]
        ),
        "dvl": (
            observation[
                "dvl"
            ].tolist()
        ),
        "gyroscope": (
            observation[
                "gyroscope"
            ].tolist()
        ),
        "accelerometer": (
            observation[
                "accelerometer"
            ].tolist()
        ),
    }}

    print(
        json.dumps(
            report,
            indent=2,
        )
    )
    '''
)


# ---------------------------------------------------------------------
# Launch a completely separate Python interpreter
# ---------------------------------------------------------------------

fresh_process_result = subprocess.run(
    [
        sys.executable,
        "-c",
        fresh_process_script,
    ],
    cwd=repo_root,
    capture_output=True,
    text=True,
    check=False,
)


# ---------------------------------------------------------------------
# Report stdout/stderr before asserting
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 11 FRESH-PROCESS VALIDATION")
print("=" * 80)

print(
    f"\nReturn code: "
    f"{fresh_process_result.returncode}"
)


print("\nFresh-process stdout")
print("-" * 80)

if fresh_process_result.stdout.strip():
    print(
        fresh_process_result.stdout
    )
else:
    print("<NO STDOUT>")


print("\nFresh-process stderr")
print("-" * 80)

if fresh_process_result.stderr.strip():
    print(
        fresh_process_result.stderr
    )
else:
    print("<NO STDERR>")


# ---------------------------------------------------------------------
# Process must succeed
# ---------------------------------------------------------------------

assert (
    fresh_process_result.returncode
    == 0
), (
    "Fresh-process validation failed.\n"
    f"stderr:\n"
    f"{fresh_process_result.stderr}"
)


# ---------------------------------------------------------------------
# Parse structured output
# ---------------------------------------------------------------------

fresh_report = json.loads(
    fresh_process_result.stdout
)


# ---------------------------------------------------------------------
# Validate package identity
# ---------------------------------------------------------------------

expected_simulator_class = (
    "hydrolab3d.core.simulator."
    "Simulator"
)

assert (
    fresh_report[
        "simulator_class"
    ]
    == expected_simulator_class
)


fresh_source_path = Path(
    fresh_report[
        "simulator_source"
    ]
).resolve()

expected_source_path = (
    repo_root
    / "src"
    / "hydrolab3d"
    / "core"
    / "simulator.py"
).resolve()

assert (
    fresh_source_path
    == expected_source_path
)


# ---------------------------------------------------------------------
# Validate reported values again in notebook process
# ---------------------------------------------------------------------

assert fresh_report[
    "reset_time"
] == 0.0

assert np.isclose(
    fresh_report[
        "step_time"
    ],
    0.02,
    rtol=0.0,
    atol=1e-15,
)


fresh_pose = np.asarray(
    fresh_report["pose"],
    dtype=float,
)

fresh_velocity = np.asarray(
    fresh_report[
        "body_velocity"
    ],
    dtype=float,
)

fresh_dvl = np.asarray(
    fresh_report["dvl"],
    dtype=float,
)

fresh_gyro = np.asarray(
    fresh_report["gyroscope"],
    dtype=float,
)

fresh_accel = np.asarray(
    fresh_report["accelerometer"],
    dtype=float,
)


assert fresh_pose.shape == (6,)
assert fresh_velocity.shape == (6,)

assert fresh_dvl.shape == (3,)
assert fresh_gyro.shape == (3,)
assert fresh_accel.shape == (3,)

assert np.all(
    np.isfinite(
        fresh_pose
    )
)

assert np.all(
    np.isfinite(
        fresh_velocity
    )
)

assert np.all(
    np.isfinite(
        fresh_dvl
    )
)

assert np.all(
    np.isfinite(
        fresh_gyro
    )
)

assert np.all(
    np.isfinite(
        fresh_accel
    )
)

assert np.isfinite(
    fresh_report["depth"]
)

assert np.isfinite(
    fresh_report["heading"]
)


# ---------------------------------------------------------------------
# Final validation report
# ---------------------------------------------------------------------

print("\nValidated package identity")
print("-" * 80)

print(
    "Simulator class :",
    fresh_report[
        "simulator_class"
    ],
)

print(
    "Source file     :",
    fresh_source_path,
)

print(
    "Python process  :",
    fresh_report[
        "python_executable"
    ],
)


print("\n" + "=" * 80)
print("STEP 11 FRESH-PROCESS VALIDATION PASSED")
print("=" * 80)

print(
    "A separate Python interpreter imported and executed the promoted "
    "HydroLab-3D Simulator successfully."
)

print(
    "The process used the repository production source directly and "
    "required no notebook-local Simulator definitions."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 11 FRESH-PROCESS VALIDATION

Return code: 0

Fresh-process stdout
--------------------------------------------------------------------------------
{
  "python_executable": "/home/agniacolyte/HydroLab-3D/.venv/bin/python",
  "simulator_class": "hydrolab3d.core.simulator.Simulator",
  "simulator_source": "/home/agniacolyte/HydroLab-3D/src/hydrolab3d/core/simulator.py",
  "reset_time": 0.0,
  "step_time": 0.02,
  "pose": [
    0.0,
    0.0,
    -5.0,
    0.0,
    0.0,
    0.0
  ],
  "body_velocity": [
    0.007807272727272727,
    -0.0002224137931034483,
    0.0,
    0.0,
    0.0,
    9.18367346938775e-05
  ],
  "depth": 5.086359110793953,
  "heading": 0.02128117690462794,
  "dvl": [
    -0.15700781270577852,
    0.04764717603523744,
    0.011777920267548323
  ],
  "gyroscope": [
    0.0016510530994819283,
    0.0009011239904636093,
    0.00212428290977906
  ],
  "accelerometer": [
    0.3982899585571935,
    0.003545916486398463,
    9.837361472510182
  ]
}

## Step 12 - Generate Persistent Simulator-Kernel Validation Artifacts

The production simulator kernel has now passed:

```text
notebook-local lifecycle validation
direct production-physics parity
deterministic reset / replay
ocean-current propagation
longer-rollout finite-state validation
sensor truth separation
seeded sensor reproducibility
configuration ownership
production promotion
production parity
fresh-process execution
```

Notebook 10 should now leave behind a compact persistent record of the validated production behavior.

All new outputs belong exclusively under:

```text
results/notebooks/10_simulator_kernel/
```

No artifact from Notebooks 00 through 09 will be modified.

---

### Artifact Philosophy

Notebook 10 does not need decorative plots.

Its persistent outputs should answer useful engineering questions:

```text
Did the production Simulator execute a real multi-step rollout?

Did simulation time advance correctly?

Did the state remain finite?

Did currents influence motion through the inherited dynamics?

Were truth and noisy observations kept separate?

Was exact direct-physics parity achieved?

Were reset and sensor replay reproducible?
```

The artifact set will therefore remain small.

---

### Production Rollout Artifact

A deterministic production `Simulator` rollout will be generated using:

```text
6-DOF ground truth
uniform ocean current
time-varying six-axis actuator wrench
seeded sensor noise
```

At every step the notebook will record:

```text
time

truth:
x
y
z
roll
pitch
yaw
u
v
w
p
q
r

observation:
depth
heading
DVL x/y/z
gyroscope x/y/z
accelerometer x/y/z
```

This produces a reusable numerical record of the integrated kernel.

---

### Summary Metrics

A compact summary table will preserve the most important Notebook 10 validation results, including:

```text
single-step direct parity error
reset/replay status
sensor replay status
truth separation status
configuration ownership status
fresh-process status
rollout duration
finite-state status
final position
final depth
```

Values will be derived from validations actually performed in this notebook.

No future test count will be predicted here.

---

### Validation Figure

One technical validation figure will summarize the production rollout using:

```text
3D ground-truth trajectory
depth truth versus noisy depth measurement
heading truth versus noisy heading measurement
```

The figure is intended as an inspection aid, not as evidence replacing numerical assertions.

Ground truth and measurements remain visually distinct concepts.

---

### Output Structure

This step will create:

```text
results/notebooks/10_simulator_kernel/
├── data/
│   ├── production_rollout.csv
│   └── summary_metrics.csv
│
└── plots/
    └── simulator_kernel_validation.png
```

The directories will be created only now because validated artifacts actually exist to populate them.

---

### Validation Criteria

This step passes if:

1. the Notebook 10 result directory is created or reused safely;
2. `data/production_rollout.csv` is written;
3. `data/summary_metrics.csv` is written;
4. `plots/simulator_kernel_validation.png` is written;
5. the rollout uses the promoted production `Simulator`;
6. all recorded truth values remain finite;
7. all recorded observation values remain finite;
8. rollout time advances correctly;
9. the generated CSV files can be read back successfully;
10. all artifacts remain inside `results/notebooks/10_simulator_kernel/`;
11. no earlier notebook artifact is modified.

After this step, only dedicated automated tests and the full repository regression suite remain before the final Notebook 10 conclusion.

In [15]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hydrolab3d.core import Simulator
from hydrolab3d.sensors import wrap_angle_pi


# ---------------------------------------------------------------------
# Notebook 10 output directories
# ---------------------------------------------------------------------

notebook10_output_dir = (
    repo_root
    / "results"
    / "notebooks"
    / "10_simulator_kernel"
)

notebook10_data_dir = (
    notebook10_output_dir
    / "data"
)

notebook10_plots_dir = (
    notebook10_output_dir
    / "plots"
)

notebook10_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

notebook10_plots_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# Persistent production-rollout configuration
# ---------------------------------------------------------------------

artifact_initial_pose = np.array(
    [
        0.0,
        0.0,
        -6.0,
        np.deg2rad(2.0),
        np.deg2rad(-1.5),
        np.deg2rad(10.0),
    ],
    dtype=float,
)

artifact_initial_velocity = np.zeros(
    6,
    dtype=float,
)

artifact_dt = 0.02

artifact_steps = 500

artifact_duration = (
    artifact_steps
    * artifact_dt
)


artifact_dynamics = {
    "mass": probe_mass,
    "inertia": (
        probe_inertia.copy()
    ),
    "added_inertia": (
        probe_added_inertia.copy()
    ),
    "linear_damping": (
        probe_linear_damping.copy()
    ),
    "quadratic_damping": (
        probe_quadratic_damping.copy()
    ),
    "fluid_density": (
        probe_fluid_density
    ),
    "displaced_volume": (
        probe_displaced_volume
    ),
    "center_of_gravity": (
        probe_center_of_gravity.copy()
    ),
    "center_of_buoyancy": (
        probe_center_of_buoyancy.copy()
    ),
    "current_velocity_world": np.array(
        [
            0.18,
            -0.06,
            0.0,
        ],
        dtype=float,
    ),
}


artifact_sensor_config = {
    "depth": {
        "bias": 0.03,
        "noise_std": 0.04,
    },
    "heading": {
        "bias": np.deg2rad(0.75),
        "noise_std": np.deg2rad(1.00),
    },
    "dvl": {
        "bias": np.array(
            [0.01, -0.01, 0.005],
            dtype=float,
        ),
        "noise_std": 0.015,
    },
    "gyroscope": {
        "bias": np.array(
            [0.001, -0.001, 0.0005],
            dtype=float,
        ),
        "noise_std": 0.002,
    },
    "accelerometer": {
        "bias": np.array(
            [0.02, -0.01, 0.03],
            dtype=float,
        ),
        "noise_std": 0.025,
    },
}


artifact_sensor_seed = 101010


# ---------------------------------------------------------------------
# Production simulator
# ---------------------------------------------------------------------

artifact_simulator = Simulator(
    initial_pose=artifact_initial_pose,
    initial_body_velocity=(
        artifact_initial_velocity
    ),
    dt=artifact_dt,
    dynamics_kwargs=artifact_dynamics,
    sensor_config=artifact_sensor_config,
    sensor_seed=artifact_sensor_seed,
)


# ---------------------------------------------------------------------
# Allocate rollout storage
# ---------------------------------------------------------------------

artifact_times = np.empty(
    artifact_steps,
    dtype=float,
)

artifact_poses = np.empty(
    (artifact_steps, 6),
    dtype=float,
)

artifact_velocities = np.empty(
    (artifact_steps, 6),
    dtype=float,
)

artifact_depth = np.empty(
    artifact_steps,
    dtype=float,
)

artifact_heading = np.empty(
    artifact_steps,
    dtype=float,
)

artifact_dvl = np.empty(
    (artifact_steps, 3),
    dtype=float,
)

artifact_gyro = np.empty(
    (artifact_steps, 3),
    dtype=float,
)

artifact_accel = np.empty(
    (artifact_steps, 3),
    dtype=float,
)

artifact_actions = np.empty(
    (artifact_steps, 6),
    dtype=float,
)


# ---------------------------------------------------------------------
# Execute deterministic production rollout
# ---------------------------------------------------------------------

for artifact_index in range(
    artifact_steps
):
    artifact_t = (
        artifact_index
        * artifact_dt
    )

    # Smooth deterministic six-axis command sequence.
    artifact_action = np.array(
        [
            10.0
            + 4.0
            * np.sin(
                0.45
                * artifact_t
            ),

            2.0
            * np.sin(
                0.30
                * artifact_t
            ),

            1.5
            * np.sin(
                0.25
                * artifact_t
            ),

            0.04
            * np.sin(
                0.70
                * artifact_t
            ),

            0.05
            * np.sin(
                0.55
                * artifact_t
            ),

            0.08
            * np.sin(
                0.35
                * artifact_t
            ),
        ],
        dtype=float,
    )

    artifact_result = (
        artifact_simulator.step(
            artifact_action
        )
    )

    artifact_truth = (
        artifact_result["truth"]
    )

    artifact_observation = (
        artifact_result[
            "observation"
        ]
    )

    artifact_times[
        artifact_index
    ] = artifact_truth[
        "time"
    ]

    artifact_poses[
        artifact_index
    ] = artifact_truth[
        "pose"
    ]

    artifact_velocities[
        artifact_index
    ] = artifact_truth[
        "body_velocity"
    ]

    artifact_depth[
        artifact_index
    ] = artifact_observation[
        "depth"
    ]

    artifact_heading[
        artifact_index
    ] = artifact_observation[
        "heading"
    ]

    artifact_dvl[
        artifact_index
    ] = artifact_observation[
        "dvl"
    ]

    artifact_gyro[
        artifact_index
    ] = artifact_observation[
        "gyroscope"
    ]

    artifact_accel[
        artifact_index
    ] = artifact_observation[
        "accelerometer"
    ]

    artifact_actions[
        artifact_index
    ] = artifact_action


# ---------------------------------------------------------------------
# Numerical rollout validation
# ---------------------------------------------------------------------

expected_artifact_times = (
    np.arange(
        1,
        artifact_steps + 1,
        dtype=float,
    )
    * artifact_dt
)

assert np.allclose(
    artifact_times,
    expected_artifact_times,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    artifact_times[-1],
    artifact_duration,
    rtol=0.0,
    atol=1e-12,
)

for artifact_array in [
    artifact_poses,
    artifact_velocities,
    artifact_depth,
    artifact_heading,
    artifact_dvl,
    artifact_gyro,
    artifact_accel,
    artifact_actions,
]:
    assert np.all(
        np.isfinite(
            artifact_array
        )
    )


# ---------------------------------------------------------------------
# Derived truth quantities
# ---------------------------------------------------------------------

artifact_true_depth = (
    -artifact_poses[:, 2]
)

artifact_true_heading = np.asarray(
    [
        wrap_angle_pi(
            value
        )
        for value in artifact_poses[
            :, 5
        ]
    ],
    dtype=float,
)


# ---------------------------------------------------------------------
# Build production rollout DataFrame
# ---------------------------------------------------------------------

artifact_rollout_df = pd.DataFrame(
    {
        "time_s": artifact_times,

        "x_m": artifact_poses[:, 0],
        "y_m": artifact_poses[:, 1],
        "z_m": artifact_poses[:, 2],

        "roll_rad": artifact_poses[:, 3],
        "pitch_rad": artifact_poses[:, 4],
        "yaw_rad": artifact_poses[:, 5],

        "u_mps": artifact_velocities[:, 0],
        "v_mps": artifact_velocities[:, 1],
        "w_mps": artifact_velocities[:, 2],

        "p_radps": artifact_velocities[:, 3],
        "q_radps": artifact_velocities[:, 4],
        "r_radps": artifact_velocities[:, 5],

        "Fx_N": artifact_actions[:, 0],
        "Fy_N": artifact_actions[:, 1],
        "Fz_N": artifact_actions[:, 2],

        "Mx_Nm": artifact_actions[:, 3],
        "My_Nm": artifact_actions[:, 4],
        "Mz_Nm": artifact_actions[:, 5],

        "depth_truth_m": (
            artifact_true_depth
        ),
        "depth_measured_m": (
            artifact_depth
        ),

        "heading_truth_rad": (
            artifact_true_heading
        ),
        "heading_measured_rad": (
            artifact_heading
        ),

        "dvl_x_mps": artifact_dvl[:, 0],
        "dvl_y_mps": artifact_dvl[:, 1],
        "dvl_z_mps": artifact_dvl[:, 2],

        "gyro_x_radps": (
            artifact_gyro[:, 0]
        ),
        "gyro_y_radps": (
            artifact_gyro[:, 1]
        ),
        "gyro_z_radps": (
            artifact_gyro[:, 2]
        ),

        "accel_x_mps2": (
            artifact_accel[:, 0]
        ),
        "accel_y_mps2": (
            artifact_accel[:, 1]
        ),
        "accel_z_mps2": (
            artifact_accel[:, 2]
        ),
    }
)


# ---------------------------------------------------------------------
# Save rollout CSV
# ---------------------------------------------------------------------

artifact_rollout_path = (
    notebook10_data_dir
    / "production_rollout.csv"
)

artifact_rollout_df.to_csv(
    artifact_rollout_path,
    index=False,
)


# ---------------------------------------------------------------------
# Build compact summary metrics
# ---------------------------------------------------------------------

artifact_summary_rows = [
    {
        "metric": (
            "direct_pose_parity_max_abs_error"
        ),
        "value": float(
            np.max(
                np.abs(
                    production_parity_truth[
                        "pose"
                    ]
                    - production_direct_pose
                )
            )
        ),
        "units": "state units",
    },
    {
        "metric": (
            "direct_velocity_parity_max_abs_error"
        ),
        "value": float(
            np.max(
                np.abs(
                    production_parity_truth[
                        "body_velocity"
                    ]
                    - production_direct_velocity
                )
            )
        ),
        "units": "state units",
    },
    {
        "metric": (
            "same_seed_truth_replay_exact"
        ),
        "value": int(
            production_same_seed_truth_exact
        ),
        "units": "bool",
    },
    {
        "metric": (
            "same_seed_observation_replay_exact"
        ),
        "value": int(
            production_same_seed_obs_exact
        ),
        "units": "bool",
    },
    {
        "metric": (
            "different_seed_truth_exact"
        ),
        "value": int(
            production_different_seed_truth_exact
        ),
        "units": "bool",
    },
    {
        "metric": (
            "configuration_owned"
        ),
        "value": int(
            production_configuration_owned
        ),
        "units": "bool",
    },
    {
        "metric": (
            "state_snapshot_independent"
        ),
        "value": int(
            production_snapshot_independent
        ),
        "units": "bool",
    },
    {
        "metric": (
            "fresh_process_return_code"
        ),
        "value": int(
            fresh_process_result.returncode
        ),
        "units": "code",
    },
    {
        "metric": (
            "artifact_rollout_duration"
        ),
        "value": float(
            artifact_times[-1]
        ),
        "units": "s",
    },
    {
        "metric": (
            "artifact_rollout_all_finite"
        ),
        "value": 1,
        "units": "bool",
    },
    {
        "metric": "final_x",
        "value": float(
            artifact_poses[-1, 0]
        ),
        "units": "m",
    },
    {
        "metric": "final_y",
        "value": float(
            artifact_poses[-1, 1]
        ),
        "units": "m",
    },
    {
        "metric": "final_z",
        "value": float(
            artifact_poses[-1, 2]
        ),
        "units": "m",
    },
    {
        "metric": "final_depth",
        "value": float(
            artifact_true_depth[-1]
        ),
        "units": "m",
    },
]


artifact_summary_df = pd.DataFrame(
    artifact_summary_rows
)

artifact_summary_path = (
    notebook10_data_dir
    / "summary_metrics.csv"
)

artifact_summary_df.to_csv(
    artifact_summary_path,
    index=False,
)


# ---------------------------------------------------------------------
# Create compact technical validation figure
# ---------------------------------------------------------------------

artifact_figure_path = (
    notebook10_plots_dir
    / "simulator_kernel_validation.png"
)

artifact_fig = plt.figure(
    figsize=(12, 10)
)

artifact_ax_trajectory = (
    artifact_fig.add_subplot(
        3,
        1,
        1,
        projection="3d",
    )
)

artifact_ax_trajectory.plot(
    artifact_poses[:, 0],
    artifact_poses[:, 1],
    artifact_poses[:, 2],
)

artifact_ax_trajectory.set_xlabel(
    "x [m]"
)

artifact_ax_trajectory.set_ylabel(
    "y [m]"
)

artifact_ax_trajectory.set_zlabel(
    "z [m]"
)

artifact_ax_trajectory.set_title(
    "Production Simulator Ground-Truth Trajectory"
)


artifact_ax_depth = (
    artifact_fig.add_subplot(
        3,
        1,
        2,
    )
)

artifact_ax_depth.plot(
    artifact_times,
    artifact_true_depth,
    label="truth",
)

artifact_ax_depth.plot(
    artifact_times,
    artifact_depth,
    label="measurement",
    alpha=0.7,
)

artifact_ax_depth.set_xlabel(
    "Time [s]"
)

artifact_ax_depth.set_ylabel(
    "Depth [m]"
)

artifact_ax_depth.set_title(
    "Depth Truth vs Sensor Measurement"
)

artifact_ax_depth.legend()


artifact_ax_heading = (
    artifact_fig.add_subplot(
        3,
        1,
        3,
    )
)

artifact_ax_heading.plot(
    artifact_times,
    artifact_true_heading,
    label="truth",
)

artifact_ax_heading.plot(
    artifact_times,
    artifact_heading,
    label="measurement",
    alpha=0.7,
)

artifact_ax_heading.set_xlabel(
    "Time [s]"
)

artifact_ax_heading.set_ylabel(
    "Heading [rad]"
)

artifact_ax_heading.set_title(
    "Heading Truth vs Sensor Measurement"
)

artifact_ax_heading.legend()


artifact_fig.tight_layout()

artifact_fig.savefig(
    artifact_figure_path,
    dpi=160,
    bbox_inches="tight",
)

plt.close(
    artifact_fig
)


# ---------------------------------------------------------------------
# Read back persistent data
# ---------------------------------------------------------------------

artifact_rollout_readback = (
    pd.read_csv(
        artifact_rollout_path
    )
)

artifact_summary_readback = (
    pd.read_csv(
        artifact_summary_path
    )
)


assert len(
    artifact_rollout_readback
) == artifact_steps

assert len(
    artifact_summary_readback
) == len(
    artifact_summary_rows
)

assert (
    artifact_figure_path.exists()
)

assert (
    artifact_figure_path.stat().st_size
    > 0
)


# ---------------------------------------------------------------------
# Confirm every artifact is inside Notebook 10 directory
# ---------------------------------------------------------------------

artifact_paths = [
    artifact_rollout_path,
    artifact_summary_path,
    artifact_figure_path,
]

for artifact_path in artifact_paths:
    artifact_path.resolve().relative_to(
        notebook10_output_dir.resolve()
    )


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 12 PERSISTENT ARTIFACTS")
print("=" * 80)


print("\nProduction rollout")
print("-" * 80)

print(
    f"steps       : "
    f"{artifact_steps}"
)

print(
    f"duration    : "
    f"{artifact_times[-1]:.12f} s"
)

print(
    "final pose  :",
    artifact_poses[-1],
)

print(
    "final velocity:",
    artifact_velocities[-1],
)

print(
    f"all finite  : "
    f"{np.all(np.isfinite(artifact_poses)) and np.all(np.isfinite(artifact_velocities))}"
)


print("\nPersistent artifacts")
print("-" * 80)

for artifact_path in artifact_paths:
    print(
        artifact_path.relative_to(
            repo_root
        )
    )


print("\nRead-back validation")
print("-" * 80)

print(
    "rollout rows :",
    len(
        artifact_rollout_readback
    ),
)

print(
    "summary rows :",
    len(
        artifact_summary_readback
    ),
)

print(
    "figure bytes :",
    artifact_figure_path.stat().st_size,
)


print("\n" + "=" * 80)
print("STEP 12 PERSISTENT ARTIFACTS PASSED")
print("=" * 80)

print(
    "Notebook 10 now has a compact persistent production-rollout record, "
    "summary metrics, and technical validation figure."
)

print(
    "All generated files are isolated under "
    "results/notebooks/10_simulator_kernel/."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 12 PERSISTENT ARTIFACTS

Production rollout
--------------------------------------------------------------------------------
steps       : 500
duration    : 10.000000000000 s
final pose  : [ 6.28294472e+00  1.04367629e+00 -5.60554728e+00 -2.04145961e-03
 -4.61034485e-05 -1.72524593e-01]
final velocity: [ 0.71403133  0.18207682  0.06760344 -0.02477905  0.00797362 -0.11077823]
all finite  : True

Persistent artifacts
--------------------------------------------------------------------------------
results/notebooks/10_simulator_kernel/data/production_rollout.csv
results/notebooks/10_simulator_kernel/data/summary_metrics.csv
results/notebooks/10_simulator_kernel/plots/simulator_kernel_validation.png

Read-back validation
--------------------------------------------------------------------------------
rollout rows : 500
summary rows : 14
figure bytes : 324595

STEP 12 PERSISTENT ARTIFACTS PASSED
Notebook 10 now has a compact persistent production-rollout recor

## Step 13 - Final Automated Regression Validation

The previous Step 13 execution successfully completed both pytest runs:

```text
dedicated Simulator suite : 15 passed
full repository suite     : 167 passed
```

The step failed only afterward while parsing those already-successful pytest summaries.

The pass-count regular expression mistakenly searched for a literal:

```text
\d
```

rather than matching numerical digits.

The incorrect expression was conceptually:

```python
r"(\\d+) passed"
```

while the correct expression is:

```python
r"(\d+) passed"
```

This is a notebook reporting error, not:

```text
a Simulator failure
a regression-test failure
a package failure
a production-source failure
```

No production source needs modification.

No test implementation needs modification.

This replacement will rerun both pytest commands under the already-corrected `src/` import environment, verify production-source integrity, and extract the real pass counts correctly.

---

### Final Test Sequence

The execution order remains:

```text
tests/test_simulator.py
        ↓
dedicated suite must pass
        ↓
full repository pytest
        ↓
complete suite must pass
```

The repository uses a `src/` package layout, so the subprocess environment explicitly provides:

```text
PYTHONPATH=<repository>/src
```

---

### Source Integrity

SHA-256 hashes of all production Python files under:

```text
src/hydrolab3d/
```

are recorded before pytest and compared afterward.

Thus successful testing must satisfy:

$$
\boxed{
\text{production source before pytest}
=
\text{production source after pytest}
}
$$

---

### Baseline Comparison

The inherited pre-Notebook-10 suite contained:

```text
152 passed
```

The final full-suite count will be read directly from pytest output.

The difference between the final count and the inherited baseline will therefore report how many additional pytest cases Notebook 10 introduced.

No count is assumed in advance.

---

### Validation Criteria

Step 13 passes if:

1. `tests/test_simulator.py` exists;
2. the dedicated Simulator suite passes;
3. the full repository suite passes;
4. both pytest processes return code `0`;
5. the actual pass counts are parsed successfully;
6. the full suite contains zero failures;
7. production Python source hashes are unchanged;
8. pytest imports HydroLab-3D from the repository `src/` tree.

Passing this step completes Notebook 10 implementation.

The only remaining notebook cell after this validation is the final Conclusion and Notebook 11 handoff.

In [18]:
from pathlib import Path
import hashlib
import os
import re
import subprocess
import sys


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

simulator_test_path = (
    repo_root
    / "tests"
    / "test_simulator.py"
)

production_source_root = (
    repo_root
    / "src"
    / "hydrolab3d"
)

source_import_root = (
    repo_root
    / "src"
)


# ---------------------------------------------------------------------
# Confirm the dedicated test file created by Step 13 exists
# ---------------------------------------------------------------------

if not simulator_test_path.exists():
    raise FileNotFoundError(
        f"Expected simulator regression test file is missing: "
        f"{simulator_test_path}"
    )


# ---------------------------------------------------------------------
# Hash production Python source
# ---------------------------------------------------------------------

def hash_python_sources(root):
    hashes = {}

    for path in sorted(
        root.rglob("*.py")
    ):
        hashes[
            str(
                path.relative_to(
                    repo_root
                )
            )
        ] = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

    return hashes


production_hashes_before = (
    hash_python_sources(
        production_source_root
    )
)


# ---------------------------------------------------------------------
# Explicit pytest subprocess environment
#
# HydroLab-3D uses a src/ package layout.
# ---------------------------------------------------------------------

pytest_environment = (
    os.environ.copy()
)

existing_pythonpath = (
    pytest_environment.get(
        "PYTHONPATH"
    )
)

if existing_pythonpath:
    pytest_environment[
        "PYTHONPATH"
    ] = (
        str(
            source_import_root
        )
        + os.pathsep
        + existing_pythonpath
    )
else:
    pytest_environment[
        "PYTHONPATH"
    ] = str(
        source_import_root
    )


# ---------------------------------------------------------------------
# Pytest runner
# ---------------------------------------------------------------------

def run_pytest(arguments):
    return subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            *arguments,
        ],
        cwd=repo_root,
        env=pytest_environment,
        capture_output=True,
        text=True,
        check=False,
    )


# ---------------------------------------------------------------------
# Dedicated Simulator regression suite
# ---------------------------------------------------------------------

dedicated_test_result = (
    run_pytest(
        [
            "-q",
            "tests/test_simulator.py",
        ]
    )
)


print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 10 - STEP 13 AUTOMATED TESTING")
print("=" * 80)


print("\nPytest subprocess environment")
print("-" * 80)

print(
    "Python executable :",
    sys.executable,
)

print(
    "PYTHONPATH prefix :",
    source_import_root,
)


print("\nDedicated simulator tests")
print("-" * 80)

if dedicated_test_result.stdout.strip():
    print(
        dedicated_test_result.stdout
    )
else:
    print(
        "<NO STDOUT>"
    )

if dedicated_test_result.stderr.strip():
    print(
        "\nDedicated-test stderr:"
    )

    print(
        dedicated_test_result.stderr
    )


assert (
    dedicated_test_result.returncode
    == 0
), (
    "Dedicated Simulator regression suite failed.\n\n"
    f"{dedicated_test_result.stdout}\n"
    f"{dedicated_test_result.stderr}"
)


# ---------------------------------------------------------------------
# Full repository regression suite
# ---------------------------------------------------------------------

full_test_result = (
    run_pytest(
        [
            "-q",
        ]
    )
)


print("\nFull repository regression suite")
print("-" * 80)

if full_test_result.stdout.strip():
    print(
        full_test_result.stdout
    )
else:
    print(
        "<NO STDOUT>"
    )

if full_test_result.stderr.strip():
    print(
        "\nFull-suite stderr:"
    )

    print(
        full_test_result.stderr
    )


assert (
    full_test_result.returncode
    == 0
), (
    "Full HydroLab-3D regression suite failed.\n\n"
    f"{full_test_result.stdout}\n"
    f"{full_test_result.stderr}"
)


# ---------------------------------------------------------------------
# Verify pytest did not alter production source
# ---------------------------------------------------------------------

production_hashes_after = (
    hash_python_sources(
        production_source_root
    )
)

production_sources_unchanged = (
    production_hashes_before
    == production_hashes_after
)

assert production_sources_unchanged


# ---------------------------------------------------------------------
# Correct pytest pass-count parser
# ---------------------------------------------------------------------

def extract_pass_count(output):
    matches = re.findall(
        r"(\d+) passed",
        output,
    )

    if not matches:
        return None

    return int(
        matches[-1]
    )


dedicated_pass_count = (
    extract_pass_count(
        dedicated_test_result.stdout
    )
)

full_pass_count = (
    extract_pass_count(
        full_test_result.stdout
    )
)


assert dedicated_pass_count is not None, (
    "Could not parse dedicated pytest pass count."
)

assert full_pass_count is not None, (
    "Could not parse full-suite pytest pass count."
)


# ---------------------------------------------------------------------
# Baseline comparison
# ---------------------------------------------------------------------

inherited_baseline_count = 152

tests_above_inherited_baseline = (
    full_pass_count
    - inherited_baseline_count
)


assert (
    full_pass_count
    >= inherited_baseline_count
)


# ---------------------------------------------------------------------
# Final implementation report
# ---------------------------------------------------------------------

print("\nTest summary")
print("-" * 80)

print(
    f"dedicated Simulator tests passed : "
    f"{dedicated_pass_count}"
)

print(
    f"full repository tests passed      : "
    f"{full_pass_count}"
)

print(
    "full repository failures          : 0"
)

print(
    f"inherited pre-Notebook-10 baseline : "
    f"{inherited_baseline_count}"
)

print(
    f"tests above inherited baseline     : "
    f"{tests_above_inherited_baseline}"
)

print(
    "production source unchanged "
    "during pytest                    :",
    production_sources_unchanged,
)


print("\nRegression test file")
print("-" * 80)

print(
    simulator_test_path.relative_to(
        repo_root
    )
)


print("\n" + "=" * 80)
print("STEP 13 AUTOMATED TESTING PASSED")
print("=" * 80)

print(
    "The dedicated production Simulator regression suite passed "
    f"{dedicated_pass_count} tests with zero failures."
)

print(
    "The complete HydroLab-3D regression suite passed "
    f"{full_pass_count} tests with zero failures."
)

print(
    "The full suite contains "
    f"{tests_above_inherited_baseline} tests above the inherited "
    f"{inherited_baseline_count}-test baseline."
)

print(
    "Production Python source remained byte-for-byte unchanged "
    "throughout pytest execution."
)

print(
    "Notebook 10 implementation is complete."
)

HYDROLAB-3D NOTEBOOK 10 - STEP 13 AUTOMATED TESTING

Pytest subprocess environment
--------------------------------------------------------------------------------
Python executable : /home/agniacolyte/HydroLab-3D/.venv/bin/python
PYTHONPATH prefix : /home/agniacolyte/HydroLab-3D/src

Dedicated simulator tests
--------------------------------------------------------------------------------
...............                                                          [100%]
15 passed in 0.47s


Full repository regression suite
--------------------------------------------------------------------------------
........................................................................ [ 43%]
........................................................................ [ 86%]
.......................                                                  [100%]
167 passed in 1.16s


Test summary
--------------------------------------------------------------------------------
dedicated Simulator tests passed : 1

# Conclusion - Simulator Kernel Integration

Notebook 10 completed the transition from independently validated HydroLab-3D subsystems into a reusable production simulator kernel.

The central objective was not to build new vehicle physics.

The objective was to orchestrate the already validated production components into a deterministic simulator lifecycle with a clean interface:

```python
simulator = Simulator(...)

state = simulator.reset()

result = simulator.step(action)
```

The resulting production implementation now lives at:

```text
src/hydrolab3d/core/simulator.py
```

and is exposed through:

```python
from hydrolab3d.core import Simulator
```

---

## Final Production Architecture

The implemented kernel owns:

```text
initial pose
initial body velocity
current pose
current body velocity
simulation time
fixed timestep
dynamics configuration
sensor configuration
sensor RNG
latest integration diagnostics
```

The public lifecycle is intentionally small.

### Reset

```python
state = simulator.reset()
```

returns authoritative ground truth:

```text
time
pose
body_velocity
```

with:

```text
pose = [x, y, z, phi, theta, psi]

body_velocity = [u, v, w, p, q, r]
```

The inherited coordinate convention remains:

```text
world +z points upward
surface z = 0
depth = -z
```

---

## Step Interface

The validated action interface is a generalized body-frame wrench:

```text
[Fx, Fy, Fz, Mx, My, Mz]
```

or:

$$
\tau
=
\begin{bmatrix}
F_x &
F_y &
F_z &
M_x &
M_y &
M_z
\end{bmatrix}^{T}.
$$

A call to:

```python
result = simulator.step(action)
```

advances exactly one fixed simulation timestep.

The returned structure is:

```text
result
├── truth
│   ├── time
│   ├── pose
│   └── body_velocity
│
└── observation
    ├── depth
    ├── heading
    ├── dvl
    ├── gyroscope
    └── accelerometer
```

This creates a deliberate separation between:

```text
authoritative simulator state
```

and:

```text
imperfect sensor measurements
```

such that:

$$
\boxed{
\text{truth}
\neq
\text{observation}
}
$$

---

## No Duplicate Physics

The Simulator does not implement its own six-degree-of-freedom equations.

Physical propagation remains delegated to:

```python
integrate_six_dof_euler(...)
```

from the existing production UUV model.

The kernel therefore does not duplicate:

```text
rigid-body dynamics
added mass
Coriolis terms
hydrodynamic damping
hydrostatic restoring effects
current-relative motion
Euler kinematics
numerical integration
```

The core simulator is an orchestration layer over validated production physics.

This was verified directly.

For identical:

```text
initial pose
initial body velocity
vehicle parameters
current
action
timestep
```

the production:

```python
Simulator.step(action)
```

produced exactly the same next:

```text
pose
body velocity
```

as a direct call to:

```python
integrate_six_dof_euler(...)
```

with measured maximum differences:

```text
pose max absolute difference     = 0.0
velocity max absolute difference = 0.0
```

The full returned integration diagnostic set also matched exactly during notebook-local parity validation.

---

## Deterministic Model Time

Simulation time is internal model time rather than wall-clock time.

For fixed timestep:

$$
\Delta t
$$

the kernel advances:

$$
t_{k+1}
=
t_k
+
\Delta t.
$$

A validated ten-action replay using:

```text
dt = 0.02 s
```

therefore ended at:

```text
0.20 s
```

exactly.

The longer validation rollouts likewise reached their expected final times without time corruption.

---

## Exact Reset and Replay

The simulator reset contract was validated over nontrivial multi-axis action sequences.

Calling:

```python
reset()
```

restores exactly:

```text
time = 0
initial pose
initial body velocity
```

and clears the latest integration diagnostics.

For deterministic dynamics:

```text
same initial state
+ same configuration
+ same action sequence
=
same state trajectory
```

was verified with exact array equality.

The repeated trajectory differences were:

```text
maximum time difference     = 0
maximum pose difference     = 0
maximum velocity difference = 0
```

This establishes deterministic experiment replay for non-stochastic truth evolution.

---

## Environmental Integration

Ocean-current effects were preserved through the production dynamics rather than recreated inside the kernel.

Two otherwise identical zero-action simulations were compared:

```text
still water:
[0.0, 0.0, 0.0] m/s

uniform current:
[0.4, -0.15, 0.0] m/s
```

over:

```text
500 steps
dt = 0.02 s
duration = 10 s
```

The still-water vehicle remained stationary in translation, while the current-driven case evolved to a different state.

The final translational separation was approximately:

```text
2.527817976447 m
```

and the final linear-velocity difference was approximately:

```text
0.371312292223 m/s
```

Both histories remained finite and both were exactly reproducible after reset.

This confirms that the kernel correctly preserves the inherited world-to-dynamics interaction.

---

## Sensor Integration

The production kernel now derives post-step observations from the existing production sensor models:

```text
depth
heading
DVL
gyroscope
accelerometer
```

No sensor equation was reimplemented in the core simulator.

The observation path is:

```text
action
  ↓
production 6-DOF step
  ↓
post-step truth
  +
nu_dot diagnostic
  ↓
production sensor models
  ↓
observation
```

The accelerometer receives the already-computed:

```python
diagnostics["nu_dot"]
```

rather than forcing a second dynamics evaluation.

---

## Truth and Sensor Randomness Are Decoupled

Sensor randomness is owned through an explicit:

```python
numpy.random.Generator
```

constructed from:

```python
sensor_seed
```

Reset recreates the configured random sequence.

Therefore:

```text
same seed
+ same actions
=
same noisy observation sequence
```

was validated exactly.

Changing only the sensor seed produced different measurements while leaving the physical trajectory unchanged.

The largest observation difference observed during production validation was approximately:

```text
1.154431587159e-01
```

while:

```text
pose history
body-velocity history
time history
```

remained exactly identical.

Thus:

$$
\boxed{
\text{sensor randomness does not alter simulator truth}
}
$$

---

## State Ownership

The production Simulator independently owns its truth state.

Arrays supplied during construction are copied.

Arrays returned by:

```python
state()
```

are also copies.

Mutating a returned snapshot therefore cannot modify internal simulator state.

This prevents accidental coupling such as:

```python
snapshot = simulator.state()
snapshot["pose"][0] = 999.0
```

silently corrupting the simulator.

The dedicated production regression tests preserve this behavior.

---

## Configuration Ownership

A subtler ownership problem was also addressed.

A shallow dictionary copy is insufficient because configurations contain mutable NumPy arrays such as:

```text
inertia
added inertia
damping coefficients
center of gravity
center of buoyancy
current velocity
sensor biases
```

The final production kernel therefore recursively copies simulator configuration.

After construction, external arrays were deliberately overwritten with extreme values such as:

```text
999
```

including:

```text
current
inertia
linear damping
DVL bias
depth sensor configuration
```

The simulator's resulting:

```text
truth
sensor measurements
reset behavior
replay behavior
```

remained unchanged relative to an untouched reference configuration.

The resulting invariant is:

$$
\boxed{
\text{external configuration mutation}
\not\Rightarrow
\text{internal simulator mutation}
}
$$

---

## Fresh-Process Validation

The promoted Simulator was tested outside the notebook in a separate Python interpreter.

The process used:

```text
/home/agniacolyte/HydroLab-3D/.venv/bin/python
```

and imported:

```text
hydrolab3d.core.simulator.Simulator
```

directly from:

```text
/home/agniacolyte/HydroLab-3D/src/hydrolab3d/core/simulator.py
```

The fresh process successfully:

```text
imported the package
constructed the Simulator
reset the Simulator
executed one physics step
advanced simulation time
generated all sensor observations
returned finite results
```

with:

```text
return code = 0
stderr      = empty
```

This eliminates dependence on notebook-local classes, stale module objects, or Jupyter execution history.

---

## Persistent Notebook 10 Artifacts

Validated production behavior was recorded under:

```text
results/notebooks/10_simulator_kernel/
```

The generated artifacts are:

```text
results/notebooks/10_simulator_kernel/
├── data/
│   ├── production_rollout.csv
│   └── summary_metrics.csv
│
└── plots/
    └── simulator_kernel_validation.png
```

The production rollout contains:

```text
500 steps
10.0 s simulated duration
6-DOF truth
6-axis actuator commands
depth observation
heading observation
DVL observation
gyroscope observation
accelerometer observation
```

The final recorded production state was approximately:

```text
pose =
[
  6.28294472,
  1.04367629,
 -5.60554728,
 -0.00204146,
 -0.00004610,
 -0.17252459
]

body velocity =
[
  0.71403133,
  0.18207682,
  0.06760344,
 -0.02477905,
  0.00797362,
 -0.11077823
]
```

All recorded truth and observation values remained finite.

---

## Production Files Added or Activated

Notebook 10 promoted:

```text
src/hydrolab3d/core/simulator.py
src/hydrolab3d/core/__init__.py
```

and added:

```text
tests/test_simulator.py
```

The following tentative scaffold files were deliberately left untouched:

```text
src/hydrolab3d/core/clock.py
src/hydrolab3d/core/state.py
src/hydrolab3d/core/entity.py
```

Notebook 10 found no engineering requirement for those abstractions yet.

This preserves the project principle:

```text
implement abstractions when validated need appears
```

rather than filling a tentative directory tree merely because it exists.

---

## Automated Regression Validation

The new dedicated Simulator regression suite contains:

```text
15 tests
```

and passed:

```text
15 passed
0 failed
```

The complete HydroLab-3D repository suite then passed:

```text
167 passed
0 failed
```

The inherited pre-Notebook-10 baseline was:

```text
152 passed
```

so Notebook 10 added:

```text
15 passing test cases
```

without regressing any previously validated behavior.

Production Python source hashes were identical before and after pytest execution.

Thus:

$$
\boxed{
167\ \text{passed},\quad 0\ \text{failed}
}
$$

is the validated repository state at the completion of Notebook 10.

---

## What Notebook 10 Did Not Add

The kernel remains intentionally narrow.

Notebook 10 did not introduce:

```text
Gymnasium
reinforcement learning
reward functions
episode termination rules
multi-agent interfaces
controller ownership
mission planning
ROS 2 integration
render-loop ownership
keyboard control
sonar
localization
SLAM
new hydrodynamic equations
new vehicle models
clock abstraction
entity abstraction
state-class hierarchy
```

Those capabilities are not required to establish the simulator kernel itself.

---

## Known Physical and Numerical Boundaries

Notebook 10 inherits the limitations of the validated production models.

The pose representation remains:

```text
[x, y, z, phi, theta, psi]
```

using Euler angles.

Therefore the inherited ZYX Euler singularity near:

$$
\theta=\pm90^\circ
$$

still exists.

Notebook 10 does not replace Euler orientation with quaternions.

Likewise, the kernel does not transform HydroLab-3D into a high-fidelity CFD or contact-dynamics simulator.

It remains a:

```text
lightweight
NumPy-first
CPU-friendly
transparent
algorithm-first
simulation-first
marine robotics sandbox
```

whose fidelity is intentionally concentrated around autonomy-relevant dynamics and sensing.

---

## Architectural Result

Before Notebook 10, HydroLab-3D contained validated components:

```text
kinematics
orientation
dynamics
hydrodynamic drag
buoyancy and gravity
thrusters
ocean currents
6-DOF UUV dynamics
sensor models
closed-loop control
```

After Notebook 10, those components can be executed through a common simulator lifecycle:

```text
               action
                 │
                 ▼
        ┌─────────────────┐
        │    Simulator    │
        └─────────────────┘
                 │
                 ▼
        production 6-DOF
             dynamics
                 │
                 ▼
          updated truth
                 │
        ┌────────┴────────┐
        │                 │
        ▼                 ▼
 ground-truth API    sensor models
                          │
                          ▼
                    observations
```

The result is the first validated integrated HydroLab-3D simulation kernel.

Its central interface is now operational:

```python
result = simulator.step(action)
```

with explicit:

```text
truth
observations
time
reset
configuration
randomness
```

boundaries.

---

## Handoff to Notebook 11

Notebook 11 can now build a learning-facing interface on top of the validated production kernel.

The important architectural direction is:

```text
Notebook 10
production simulator kernel
        ↓
Notebook 11
learning-interface validation
```

Notebook 11 should therefore treat:

```python
hydrolab3d.core.Simulator
```

as an existing production capability rather than recreating vehicle propagation.

Its work can focus on the learning boundary itself, for example:

```text
observation formatting
action formatting
episode reset semantics
termination / truncation logic
reward experiments
learning-oriented wrappers
```

while preserving the distinction between:

```text
core simulator
```

and:

```text
learning interface
```

Notebook 11 should not require the core Simulator to become Gymnasium-specific.

The core kernel should remain usable independently for:

```text
classical control
trajectory experiments
autonomy algorithms
sensor studies
benchmarking
planning
learning wrappers
```

---

# Final Status

Notebook 10 is complete.

The production HydroLab-3D simulator kernel now provides:

```text
✓ deterministic reset
✓ deterministic model time
✓ generalized 6-DOF action stepping
✓ exact inherited-physics parity
✓ current-aware dynamics
✓ finite longer rollouts
✓ ground-truth state ownership
✓ configuration ownership
✓ production sensor observations
✓ truth / observation separation
✓ explicit seeded randomness
✓ deterministic noisy replay
✓ fresh-process package execution
✓ persistent validation artifacts
✓ dedicated regression protection
✓ full repository regression compatibility
```

with final automated validation:

```text
Dedicated Simulator suite : 15 passed
Complete repository suite : 167 passed
Failures                  : 0
```

HydroLab-3D now has the production integration boundary required for subsequent autonomy and learning-interface work.